In [ ]:
# All imports
import os
import sys
sys.path.insert(0, '../')

# Import Constants
from scripts.constants import ADG_SKYNET_ROOT_ADDR

# Import Libraries
import pandas as pd
import numpy as np
import ast
import re
import json
from tqdm import tqdm
import networkx as nx
from pyvis.network import Network
import math  # Import math module for natural log
import datetime

# Plotting libraries
import seaborn as sns
import matplotlib.pyplot as plt
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.express as px
from plotly.colors import qualitative
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from pprint import pprint

# Classes

In [ ]:
# quantile_manager.py
import pandas as pd
import numpy as np

class PSquareQuantileApproximator:
    """
    Implements the extended P² algorithm for dynamic quantile approximation.
    Tracks multiple quantiles using multiple markers.
    """
    def __init__(self, quantiles):
        """
        Initialize the approximator with a list of quantiles.

        :param quantiles: List of quantiles to estimate (e.g., [0.25, 0.5, 0.75])
        """
        self.quantiles = sorted(quantiles)
        self.num_quantiles = len(self.quantiles)
        self.reset()

    def reset(self):
        """Initialize or reset the quantile approximator."""
        self.initial_sample = []
        self.num_markers = 2 * self.num_quantiles + 3  # Number of markers
        self.q = []  # Marker heights
        self.n = []  # Marker positions
        self.n_desired = []  # Desired marker positions
        self.p_inc = []  # Desired position increments

    def fit(self, X):
        """Fit the model to the data."""
        self.reset()
        return self.partial_fit(X)

    def partial_fit(self, X):
        """Incrementally fit the model to the data."""
        for x in X:
            self._partial_fit_single(x)
        return self

    def _partial_fit_single(self, x):
        """Fit the model to a single data point."""
        # Collect initial samples
        if len(self.initial_sample) < self.num_markers:
            self.initial_sample.append(x)
            if len(self.initial_sample) == self.num_markers:
                self.initial_sample.sort()
                self.q = self.initial_sample.copy()
                self.n = list(range(1, self.num_markers + 1))
                self.p_inc = [0] + self.quantiles + [1] + [1 - q for q in reversed(self.quantiles)] + [0]
                total_p_inc = sum(self.p_inc)
                self.n_desired = [1 + (total_p_inc - sum(self.p_inc[:i+1])) * (self.num_markers - 1)
                                  for i in range(self.num_markers)]
        else:
            # Update markers
            if x < self.q[0]:
                self.q[0] = x  # Update minimum
                k = 0
            elif x >= self.q[-1]:
                self.q[-1] = x  # Update maximum
                k = self.num_markers - 2
            else:
                # Find k such that q[k] <= x < q[k+1]
                k = next(i for i in range(self.num_markers - 1) if self.q[i] <= x < self.q[i+1])
            # Increment positions of markers greater than k
            for i in range(k + 1, self.num_markers):
                self.n[i] += 1
            # Update desired positions
            for i in range(self.num_markers):
                self.n_desired[i] += self.p_inc[i]
            # Adjust marker heights
            for i in range(1, self.num_markers - 1):
                d = self.n_desired[i] - self.n[i]
                if (d >= 1 and self.n[i + 1] - self.n[i] > 1) or (d <= -1 and self.n[i - 1] - self.n[i] < -1):
                    d = int(np.sign(d))
                    q_temp = self._parabolic(i, d)
                    if self.q[i - 1] < q_temp < self.q[i + 1]:
                        self.q[i] = q_temp
                    else:
                        self.q[i] = self._linear(i, d)
                    self.n[i] += d

    def _parabolic(self, i, d):
        """Parabolic prediction for marker height adjustment."""
        i = int(i)
        n_i = self.n[i]
        n_im1 = self.n[i - 1]
        n_ip1 = self.n[i + 1]
        q_i = self.q[i]
        q_im1 = self.q[i - 1]
        q_ip1 = self.q[i + 1]
        numerator = d * ((n_i - n_im1 + d) * (q_ip1 - q_i) / (n_ip1 - n_i) +
                         (n_ip1 - n_i - d) * (q_i - q_im1) / (n_i - n_im1))
        denominator = n_ip1 - n_im1
        return q_i + numerator / denominator

    def _linear(self, i, d):
        """Linear prediction for marker height adjustment."""
        i = int(i)
        return self.q[i] + d * (self.q[i + d] - self.q[i]) / (self.n[i + d] - self.n[i])

    def is_initialized(self):
        """Check if the approximator has been initialized with enough data points."""
        return len(self.q) == self.num_markers

    def get_markers(self):
        """Return the current quantile estimates."""
        if self.is_initialized():
            # The quantile estimates are at positions corresponding to the quantiles
            quantile_indices = [1 + i for i in range(self.num_quantiles)]
            quantile_values = [self.q[i] for i in quantile_indices]
            return [self.q[0]] + quantile_values + [self.q[-1]]  # Include min and max
        else:
            # Return the quantiles from the initial sample
            if self.initial_sample:
                quantile_values = [np.percentile(self.initial_sample, q * 100) for q in self.quantiles]
                return [min(self.initial_sample)] + quantile_values + [max(self.initial_sample)]
            else:
                return None

    def get_state(self):
        """Get the internal state for saving or exporting."""
        return {
            'q': self.q,
            'n': self.n,
            'n_desired': self.n_desired,
            'p_inc': self.p_inc,
            'initial_sample': self.initial_sample
        }

    def set_state(self, state):
        """Set the internal state from saved data."""
        self.q = state['q']
        self.n = state['n']
        self.n_desired = state['n_desired']
        self.p_inc = state['p_inc']
        self.initial_sample = state['initial_sample']

    def get_quantile_names(self):
        """Return the names of the quantiles, including 'min' and 'max'."""
        quantile_names = ['min'] + [f'p{int(round(q * 100))}' for q in self.quantiles] + ['max']
        return quantile_names


# gk_quantile_approximator.py
import bisect

class GKQuantileApproximator:
    """
    Implements the Greenwald-Khanna algorithm for quantile estimation.
    """
    def __init__(self, quantiles, epsilon=0.01):
        """
        :param quantiles: List of quantiles to estimate (e.g., [0.25, 0.5, 0.75])
        :param epsilon: Acceptable error in the quantile approximation.
        """
        self.quantiles = sorted(quantiles)
        self.epsilon = epsilon
        self.reset()

    def reset(self):
        """Reset the GK approximator."""
        self.n = 0  # Total number of data points
        self.summary = []  # List of tuples (value, g, delta)

    def partial_fit(self, X):
        """Incrementally fit the model to the data."""
        for x in X:
            self.insert(x)
        return self

    def insert(self, x):
        """Insert a new data point into the summary."""
        self.n += 1
        r = {'value': x, 'g': 1, 'delta': 0}
        if not self.summary:
            self.summary.append(r)
            return

        idx = bisect.bisect_left([s['value'] for s in self.summary], x)
        if idx == 0:
            delta = 0
        elif idx == len(self.summary):
            delta = 0
        else:
            delta = int(2 * self.epsilon * self.n)
        r['delta'] = delta
        self.summary.insert(idx, r)
        self.compress()

    def compress(self):
        """Compress the summary to maintain the error guarantees."""
        i = 0
        while i < len(self.summary) - 1:
            r1 = self.summary[i]
            r2 = self.summary[i + 1]
            if (r1['g'] + r2['g'] + r2['delta']) <= int(2 * self.epsilon * self.n):
                r2['g'] += r1['g']
                self.summary.pop(i)
            else:
                i += 1

    def get_markers(self):
        """Return the quantile estimates."""
        if not self.summary:
            return None
        markers = [self.summary[0]['value']]  # min
        for quantile in self.quantiles:
            rank_min = 0
            desired_rank = quantile * self.n
            for i, s in enumerate(self.summary):
                rank_min += s['g']
                rank_max = rank_min + s['delta']
                if rank_max >= desired_rank + self.epsilon * self.n:
                    markers.append(s['value'])
                    break
        markers.append(self.summary[-1]['value'])  # max
        return markers

    def is_initialized(self):
        """Check if the approximator has enough data."""
        return len(self.summary) > 0

    def get_state(self):
        """Get the internal state for saving or exporting."""
        return {
            'n': self.n,
            'summary': self.summary
        }

    def set_state(self, state):
        """Set the internal state from saved data."""
        self.n = state['n']
        self.summary = state['summary']

    def get_quantile_names(self):
        """Return the names of the quantiles, including 'min' and 'max'."""
        quantile_names = ['min'] + [f'p{int(round(q * 100))}' for q in self.quantiles] + ['max']
        return quantile_names


# t_digest_quantile_approximator.py
from tdigest import TDigest

class TDigestQuantileApproximator:
    """
    Implements the T-Digest algorithm for quantile estimation.
    """
    def __init__(self, quantiles):
        """
        :param quantiles: List of quantiles to estimate (e.g., [0.25, 0.5, 0.75])
        """
        self.quantiles = sorted(quantiles)
        self.reset()

    def reset(self):
        """Reset the T-Digest approximator."""
        self.digest = TDigest()

    def partial_fit(self, X):
        """Incrementally fit the model to the data."""
        for x in X:
            self.digest.update(x)
        return self

    def get_markers(self):
        """Return the quantile estimates."""
        if self.digest.n == 0:
            return None
        markers = [self.percentile(0)]  # min
        for q in self.quantiles:
            markers.append(self.percentile(q * 100))
        markers.append(self.percentile(100))  # max
        return markers

    def percentile(self, p):
        """Compute the percentile value using the digest's percentile method."""
        return self.digest.percentile(p)

    def is_initialized(self):
        """Check if the approximator has enough data."""
        return self.digest.n > 0

    def get_state(self):
        """Get the internal state for saving or exporting."""
        # Use the to_dict() method for serialization
        return self.digest.to_dict()

    def set_state(self, state):
        """Set the internal state from saved data."""
        self.digest = TDigest()
        self.digest.update_from_dict(state)

    def get_quantile_names(self):
        """Return the names of the quantiles, including 'min' and 'max'."""
        quantile_names = ['min'] + [f'p{int(round(q * 100))}' for q in self.quantiles] + ['max']
        return quantile_names

class QuantileManager:
    """
    Manages quantile approximators for different KPIs using specified algorithms and parameters.
    """
    def __init__(self, kpi_approximators):
        """
        :param kpi_approximators: Dictionary mapping KPI names to either:
            - An initialized quantile approximator instance
            - A tuple of (algorithm_class, parameters_dict)
        """
        self.quantile_approximators = {}
        for kpi, approximator_info in kpi_approximators.items():
            if isinstance(approximator_info, tuple):
                # It's a tuple of (algorithm_class, parameters_dict)
                algorithm_class, params = approximator_info
                self.quantile_approximators[kpi] = algorithm_class(**params)
            elif isinstance(approximator_info, type):
                # It's an algorithm class, instantiate with default params
                self.quantile_approximators[kpi] = algorithm_class()
            else:
                # Assume it's an initialized approximator instance
                self.quantile_approximators[kpi] = approximator_info

    def fit(self):
        """Initialize all approximators with empty lists."""
        for approximator in self.quantile_approximators.values():
            approximator.fit([])

    def partial_fit(self, kpi_name, value):
        """
        Update the quantile approximations for a specific KPI.
        
        This method handles both scalar values and arrays, ensuring proper formatting
        and preventing nested arrays.
        
        Args:
            kpi_name (str): The name of the KPI to update
            value (float, int, list, np.ndarray): The value or values to fit
                - If scalar (float/int): Will be wrapped in a list
                - If list/array: Will be processed as is
        
        Returns:
            None
        
        Raises:
            TypeError: If value is not a number, list, or numpy array
            ValueError: If value is a nested array or contains non-numeric elements
        """
        if kpi_name not in self.quantile_approximators:
            return
            
        # Handle different input types
        if isinstance(value, (int, float)):
            # Single numeric value - wrap in a list
            self.quantile_approximators[kpi_name].partial_fit([value])
            
        elif isinstance(value, (list, np.ndarray)):
            # Check if it's a flat array with numeric elements
            if not value:
                # Empty list/array - skip
                return
                
            # Check for nested arrays
            if any(isinstance(item, (list, np.ndarray)) for item in value):
                raise ValueError(f"Nested arrays are not supported for KPI '{kpi_name}'. "
                                 f"Expected a flat array of numbers.")
                
            # Check that all elements are numeric
            if not all(isinstance(item, (int, float)) for item in value):
                raise ValueError(f"Non-numeric elements found in array for KPI '{kpi_name}'. "
                                 f"All elements must be numbers.")
                
            # It's a flat numeric array - pass directly to partial_fit
            self.quantile_approximators[kpi_name].partial_fit(value)
            
        else:
            # Not a supported type
            raise TypeError(f"Unsupported value type for KPI '{kpi_name}': {type(value)}. "
                           f"Expected a number, list, or numpy array.")

    def get_markers(self, kpi_name):
        """Get the markers (quantile estimates) for a specific KPI."""
        if kpi_name in self.quantile_approximators:
            return self.quantile_approximators[kpi_name].get_markers()
        else:
            return None

    def reset(self):
        """Reset all approximators."""
        for approximator in self.quantile_approximators.values():
            approximator.reset()

    def export_markers(self):
        """
        Export all markers and their associated metadata to a DataFrame.
        """
        export_data = []
        for kpi, approx in self.quantile_approximators.items():
            if approx.is_initialized():
                export_data.append({
                    'kpi': kpi,
                    'markers': approx.get_markers(),
                    'state': approx.get_state(),
                    'algorithm': approx.__class__.__name__
                })
        return pd.DataFrame(export_data)

    def load_markers(self, markers_df):
        """
        Load markers and their associated metadata from a DataFrame.
        """
        for _, row in markers_df.iterrows():
            kpi = row['kpi']
            if kpi in self.quantile_approximators:
                approx = self.quantile_approximators[kpi]
                state_str = row['state']
                
                # Convert the string representation to a dictionary
                try:
                    # Try using ast.literal_eval which is safer than eval
                    state_dict = ast.literal_eval(state_str)
                    approx.set_state(state_dict)
                except (SyntaxError, ValueError) as e:
                    print(f"Error parsing state for {kpi}: {e}")

    def represent_markers(self):
        """
        Return a DataFrame with the quantile estimates for each KPI,
        with quantiles as columns named appropriately.
        """
        markers_dict = {}
        for kpi, approximator in self.quantile_approximators.items():
            markers = approximator.get_markers()
            if markers is not None:
                markers_dict[kpi] = {}
                quantile_names = approximator.get_quantile_names()
                for name, value in zip(quantile_names, markers):
                    markers_dict[kpi][name] = value
        df = pd.DataFrame.from_dict(markers_dict, orient='index')
        df.reset_index(inplace=True)
        df.rename(columns={'index': 'kpi'}, inplace=True)
        return df


In [ ]:
from scipy.stats import linregress
import numpy as np
import re
from typing import Dict, List, NamedTuple, Any, Union

class Symbolizer:
    def __init__(
        self,
        quantile_manager,
        kpis: Dict[str, str],
        decisions: Dict[str, str],
        kpi_change_threshold_percent: int = 10,
    ):
        """
        Initializes the Symbolizer with a QuantileManager, KPI list, and decision list.

        :param quantile_manager: An instance of QuantileManager to manage quantiles for KPIs.
        :param kpis: A dictionary mapping KPI human-readable names to environment kpi names.
        :param decisions: A dictionary mapping decision human-readable names to environment decision names.
        :param kpi_change_threshold_percent: Threshold percentage to determine if a KPI has increased or decreased.
        """
        self.quantile_manager = quantile_manager
        self.kpis = kpis
        self.decisions = decisions
        self.kpi_change_threshold_percent = kpi_change_threshold_percent

        # Define category names for standard KPIs
        self.standard_category_names = ["VeryLow", "Low", "Medium", "High", "VeryHigh"]
        # Define category names for special KPIs
        self.special_category_names = ["Low", "Medium", "High"]
        # List of KPIs that use the special three-category system
        self.special_kpis = []  # Assuming no special KPIs in Pensive

        self.array_kpis = ['mase_all']
        self.trend_categories = ["fluctuating", "spiking", "dropping"]

        self.slopes_hist = {'mase_all': []}

        # Map array KPIs to their base KPIs for quantile management
        self.array_to_base_kpi = {
            'mase_all': 'mase',
        }

        # Define current timestep index for each array KPI
        self.array_current_timestep_index = {
            'mase_all': 0,
        }

        self.prev_state_tuple = None

    def create_symbolic_form(self, curr_step_tuple, update_approximators=True):
        """
        Receives the current state of an agent at a timestep as a namedtuple and returns the symbolic representation.
        Internally manages the previous state.

        :param curr_step_tuple: Current timestep namedtuple containing state information.
        :param update_approximators: Whether to update the approximators with current data.
        :return: A list of dictionaries with symbolic representations for each slice.
        """
        effects_symbolic_representation = []
        prev_step_tuple = self.prev_state_tuple

        if prev_step_tuple is not None:
            # 1. Create symbolic rep for each KPI
            symbolic_state = self._calculate_symbolic_state(curr_step_tuple, prev_step_tuple)

            symbolic_representation = {
                "timestep": getattr(curr_step_tuple, "timestep", None),
                **symbolic_state,
                "reward": getattr(curr_step_tuple, "reward", None),
            }
            # pprint(symbolic_representation)
            effects_symbolic_representation.append(symbolic_representation)
        else:
            # Handle the first timestep where there is no previous state
            symbolic_representation = {
                "timestep": getattr(curr_step_tuple, "timestep", None),
                **self._initialize_symbols(curr_step_tuple),
                "reward": getattr(curr_step_tuple, "reward", None),
            }
            # pprint(symbolic_representation)
            effects_symbolic_representation.append(symbolic_representation)

        if update_approximators:
            # Update the quantile approximators with current KPI data
            self._add_timestep_kpi_data_to_approximator(curr_step_tuple)

            # Update the internal previous state with the current slice data
            self.prev_state_tuple = curr_step_tuple

        return effects_symbolic_representation

    def _initialize_symbols(self, curr_step_tuple):
        """
        Initialize symbols for the first timestep.

        :param curr_step_tuple: Current step namedtuple.
        :return: A dictionary with initial symbolic representations.
        """
        symbolic_representation = {}
        
        # Process KPIs (which are now arrays)
        for key, env_name in self.kpis.items():
            curr_values = self._safe_get_attr(curr_step_tuple, env_name)
            # Check if the value is an array/list
            if isinstance(curr_values, list):
                # Process each value in the array
                symbolic_values = []
                for value in curr_values:
                    symbolic_values.append(self._get_initial_symb(value, env_name))
                symbolic_representation[env_name] = symbolic_values
            else:
                # Handle single values (in case some KPIs aren't arrays)
                symbolic_representation[env_name] = self._get_initial_symb(curr_values, env_name)
        
        # Process decisions (which might be single values)
        for key, env_name in self.decisions.items():
            curr_value = self._safe_get_attr(curr_step_tuple, env_name)
            if curr_value is not None:
                # Pass curr_step_tuple to include group information
                symbolic_representation[env_name] = self._create_action_symbols(curr_value, curr_step_tuple)
        
        return symbolic_representation

    def _get_initial_symb(self, curr_value, env_name):
        """
        Generate KPI symbolic representation for the first timestep.

        :param curr_value: Current KPI value.
        :param env_name: Name of the KPI.
        :return: Symbolic representation string.
        """
        markers = self.quantile_manager.get_markers(env_name)
        if markers is None:
            return f"Unknown({env_name})"  # Handle the case where markers are not available
        
        if env_name in self.array_kpis:
            current_timestep_index = self.array_current_timestep_index[env_name]
            change_percentage = self._find_change_percentage(curr_value[current_timestep_index], 0)
            predicate = self._get_predicate(change_percentage)

            # Use the base KPI's markers for categorization
            base_kpi = self.array_to_base_kpi[env_name]
            markers = self.quantile_manager.get_markers(base_kpi)  # Use base KPI here
            if markers is None:
                return f"Unknown({env_name})"

            category = self._get_category(curr_value[current_timestep_index], markers, base_kpi)  # Pass base KPI here
            trend = self._analyze_trend(curr_value, env_name)

            return f"{predicate}({env_name}, {category}, {trend})"
        
        category = self._get_category(curr_value, markers, env_name)
        predicate = "const"
        return f"{predicate}({env_name}, {category})"

    def _calculate_symbolic_state(self, curr_step_tuple, prev_step_tuple):
        """
        Calculate the symbolic state for KPIs and decisions based on current and previous values.

        :param curr_step_tuple: Current namedtuple.
        :param prev_step_tuple: Previous namedtuple.
        :return: A dictionary with symbolic representations.
        """
        symbolic_representation = {}
        
        # Process KPIs (which can be arrays)
        for key, env_name in self.kpis.items():
            curr_values = self._safe_get_attr(curr_step_tuple, env_name)
            prev_values = self._safe_get_attr(prev_step_tuple, env_name)
            
            # print(f"symbolic form of {env_name} with curr_values: {curr_values} and prev_values: {prev_values}")
            
            # Check if the values are arrays/lists
            if isinstance(curr_values, list) and isinstance(prev_values, list):
                # Process each pair of values in the arrays
                symbolic_values = []
                for curr_val, prev_val in zip(curr_values, prev_values):
                    symbolic_values.append(self._get_symb(curr_val, prev_val, env_name))
                symbolic_representation[env_name] = symbolic_values
            else:
                # Handle single values (in case some KPIs aren't arrays)
                symbolic_representation[env_name] = self._get_symb(curr_values, prev_values, env_name)
        
        # Process decisions (which might be single values)
        for key, env_name in self.decisions.items():
            curr_value = self._safe_get_attr(curr_step_tuple, env_name)
            if curr_value is not None:
                # Pass curr_step_tuple to include group information
                symbolic_representation[env_name] = self._create_action_symbols(curr_value, curr_step_tuple)
        
        return symbolic_representation

    def _parse_action_string(self, action_str):
        """
        Parse an action string like '(0, 1, 3, 4, 6)' into a list of scheduled user indices.
        
        :param action_str: String representation of the action.
        :return: List of integers representing scheduled user indices.
        """
        if not action_str:
            return []
            
        # Extract numbers using regex
        scheduled_users = []
        if isinstance(action_str, str):
            # Extract all numbers from the string
            numbers = re.findall(r'\d+', action_str)
            scheduled_users = [int(num) for num in numbers]
        
        return scheduled_users

    def _create_action_symbols(self, action_value, curr_step_tuple=None):
        """
        Create symbolic representation for action, now including group information.
        
        :param action_value: Value from the action column (string or other format).
        :param curr_step_tuple: Current timestep tuple containing group information.
        :return: Array of 7 strings with 'Sched(user, G{group_number})' or 'notSched(user, G{group_number})'.
        """
        # Parse the action value to get scheduled user indices
        scheduled_users = self._parse_action_string(action_value)
        
        # Get group information if available
        groups = None
        if curr_step_tuple is not None:
            groups = self._safe_get_attr(curr_step_tuple, 'group')
        
        # Create symbolic representation for each user (7 users total)
        action_symbols = []
        for user_idx in range(7):
            # Determine group suffix
            group_suffix = ''
            if groups is not None and user_idx < len(groups):
                group_suffix = f', G{groups[user_idx]}'
                
            # Create appropriate symbol based on scheduling status
            if user_idx in scheduled_users:
                action_symbols.append(f"Sched(user{group_suffix})")
            else:
                action_symbols.append(f"notSched(user{group_suffix})")
                    
        return action_symbols

    def _get_symb(self, curr_value, prev_value, kpi_name):
        """Calculate the symbolic representation of KPI changes."""
        if kpi_name in self.array_kpis:
            current_timestep_index = self.array_current_timestep_index[kpi_name]
            # print(f"current timestep value for {kpi_name} is {curr_value[current_timestep_index]}")
            # print(f"previous timestep value for {kpi_name} is {prev_value[current_timestep_index]}")
            change_percentage = self._find_change_percentage(curr_value[current_timestep_index], prev_value[current_timestep_index])
            predicate = self._get_predicate(change_percentage)
            # print(f"change percentage for {kpi_name} is {change_percentage}")
            # print(f"predicate for {kpi_name} is {predicate}")

            # Use the base KPI's markers for categorization
            base_kpi = self.array_to_base_kpi[kpi_name]
            markers = self.quantile_manager.get_markers(base_kpi)  # Use base KPI here
            if markers is None:
                return f"Unknown({kpi_name})"

            category = self._get_category(curr_value[current_timestep_index], markers, base_kpi)  # Pass base KPI here
            trend = self._analyze_trend(curr_value, kpi_name)
            # print(f"curr_value: {curr_value}, prev_value: {prev_value}")
            # print(f"category for {kpi_name} is {category}")
            # print(f"trend for {kpi_name} is {trend}")
            return f"{predicate}({kpi_name}, {category}, {trend})"

        # Original logic for non-array KPIs
        change_percentage = self._find_change_percentage(curr_value, prev_value)
        predicate = self._get_predicate(change_percentage)
        markers = self.quantile_manager.get_markers(kpi_name)
        if markers is None:
            return f"Unknown({kpi_name})"
        category = self._get_category(curr_value, markers, kpi_name)
        return f"{predicate}({kpi_name}, {category})"

    def _get_category(self, value, markers, kpi_name):
        """Categorize a value based on percentile markers."""
        if kpi_name in self.special_kpis:
            # Use fixed percentiles for special KPIs
            if markers is None or len(markers) < 3:
                return "Unknown"

            p30 = markers[1]
            p70 = markers[2]

            if value < p30:
                return self.special_category_names[0]  # Low
            elif value <= p70:
                return self.special_category_names[1]  # Medium
            else:
                return self.special_category_names[2]  # High
        else:
            # Use standard 5 categories for other KPIs
            if markers is None or len(markers) < 5:
                return "Unknown"

            p20 = markers[1]
            p40 = markers[2]
            p60 = markers[3]
            p80 = markers[4]

            if value <= p20:
                return self.standard_category_names[0]  # VeryLow
            elif value <= p40:
                return self.standard_category_names[1]  # Low
            elif value <= p60:
                return self.standard_category_names[2]  # Medium
            elif value <= p80:
                return self.standard_category_names[3]  # High
            else:
                return self.standard_category_names[4]  # VeryHigh

    def _analyze_trend(self, value_array: List[float], kpi_name: str) -> str:
        """
        Analyzes trend in array values, ignoring zeros.
        Returns 'fluctuating', 'spiking', or 'dropping' based on dynamic quantile thresholds.

        :param value_array: List of KPI values.
        :param kpi_name: Name of the KPI being analyzed.
        :return: String representing the trend.
        """
        values = [v for v in value_array if v != 0.0]
        if len(values) < 2:
            return "fluctuating"

        x = np.arange(len(values))
        slope, _, _, _, _ = linregress(x, values)

        self.slopes_hist[kpi_name].append(slope)

        # Fetch dynamic thresholds from quantile manager for slopes
        slope_kpi_name = f"{kpi_name}_slope"  # e.g., 'dl_tput_all_slope'
        slope_markers = self.quantile_manager.get_markers(slope_kpi_name)

        if slope_markers is None or len(slope_markers) < 3:  # min, p20, p80, max - we expect at least min, p20, p80
            # Fallback to fixed thresholds if quantiles are not available
            low_threshold = -0.0074 if kpi_name == 'dl_tput_all' else -0.0151
            high_threshold = 0.0067 if kpi_name == 'dl_tput_all' else 0.0259
        else:
            p20_threshold = slope_markers[1]  # p20 - Correct index is 1
            p80_threshold = slope_markers[2]  # p80 - Correct index is 2
            low_threshold = p20_threshold
            high_threshold = p80_threshold

        if slope > high_threshold:
            return "spiking"
        elif slope < low_threshold:
            return "dropping"
        else:
            return "fluctuating"

    def _find_change_percentage(self, curr_value, prev_value):
        """Calculate the change percentage of the given parameter."""
        if isinstance(curr_value, list):
            # For arrays, use the specified current timestep value
            curr = curr_value
            prev = prev_value
        else:
            curr = curr_value
            prev = prev_value

        if prev == 0:
            if curr == 0:
                return 0
            else:
                return "inf"
        else:
            return int(100 * (curr - prev) / prev)

    def _get_predicate(self, change_percentage):
        """
        Return the correct predicate according to the change percentage.

        :param change_percentage: The percentage change.
        :return: A string representing the predicate ('inc', 'dec', 'const').
        """
        if change_percentage == "inf":
            return "inc"
        elif change_percentage > self.kpi_change_threshold_percent:
            return "inc"
        elif change_percentage < -self.kpi_change_threshold_percent:
            return "dec"
        else:
            return "const"

    def _add_timestep_kpi_data_to_approximator(self, timestep_tuple):
        """
        Adds KPI data of one timestep to the quantile approximators, including slopes for '_all' KPIs.
        Modified to work with namedtuples and handle array values correctly.
        
        :param timestep_tuple: The current timestep data as a namedtuple
        """
        # Loop through each KPI in the mapping
        for kpi_name in self.kpis.values():
            # Handle different KPI types appropriately
            if kpi_name not in self.array_kpis:
                # For scalar KPIs, directly fit the value
                kpi_value = self._safe_get_attr(timestep_tuple, kpi_name)
                self.quantile_manager.partial_fit(kpi_name, kpi_value)
            else:
                # print(f"Adding timestep data for array KPI: {kpi_name}")
                # print(f"Current value for {kpi_name}: {self._safe_get_attr(timestep_tuple, kpi_name)}")
                users_data = self._safe_get_attr(timestep_tuple, kpi_name)
                for user_data in users_data:
                    kpi_value = user_data
                    # Get the current timestep index for this array KPI
                    current_timestep_index = self.array_current_timestep_index.get(kpi_name, -1)
                    specific_value = kpi_value[current_timestep_index]
                    
                    self.quantile_manager.partial_fit(kpi_name, specific_value)
                    
                    # Calculate and fit slope to quantile manager
                    slope_kpi_name = f"{kpi_name}_slope"  # e.g., 'dl_tput_all_slope'
                    
                    # Get non-zero values for slope calculation
                    non_zero_values = [v for v in kpi_value if v != 0.0]
                    
                    if len(non_zero_values) >= 1:  # Need at least 2 non-zero values to calculate slope
                        x = np.arange(len(non_zero_values))
                        slope, _, _, _, _ = linregress(x, non_zero_values)
                        self.slopes_hist[kpi_name].append(slope)
                        # Fit slope to slope's quantile manager
                        self.quantile_manager.partial_fit(slope_kpi_name, slope)
                        # print(f"Fitting slope {slope} from {non_zero_values} for {slope_kpi_name}")
    
    
    
    def _safe_get_attr(self, tuple_obj, attr_name):
        """
        Safely get an attribute from a tuple object, handling lowercase conversion and missing attributes.
        
        :param tuple_obj: The namedtuple object
        :param attr_name: The attribute name to get
        :return: The attribute value or None if not found
        """
        if tuple_obj is None:
            return None
            
        # Try the original attribute name
        if hasattr(tuple_obj, attr_name):
            return getattr(tuple_obj, attr_name)
        
        # Try lowercase version
        attr_name_lower = attr_name.lower()
        if hasattr(tuple_obj, attr_name_lower):
            return getattr(tuple_obj, attr_name_lower)
        
        # If both attempts fail, return None
        return None


In [ ]:
import re
from typing import Dict, Any, Optional, Tuple, List

import networkx as nx
from pyvis.network import Network

class KnowledgeGraph:
    def __init__(self, state_kpis: List[str]) -> None:
        """
        Initializes the KnowledgeGraph.
        
        Args:
            state_kpis (List[str]): A list of keys from the symbolic form dictionary that will be used to create the state and state_id.
        """
        self.G = nx.DiGraph()
        self.net = None
        self.state_kpis = state_kpis
        
        if not isinstance(state_kpis, list) or not all(isinstance(kpi, str) for kpi in state_kpis):
            raise TypeError("state_kpis must be a list of strings.")

    def _extract_state(self, symbolic_form_dict: Dict[str, Any]) -> Tuple[Tuple, str]:
        """Extracts the current state and state_id based on the configured KPIs."""
        current_state = tuple(symbolic_form_dict.get(kpi) for kpi in self.state_kpis)
        state_id = "_".join(str(x) for x in current_state if x is not None)
        return current_state, state_id

    def _parse_action(self, action_str: str) -> Optional[float]:
        """Parses bitrate from action string using regex."""
        match = re.search(r"[-+]?\d*\.\d+|\d+", action_str)  # Matches floats or integers
        return float(match.group()) if match else None

    def update_graph(self, current_symbolic_form: Dict[str, Any], previous_symbolic_form: Optional[Dict[str, Any]] = None, print_log: bool = False) -> None:
        """
        Updates the graph with a transition from previous state to current state.
        
        Args:
            current_symbolic_form: The current symbolic form dictionary
            previous_symbolic_form: The previous symbolic form dictionary, or None if this is the first state
            print_log: Whether to print detailed logs of the graph updates (default: False)
        """
        # Extract current state information
        current_state, current_state_id = self._extract_state(current_symbolic_form)
        
        # Use current reward only for node creation
        current_reward = 0
        if previous_symbolic_form:
            current_reward = previous_symbolic_form.get("reward", 0)
        
        # Add current state node if it doesn't exist
        is_new_node = current_state_id not in self.G.nodes
        self.G.add_node(
            current_state_id, 
            state=current_state, 
            occurrence=self.G.nodes.get(current_state_id, {}).get('occurrence', 0) + 1,
            total_reward=self.G.nodes.get(current_state_id, {}).get('total_reward', 0.0) + current_reward
        )
        node_data = self.G.nodes[current_state_id]
        node_data['mean_reward'] = node_data['total_reward'] / node_data['occurrence']
        
        # Print node information if logging is enabled
        if print_log and is_new_node:
            print(f"🟢 Added new node: {current_state_id}")
            print(f"   State: {current_state}")
            print(f"   Reward: {current_reward:.3f}")
        elif print_log:
            print(f"🔄 Updated node: {current_state_id}")
            print(f"   Occurrences: {node_data['occurrence']}")
            print(f"   Mean reward: {node_data['mean_reward']:.3f}")

        # If there's a previous state, add a transition
        if previous_symbolic_form:
            # Extract previous state information
            prev_state, previous_state_id = self._extract_state(previous_symbolic_form)
            
            # CHANGE #1: Get reward from the previous state
            prev_reward = previous_symbolic_form.get("reward", 0)
            
            # CHANGE #2: Get action from the previous state
            action = previous_symbolic_form.get("action", "unknown")
            
            # Check if this is a new edge
            is_new_edge = not self.G.has_edge(previous_state_id, current_state_id)
            
            # Add edge data using previous state's reward
            edge_data = self.G.get_edge_data(previous_state_id, current_state_id, default={})
            edge_data['occurrence'] = edge_data.get('occurrence', 0) + 1
            
            # Use previous state's reward for the edge
            edge_data['total_reward'] = edge_data.get('total_reward', 0.0) + prev_reward
            edge_data['mean_reward'] = edge_data['total_reward'] / edge_data['occurrence']
            
            # Update action data with previous state's action
            actions = edge_data.setdefault('actions', {})
            actions.setdefault(action, {'count': 0, 'total_reward': 0.0})['count'] += 1
            actions[action]['total_reward'] += prev_reward
            
            for act_data in actions.values():
                act_data['mean_reward'] = act_data['total_reward'] / act_data['count']
                act_data['probability'] = act_data['count'] / edge_data['occurrence']
            
            self.G.add_edge(previous_state_id, current_state_id, **edge_data)
            
            # Print edge information if logging is enabled
            if print_log and is_new_edge:
                print(f"➡️ Added new edge: {previous_state_id} → {current_state_id}")
                print(f"   Action: {action}")
                print(f"   Reward: {prev_reward:.3f}")
            elif print_log:
                print(f"🔄 Updated edge: {previous_state_id} → {current_state_id}")
                print(f"   Occurrences: {edge_data['occurrence']}")
                print(f"   Action: {action} (count: {actions[action]['count']})")
                print(f"   Mean reward: {edge_data['mean_reward']:.3f}")
        elif print_log:
            print(f"ℹ️ No previous state for this transition")
        
        if print_log:
            print("-" * 40)  # Separator for readability

    def get_recommendation(self, symbolic_form: Dict[str, Any]) -> Optional[float]:
        _, state_id = self._extract_state(symbolic_form)
        current_action = symbolic_form.get("sel_brate")
        current_bitrate = self._parse_action(current_action)

        if state_id not in self.G.nodes:
            return None

        best_alt_action = None
        best_alt_reward = -float('inf')

        for _, next_node, edge_data in self.G.out_edges(state_id, data=True):
            if 'actions' in edge_data:
                for action, data in edge_data['actions'].items():
                    if action != current_action and self._parse_action(action) == current_bitrate:
                        if data['mean_reward'] > best_alt_reward:
                            best_alt_reward = data['mean_reward']
                            best_alt_action = action

        return self._parse_action(best_alt_action) if best_alt_action else None

    def _update_probabilities_and_sizes(self):
        total_node_occurrence = sum(nx.get_node_attributes(self.G, 'occurrence').values())
        if total_node_occurrence == 0:
            return

        for node, data in self.G.nodes(data=True):
            prob = data['occurrence'] / total_node_occurrence
            data['probability'] = prob
            data['size'] = 30 + 120 * (prob**0.5)

        for u, v, data in self.G.edges(data=True):
            total_transitions_from_u = sum(self.G[u][nbr]['occurrence'] for nbr in self.G.successors(u))
            data['probability'] = data['occurrence'] / total_transitions_from_u if total_transitions_from_u > 0 else 0
            data['width'] = 1 + 5 * data['probability']

    def build_graph(self):
        self._update_probabilities_and_sizes()

        self.net = Network(height="1500px", width="100%", bgcolor="#222222", font_color="white",
                           directed=True, notebook=True, filter_menu=True, select_menu=True, cdn_resources="in_line")
        self.net.from_nx(self.G)

        for node in self.net.nodes:
            data = self.G.nodes[node['id']]
            node['title'] = f"State: {data.get('state')} \nOccurrence: {data.get('occurrence')} \n" \
                            f"Mean Reward: {data.get('mean_reward'):.2f} \nProbability: {100 * data.get('probability', 0):.1f}%"

        for edge in self.net.edges:
            u, v = edge['from'], edge['to']
            data = self.G[u][v]
            action_info = "\nActions:\n"
            for action, act_data in data.get('actions', {}).items():
                action_info += f"  {action}: Count: {act_data['count']}, Mean Reward: {act_data['mean_reward']:.2f}, Probability: {act_data['probability']:.2f}\n"

            edge['title'] = f"Transition: {u} -> {v}\nOccurrence: {data.get('occurrence')}\nMean Reward: {data.get('mean_reward'):.2f}\nProbability: {100 * data.get('probability', 0):.1f}%{action_info}"

        num_nodes = self.G.number_of_nodes()
        num_edges = self.G.number_of_edges()
        self.net.add_node("size_info", label=f"Nodes: {num_nodes}<br>Edges: {num_edges}", shape="text", x='-95%', y=0, physics=False)

        self.net.barnes_hut(overlap=1)
        self.net.show_buttons(filter_=['physics'])

    def get_graph(self, mode="all"):
        self.build_graph()
        return self.G, self.net


In [ ]:
# # %%
# import re
# from typing import Dict, Any, Optional, Tuple, List

# import networkx as nx
# from pyvis.network import Network

# class KnowledgeGraph:
#     def __init__(self, state_kpis: List[str]) -> None:
#         """
#         Initializes the KnowledgeGraph.
        
#         Args:
#             state_kpis (List[str]): A list of keys from the symbolic form dictionary that will be used to create the state and state_id.
#         """
#         self.G = nx.DiGraph()
#         self.net = None
#         self.state_kpis = state_kpis
        
#         if not isinstance(state_kpis, list) or not all(isinstance(kpi, str) for kpi in state_kpis):
#             raise TypeError("state_kpis must be a list of strings.")

#     def _extract_state(self, symbolic_form_dict: Dict[str, Any]) -> Tuple[Tuple, str]:
#         """Extracts the current state and state_id based on the configured KPIs."""
#         current_state = tuple(symbolic_form_dict.get(kpi) for kpi in self.state_kpis)
#         state_id = "_".join(str(x) for x in current_state if x is not None)
#         return current_state, state_id

#     def _parse_action(self, action_str: str) -> Optional[float]:
#         """Parses bitrate from action string using regex."""
#         match = re.search(r"[-+]?\d*\.\d+|\d+", action_str)  # Matches floats or integers
#         return float(match.group()) if match else None

#     def update_graph(self, current_symbolic_form: Dict[str, Any], previous_symbolic_form: Optional[Dict[str, Any]] = None, print_log: bool = False) -> None:
#         """
#         Updates the graph with a transition from previous state to current state.
        
#         Args:
#             current_symbolic_form: The current symbolic form dictionary
#             previous_symbolic_form: The previous symbolic form dictionary, or None if this is the first state
#             print_log: Whether to print detailed logs of the graph updates (default: False)
#         """
#         # Extract current state information
#         current_state, current_state_id = self._extract_state(current_symbolic_form)
#         current_reward = current_symbolic_form.get("reward", 0)
#         action = current_symbolic_form.get("action", "unknown")
        
#         # Add current state node if it doesn't exist
#         is_new_node = current_state_id not in self.G.nodes
#         self.G.add_node(
#             current_state_id, 
#             state=current_state, 
#             occurrence=self.G.nodes.get(current_state_id, {}).get('occurrence', 0) + 1,
#             total_reward=self.G.nodes.get(current_state_id, {}).get('total_reward', 0.0) + current_reward
#         )
#         node_data = self.G.nodes[current_state_id]
#         node_data['mean_reward'] = node_data['total_reward'] / node_data['occurrence']
        
#         # Print node information if logging is enabled
#         if print_log and is_new_node:
#             print(f"🟢 Added new node: {current_state_id}")
#             print(f"   State: {current_state}")
#             print(f"   Reward: {current_reward:.3f}")
#         elif print_log:
#             print(f"🔄 Updated node: {current_state_id}")
#             print(f"   Occurrences: {node_data['occurrence']}")
#             print(f"   Mean reward: {node_data['mean_reward']:.3f}")

#         # If there's a previous state, add a transition
#         if previous_symbolic_form:
#             _, previous_state_id = self._extract_state(previous_symbolic_form)
            
#             # Check if this is a new edge
#             is_new_edge = not self.G.has_edge(previous_state_id, current_state_id)
            
#             # Add edge data
#             edge_data = self.G.get_edge_data(previous_state_id, current_state_id, default={})
#             edge_data['occurrence'] = edge_data.get('occurrence', 0) + 1
#             edge_data['total_reward'] = edge_data.get('total_reward', 0.0) + current_reward
#             edge_data['mean_reward'] = edge_data['total_reward'] / edge_data['occurrence']
            
#             # Update action data
#             actions = edge_data.setdefault('actions', {})
#             actions.setdefault(action, {'count': 0, 'total_reward': 0.0})['count'] += 1
#             actions[action]['total_reward'] += current_reward
            
#             for act_data in actions.values():
#                 act_data['mean_reward'] = act_data['total_reward'] / act_data['count']
#                 act_data['probability'] = act_data['count'] / edge_data['occurrence']
            
#             self.G.add_edge(previous_state_id, current_state_id, **edge_data)
            
#             # Print edge information if logging is enabled
#             if print_log and is_new_edge:
#                 print(f"➡️ Added new edge: {previous_state_id} → {current_state_id}")
#                 print(f"   Action: {action}")
#                 print(f"   Reward: {current_reward:.3f}")
#             elif print_log:
#                 print(f"🔄 Updated edge: {previous_state_id} → {current_state_id}")
#                 print(f"   Occurrences: {edge_data['occurrence']}")
#                 print(f"   Action: {action} (count: {actions[action]['count']})")
#                 print(f"   Mean reward: {edge_data['mean_reward']:.3f}")
#         elif print_log:
#             print(f"ℹ️ No previous state for this transition")
        
#         if print_log:
#             print("-" * 40)  # Separator for readability

#     def get_recommendation(self, symbolic_form: Dict[str, Any]) -> Optional[float]:
#         _, state_id = self._extract_state(symbolic_form)
#         current_action = symbolic_form.get("sel_brate")
#         current_bitrate = self._parse_action(current_action)

#         if state_id not in self.G.nodes:
#             return None

#         best_alt_action = None
#         best_alt_reward = -float('inf')

#         for _, next_node, edge_data in self.G.out_edges(state_id, data=True):
#             if 'actions' in edge_data:
#                 for action, data in edge_data['actions'].items():
#                     if action != current_action and self._parse_action(action) == current_bitrate:
#                         if data['mean_reward'] > best_alt_reward:
#                             best_alt_reward = data['mean_reward']
#                             best_alt_action = action

#         return self._parse_action(best_alt_action) if best_alt_action else None

#     def _update_probabilities_and_sizes(self):
#         total_node_occurrence = sum(nx.get_node_attributes(self.G, 'occurrence').values())
#         if total_node_occurrence == 0:
#             return

#         for node, data in self.G.nodes(data=True):
#             prob = data['occurrence'] / total_node_occurrence
#             data['probability'] = prob
#             data['size'] = 30 + 120 * (prob**0.5)

#         for u, v, data in self.G.edges(data=True):
#             total_transitions_from_u = sum(self.G[u][nbr]['occurrence'] for nbr in self.G.successors(u))
#             data['probability'] = data['occurrence'] / total_transitions_from_u if total_transitions_from_u > 0 else 0
#             data['width'] = 1 + 5 * data['probability']

#     def build_graph(self):
#         self._update_probabilities_and_sizes()

#         self.net = Network(height="1500px", width="100%", bgcolor="#222222", font_color="white",
#                            directed=True, notebook=True, filter_menu=True, select_menu=True, cdn_resources="in_line")
#         self.net.from_nx(self.G)

#         for node in self.net.nodes:
#             data = self.G.nodes[node['id']]
#             node['title'] = f"State: {data.get('state')} \nOccurrence: {data.get('occurrence')} \n" \
#                             f"Mean Reward: {data.get('mean_reward'):.2f} \nProbability: {100 * data.get('probability', 0):.1f}%"

#         for edge in self.net.edges:
#             u, v = edge['from'], edge['to']
#             data = self.G[u][v]
#             action_info = "\nActions:\n"
#             for action, act_data in data.get('actions', {}).items():
#                 action_info += f"  {action}: Count: {act_data['count']}, Mean Reward: {act_data['mean_reward']:.2f}, Probability: {act_data['probability']:.2f}\n"

#             edge['title'] = f"Transition: {u} -> {v}\nOccurrence: {data.get('occurrence')}\nMean Reward: {data.get('mean_reward'):.2f}\nProbability: {100 * data.get('probability', 0):.1f}%{action_info}"

#         num_nodes = self.G.number_of_nodes()
#         num_edges = self.G.number_of_edges()
#         self.net.add_node("size_info", label=f"Nodes: {num_nodes}<br>Edges: {num_edges}", shape="text", x='-95%', y=0, physics=False)

#         self.net.barnes_hut(overlap=1)
#         self.net.show_buttons(filter_=['physics'])

#     def get_graph(self, mode="all"):
#         self.build_graph()
#         return self.G, self.net

# Constants

In [ ]:
# Define the KPI approximators
kpi_approximators = {
    'mase': (
        TDigestQuantileApproximator,
        {'quantiles': [0.2, 0.4, 0.6, 0.8]}
    ),
    # 'dtu': (
    #     TDigestQuantileApproximator,
    #     {'quantiles': [0.2, 0.4, 0.6, 0.8]}
    # ),
    'ddtu': (
        TDigestQuantileApproximator,
        {'quantiles': [0.2, 0.4, 0.6, 0.8]}
    ),
    'rddtu_a': (
        TDigestQuantileApproximator,
        {'quantiles': [0.2, 0.4, 0.6, 0.8]}
    ),
    'rddtu_b': (
        TDigestQuantileApproximator,
        {'quantiles': [0.2, 0.4, 0.6, 0.8]}
    ),
    'mase_all_slope': (
        TDigestQuantileApproximator,
        {'quantiles': [0.4, 0.6]} # Using p20 and p80 for thresholds
    ),
}

# Helper Functions

In [ ]:
def extract_features(df):
    """
    Extract mase, dtu, group information from the observation column and calculate
    ddtu and relative transmission speed metrics.
    Also calculate ratio to mean for both ddtu and mase values, including zero values in calculations.
    Added mase_all column that combines mase, mase1, and mase2 for each user.

    Args:
        df (pd.DataFrame): DataFrame containing an 'observation' column with arrays of 21 values

    Returns:
        pd.DataFrame: The original DataFrame with additional columns for all metrics
    """
    # Create a copy to avoid modifying the original DataFrame
    result_df = df.copy()

    # Function to safely convert string representation of array to actual array
    def parse_array(array_str):
        # Handle potential formatting issues in the string representation
        if isinstance(array_str, str):
            try:
                # Try to use ast.literal_eval which is safer than eval
                return ast.literal_eval(array_str)
            except (SyntaxError, ValueError):
                # If that fails, try to extract numbers using regex
                numbers = re.findall(r"[-+]?\d*\.\d+|\d+", array_str)
                return [float(num) for num in numbers]
        elif isinstance(array_str, list):
            # If it's already a list, return it directly
            return array_str
        else:
            # Return empty list for any other case
            return []

    # Apply the parsing function to the observation column
    # This creates a Series of lists that we can work with
    parsed_observations = result_df['observation'].apply(parse_array)

    # Check if the arrays have the expected length of 21
    # If not, we'll pad or truncate them
    def ensure_length(arr, target_length=35):
        if len(arr) < target_length:
            return arr + [0.0] * (target_length - len(arr))
        return arr[:target_length]

    parsed_observations = parsed_observations.apply(lambda x: ensure_length(x))

    # Extract mase (first 7 values) and round to 3 decimal places
    result_df['mase'] = parsed_observations.apply(lambda x: [round(val, 3) for val in x[:7]])
    result_df['mase1'] = parsed_observations.apply(lambda x: [round(val, 3) for val in x[7:14]])
    result_df['mase2'] = parsed_observations.apply(lambda x: [round(val, 3) for val in x[14:21]])

    # Create mase_all column with combined mase values for each user
    result_df['mase_all'] = [[] for _ in range(len(result_df))]
    for i in range(len(result_df)):
        mase = result_df.iloc[i]['mase']
        mase1 = result_df.iloc[i]['mase1']
        mase2 = result_df.iloc[i]['mase2']
        mase_all = []
        for j in range(len(mase)):
            mase_all.append([mase[j], mase1[j], mase2[j]])
        result_df.at[i, 'mase_all'] = mase_all

    # Extract dtu (next 7 values) and round to 3 decimal places
    result_df['dtu'] = parsed_observations.apply(lambda x: [round(val, 3) for val in x[21:28]])

    # Extract group (last 7 values) and convert to integers
    result_df['group'] = parsed_observations.apply(lambda x: [int(val) for val in x[28:35]])

    # Add a timestep column
    result_df['timestep'] = range(1, len(result_df) + 1)

    # Calculate delta dtu (ddtu)
    # Initialize with empty lists
    result_df['ddtu'] = [[] for _ in range(len(result_df))]

    # For each timestep, calculate the delta from previous timestep
    for i in range(len(result_df)):
        current_dtu = result_df.iloc[i]['dtu']

        if i == 0:  # First timestep
            result_df.at[i, 'ddtu'] = [0.0] * len(current_dtu)
        else:
            prev_dtu = result_df.iloc[i-1]['dtu']
            # Calculate delta for each user
            deltas = []
            for j in range(len(current_dtu)):
                # If current DTU is less than previous DTU, a reset occurred
                # In this case, use current DTU as the delta (assuming it reset to 0)
                if current_dtu[j] < prev_dtu[j]:
                    deltas.append(round(current_dtu[j], 3))
                else:
                    deltas.append(round(current_dtu[j] - prev_dtu[j], 3))
            result_df.at[i, 'ddtu'] = deltas

    # Initialize columns for relative metrics
    result_df['rmase'] = [[] for _ in range(len(result_df))]  # Ratio to mean for mase
    result_df['rddtu_a'] = [[] for _ in range(len(result_df))]  # Ratio to mean for ddtu
    result_df['rddtu_b'] = [[] for _ in range(len(result_df))]  # Percentile ranking

    # Process each timestep to calculate relative metrics
    for i in range(len(result_df)):
        ddtu_values = result_df.iloc[i]['ddtu']
        mase_values = result_df.iloc[i]['mase']

        # Calculate relative metrics for mase
        valid_mase = [val for val in mase_values if val is not None]  # Include zeros, only filter None values
        if valid_mase:
            mean_mase = sum(valid_mase) / len(valid_mase)  # This mean now includes zeros
            rmase_values = []
            for val in mase_values:
                if val is not None and mean_mase != 0:  # Still need to check mean != 0 to avoid division by zero
                    rmase_values.append(round(val / mean_mase, 3))
                else:
                    rmase_values.append(0.0)  # For None values or if mean is zero
            result_df.at[i, 'rmase'] = rmase_values
        else:
            result_df.at[i, 'rmase'] = [0.0] * len(mase_values)

        # Skip rows with invalid or empty ddtu values
        if not isinstance(ddtu_values, list) or len(ddtu_values) == 0:
            result_df.at[i, 'rddtu_a'] = [0.0] * len(ddtu_values)
            result_df.at[i, 'rddtu_b'] = [0.0] * len(ddtu_values)
            continue

        # Filter out None values but include zeros for calculation
        valid_values = [val for val in ddtu_values if val is not None]  # Changed to include zeros

        # If there are no valid values, skip this row
        if not valid_values:
            result_df.at[i, 'rddtu_a'] = [0.0] * len(ddtu_values)
            result_df.at[i, 'rddtu_b'] = [0.0] * len(ddtu_values)
            continue

        # Calculate metrics including zeros
        mean_ddtu = sum(valid_values) / len(valid_values)  # Mean now includes zeros

        # Percentile ranking preparation - include zeros in the ranking
        value_index_pairs = [(val, idx) for idx, val in enumerate(ddtu_values) if val is not None]
        sorted_pairs = sorted(value_index_pairs, key=lambda x: x[0])
        ranks = {}
        for rank, (_, idx) in enumerate(sorted_pairs):
            ranks[idx] = round((rank + 1) / len(sorted_pairs) * 100, 3)  # Convert to percentile and round to 3 decimal places

        # Calculate all metrics for each user
        rddtu_a_values = []
        rddtu_b_values = []

        for j in range(len(ddtu_values)):
            val = ddtu_values[j]

            # Calculate simple ratio to mean - handle zeros differently
            if val is not None and mean_ddtu != 0:  # Only check for None and division by zero
                ratio = val / mean_ddtu
            else:
                ratio = 0

            # Get percentile rank (method b)
            percentile = ranks.get(j, 0) if val is not None else 0
            
            rddtu_a_values.append(round(ratio, 3))
            rddtu_b_values.append(round(percentile, 3))  # Changed from 1 to 3 decimal places

        result_df.at[i, 'rddtu_a'] = rddtu_a_values
        result_df.at[i, 'rddtu_b'] = rddtu_b_values

    # Drop the observation column
    result_df.drop(columns=['observation'], inplace=True)

    # Reorder columns
    result_df = result_df[['timestep', 'mase', 'mase1', 'mase2', 'mase_all', 'ddtu', 'rddtu_a', 'rddtu_b', 'group', 'action', 'reward']]

    return result_df

def load_and_process_data(data_path):
    """
    Load and process the dataframe from CSV.

    Args:
        data_path (str): Path to the CSV file

    Returns:
        pd.DataFrame: Processed DataFrame with all extracted features
    """
    # Load data
    df = pd.read_csv(data_path)

    # Process dataframe
    processed_df = extract_features(df)

    return processed_df

def flatten_features(df, kpi_columns=['mase', 'mase1', 'mase2', 'mase_all', 'ddtu', 'rddtu_a', 'rddtu_b', 'group']):
    """
    Convert the list columns into individual columns for easier analysis
    and reorder columns to match the updated structure.

    Args:
        df (pd.DataFrame): DataFrame containing columns with lists to be flattened
        kpi_columns (list): List of column names to flatten

    Returns:
        pd.DataFrame: A dataframe with flattened features and reordered columns
    """
    # Validate that all requested KPI columns exist in the dataframe
    for col in kpi_columns:
        if col not in df.columns:
            raise ValueError(f"Column '{col}' not found in the DataFrame")

    flat_df = df.copy()

    # Dict to store the flattened column names for each KPI
    flattened_cols_by_kpi = {}

    # Flatten each KPI column
    for kpi in kpi_columns:
        flattened_cols = []
        for i in range(7):  # Assuming 7 users
            new_col = f'{kpi}_user_{i+1}'
            flat_df[new_col] = flat_df[kpi].apply(lambda x: x[i] if isinstance(x, list) and len(x) > i else None)
            flattened_cols.append(new_col)

        flattened_cols_by_kpi[kpi] = flattened_cols

    # Drop the original list columns
    flat_df.drop(columns=kpi_columns, inplace=True)

    # Prepare column order
    timestep_cols = ['timestep'] if 'timestep' in flat_df.columns else []

    # Build the ordered column list based on the KPIs provided
    ordered_columns = timestep_cols

    # Add flattened columns in the order specified by kpi_columns
    for kpi in kpi_columns:
        ordered_columns.extend(flattened_cols_by_kpi[kpi])

    # Additional columns that should be at the end
    action_reward_cols = []
    if 'action' in flat_df.columns:
        action_reward_cols.append('action')
    if 'reward' in flat_df.columns:
        action_reward_cols.append('reward')

    ordered_columns.extend(action_reward_cols)

    # Make sure we only include columns that actually exist in the dataframe
    ordered_columns = [col for col in ordered_columns if col in flat_df.columns]

    # Add any remaining columns not explicitly ordered
    remaining_cols = [col for col in flat_df.columns if col not in ordered_columns]
    all_columns = ordered_columns + remaining_cols

    # Return reordered dataframe
    return flat_df[all_columns]

def preprocess(row, average_by_group=False):
    """
    Preprocess a row of data into a nested dictionary organized by group.

    Args:
        row: A namedtuple row from data_trace.itertuples()
        average_by_group: If True, return average values for each group instead of individual user values

    Returns:
        dict: A nested dictionary with structure:
            If average_by_group=False:
            {
                group_id: {
                    'mase': {user_idx: value, ...},
                    'dtu': {user_idx: value, ...},
                    'group_members': [user_idx1, user_idx2, ...]
                },
                ...
            }

            If average_by_group=True:
            {
                group_id: {
                    'mase': average_value (rounded to 3 decimal places),
                    'dtu': average_value (rounded to 3 decimal places),
                    'group_members': [user_idx1, user_idx2, ...]
                },
                ...
            }
    """
    # Extract MASE, DTU, and group values from the row
    mase_values = row.mase if hasattr(row, 'mase') else []
    dtu_values = row.dtu if hasattr(row, 'dtu') else []
    group_values = row.group if hasattr(row, 'group') else []

    # Create a dictionary to hold our result
    result = {}

    # First, collect all group members regardless of averaging mode
    group_members = {}
    for user_idx, group in enumerate(group_values, 1):
        if group not in group_members:
            group_members[group] = []
        group_members[group].append(user_idx)

    if not average_by_group:
        # Organize data by group, with individual user values
        for user_idx, (mase, dtu, group) in enumerate(zip(mase_values, dtu_values, group_values), 1):
            # Initialize the group entry if it doesn't exist
            if group not in result:
                result[group] = {
                    'mase': {},
                    'dtu': {},
                    'group_members': group_members[group]
                }

            # Add MASE and DTU values for this user to the appropriate group
            result[group]['mase'][user_idx] = mase
            result[group]['dtu'][user_idx] = dtu
    else:
        # Initialize data structures for averaging
        group_totals = {}  # {group: {'mase_sum': value, 'dtu_sum': value, 'count': value}}

        # Collect data for averaging
        for user_idx, (mase, dtu, group) in enumerate(zip(mase_values, dtu_values, group_values), 1):
            if group not in group_totals:
                group_totals[group] = {'mase_sum': 0, 'dtu_sum': 0, 'count': 0}

            group_totals[group]['mase_sum'] += mase
            group_totals[group]['dtu_sum'] += dtu
            group_totals[group]['count'] += 1

        # Calculate averages and format result
        for group, data in group_totals.items():
            count = data['count']
            if count > 0:  # Avoid division by zero
                result[group] = {
                    'mase': round(data['mase_sum'] / count, 3),
                    'dtu': round(data['dtu_sum'] / count, 3),
                    'group_members': group_members[group]
                }
            else:
                result[group] = {
                    'mase': 0,
                    'dtu': 0,
                    'group_members': group_members[group]
                }

    return result

In [ ]:
def create_visualization_folder_path(vis_name, dataset_name, specific_name=""):
    """
    Creates the folder path for storing visualization figures and ensures the path exists.
    Includes an optional specific_name in the path.

    Args:
        vis_name (str): The name of the visualization type (folder name).
        dataset_name (str): The name of the dataset (subfolder name).
        specific_name (str, optional): An optional specific name to include in the folder path. Defaults to "".

    Returns:
        str: The full path to the visualization folder.
    """
    base_dir = "user_fr_visualizations"
    folder_path = os.path.join(base_dir, vis_name, dataset_name)
    if specific_name:  # Add specific_name to path if it's not empty
        folder_path = os.path.join(folder_path, specific_name)
    os.makedirs(folder_path, exist_ok=True)  # Create directory if it doesn't exist
    return folder_path

def setup_symbolic_components(kpi_approximators, quantile_markers_path=None, kpi_change_threshold=5):
    """
    Sets up the symbolic components: QuantileManager, Symbolizer, and KnowledgeGraph.

    Args:
        quantile_markers_path (str): Path to the CSV file with quantile markers.
        kpi_change_threshold (int): Threshold for KPI change percentage.

    Returns:
        tuple: QuantileManager, Symbolizer, and KnowledgeGraph objects.
    """
     # Map the column names in your dataframe to the names expected by the symbolizer
    kpis = {
        'mase': 'mase',
        'mase_all': 'mase_all',
        'ddtu': 'ddtu',
        'rddtu_a': 'rddtu_a',
        'rddtu_b': 'rddtu_b',
    }

    decisions = {
        'action': 'action'
    }


    qm = QuantileManager(kpi_approximators)
    qm.reset()
    if quantile_markers_path:
        qm.load_markers(pd.read_csv(quantile_markers_path))

    # Initialize the symbolizer
    symbolizer = Symbolizer(
        quantile_manager=qm,
        kpis=kpis,
        decisions=decisions,
        kpi_change_threshold_percent=kpi_change_threshold
    )

    # Define which KPIs to use for the Knowledge Graph state
    rt_kg = { # Define rt_kg dictionary for each KPI
        'mase': KnowledgeGraph(["mase"]),
        "dtu": KnowledgeGraph(["dtu"]),
    }

    print('Symbolic components initialized.')
    return qm, symbolizer, rt_kg

def train_quantile_manager(data_trace: pd.DataFrame, quantile_manager, kpi_approximators):
    """
    Train the quantile manager on the provided data trace.

    Args:
        data_trace (pd.DataFrame): DataFrame containing the data to process
        quantile_manager: Quantile manager to be trained
        kpi_approximators (dict): Dictionary of KPI approximators

    Returns:
        pd.DataFrame: DataFrame of markers generated during training
    """
    markers_df_list = []
    total_rows = len(data_trace)
    
    # For storing slope history of array KPIs
    array_kpis = {'mase_all_slope': 'mase_all'}
    slopes_hist = {kpi: [] for kpi in array_kpis}
    prev_row = None

    for i, row in enumerate(data_trace.itertuples(), 1):
        # Print progress on the same line
        print(f"\rProcessing row {i}/{total_rows} ({i/total_rows*100:.2f}%)", end='', flush=True)
        # print(f"Processing row {i}/{total_rows} ({i/total_rows*100:.2f}%)")
        
        # Regular KPI training
        for kpi in kpi_approximators.keys():
            if kpi not in array_kpis:
                # Regular scalar KPIs - fit directly
                kpi_values = getattr(row, kpi)
                quantile_manager.partial_fit(kpi, kpi_values)
            else:
                kpi_name = array_kpis[kpi]
                for user_idx, kpi_array in enumerate(getattr(row, kpi_name)):
                    # Calculate and fit slope for array KPIs
                    slope_kpi_name = f"{kpi_name}_slope"
                    # Only calculate slope if we have enough non-zero values
                    non_zero_values = [v for v in kpi_array if v != 0.0]
                    if len(non_zero_values) >= 2:
                        x = np.arange(len(non_zero_values))
                        slope, _, _, _, _ = linregress(x, non_zero_values)
                        
                        # Store slope history
                        slopes_hist[kpi].append(slope)
                        
                        # Fit slope to quantile manager
                        quantile_manager.partial_fit(slope_kpi_name, round(slope, 3))
        # Get markers at each timestep and add timestep information
        markers_df = quantile_manager.represent_markers()
        markers_df['timestep'] = row.timestep

        # Append this timestep's markers
        markers_df_list.append(markers_df)
        # print("-" * 40)
        # if i > 20:
        #     break

    # Print a newline after processing is complete
    print()

    # Concatenate all markers dataframes
    return pd.concat(markers_df_list, ignore_index=True)

def run_agent(data_trace: pd.DataFrame, symbolizer, rt_kg):
    """
    Runs the agent in the environment, collects data, and updates Knowledge Graph.

    Args:
        env (ABREnv): The ABR environment.
        actor (network.Network): The PPO agent.
        symbolizer (Symbolizer): The Symbolizer object.
        rt_kg (KnowledgeGraph): The KnowledgeGraph dictionary.

    Returns:
        tuple: DataFrames for raw and symbolic data, arrays for scores, average bitrates, total rebuffer, and markers_df.
    """
    symbolic_df = []
    raw_df = []
    marker_df_list = []  # Initialize list for marker dataframes

    counter = 0

    total_rows = len(data_trace)
    for i, row in enumerate(data_trace.itertuples(), 1):
        print(f"\rProcessing row {i}/{total_rows} ({i/total_rows*100:.2f}%)", end='', flush=True)
        # Get preprocessed data by group
        preprocessed_data = preprocess(row, average_by_group=True)

        # Get symbolic forms for each group
        symbolic_forms = symbolizer.create_symbolic_form(row)
        symbolic_forms[0]['group'] = row.group
        symbolic_df.extend(symbolic_forms)

        # Get the markers from the quantile manager and append to the list
        marker_data = symbolizer.quantile_manager.represent_markers()
        marker_data['timestep'] = row.timestep
        marker_df_list.append(marker_data)
        # counter += 1
        # print("-" * 40)
        if counter > 20:
            break

    symbolic_df = pd.DataFrame(symbolic_df)
    markers_df = pd.concat(marker_df_list)

    return symbolic_df, markers_df


# Run Experiment

In [ ]:
# Load and process the data with all KPIs calculated
test_df = load_and_process_data('./data/sac_testfc_results_erfan.csv')
# Flatten all KPIs (by default it will include the new rddtu metrics)
test_flat_df = flatten_features(test_df)

# Load and process the data with all KPIs calculated
train_df = load_and_process_data('./data/sac_trainfc_results_erfan.csv')
# Flatten all KPIs (by default it will include the new rddtu metrics)
train_flat_df = flatten_features(train_df)

### train Approximator

In [ ]:
qm, symbolizer, rt_kg = setup_symbolic_components(kpi_approximators) 

markers_df = train_quantile_manager(train_df, qm, kpi_approximators)
qm_exported_df = qm.export_markers()
qm_exported_df.to_csv('./data/train_dataset_user_with_fr_qm_export.csv')

### Run the analysis on agent's data

In [ ]:
qm, symbolizer, rt_kg = setup_symbolic_components(kpi_approximators, quantile_markers_path='./data/train_dataset_user_with_fr_qm_export.csv') 

train_symbolic_df,train_markers_df = run_agent(train_df, symbolizer, rt_kg)
test_symbolic_df, test_markers_df = run_agent(test_df, symbolizer, rt_kg)

# Visualizations

## Plot of Numerical KPI per user

In [ ]:
def plot_individual_user_metrics(test_df, kpis=['mase', 'ddtu'], dataset_name="test_data"):
    """
    Creates individual plots for each user with specified KPIs.

    Parameters:
        test_df (pd.DataFrame): DataFrame containing timestep and KPI columns with arrays of values
        kpis (list): List of KPI names to plot (e.g., ['mase', 'dtu', 'ddtu'])
        dataset_name (str): Name of the dataset for folder creation

    Returns:
        list: List of figures, one for each user
    """
    # Validate that all requested KPIs exist in the dataframe
    for kpi in kpis:
        if kpi not in test_df.columns:
            raise ValueError(f"KPI '{kpi}' not found in the DataFrame columns")

    # Create visualization folder
    vis_name = "individual_user_metrics"
    folder_path = create_visualization_folder_path(vis_name, dataset_name)

    # Define colors for each user
    colors = [
        '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728',
        '#9467bd', '#8c564b', '#e377c2'
    ]

    # Create a display name mapping for KPIs
    kpi_display_names = {
        'mase': 'MASE',
        'dtu': 'DTU',
        'ddtu': 'DDTU'
        # Add more KPIs here as needed
    }

    figures = []

    # Create individual figure for each user
    for user_idx in range(7):
        # Create n×1 subplots for this user (n = number of KPIs)
        num_kpis = len(kpis)
        subplot_titles = [f"User {user_idx+1} {kpi_display_names.get(kpi, kpi.upper())}" for kpi in kpis]

        fig = make_subplots(
            rows=num_kpis, cols=1,
            subplot_titles=subplot_titles,
            shared_xaxes=True,
            vertical_spacing=0.1
        )

        # Extract and plot each KPI for this user
        for i, kpi in enumerate(kpis):
            # Extract values for this KPI and user
            values = []
            for row in test_df.itertuples():
                kpi_values = getattr(row, kpi)
                if isinstance(kpi_values, list) and len(kpi_values) > user_idx:
                    values.append(kpi_values[user_idx])
                else:
                    values.append(None)  # Handle missing data

            # Add trace for this KPI
            fig.add_trace(
                go.Scatter(
                    x=test_df['timestep'],
                    y=values,
                    mode='lines',
                    name=kpi_display_names.get(kpi, kpi.upper()),
                    line=dict(color=colors[user_idx]),
                ),
                row=i+1, col=1
            )

            # Update y-axis label for this KPI
            fig.update_yaxes(
                title_text=f"{kpi_display_names.get(kpi, kpi.upper())} Value",
                row=i+1,
                col=1
            )

            # Add x-axis label only for the bottom plot
            if i == num_kpis - 1:
                fig.update_xaxes(title_text="Timestep", row=i+1, col=1)

        # Update layout for better visualization
        fig.update_layout(
            height=300 * num_kpis,  # Height scales with number of KPIs
            title_text=f"Metrics for User {user_idx+1} Over Time",
            hovermode='x unified',
            legend=dict(
                orientation="h",
                yanchor="bottom",
                y=-0.15,
                xanchor="center",
                x=0.5
            )
        )

        # Save the figure
        # kpi_str = "_".join(kpis)
        # png_filename = os.path.join(folder_path, f"user_{user_idx+1}_{kpi_str}_{dataset_name}.png")
        # fig.write_image(png_filename)
        # print(f"User {user_idx+1} metrics plot saved as PNG to: {png_filename}")

        # html_filename = os.path.join(folder_path, f"user_{user_idx+1}_{kpi_str}_{dataset_name}.html")
        # fig.write_html(html_filename)
        # print(f"User {user_idx+1} metrics plot saved as HTML to: {html_filename}")

        fig.show()
        # figures.append(fig)

    # return figures

# Example usage:
# Plot all three metrics
all_metrics_figures = plot_individual_user_metrics(test_df, kpis=['mase_all'], dataset_name="test_dataset")

# Plot only MASE and DDTU
# mase_ddtu_figures = plot_individual_user_metrics(test_df, kpis=['mase', 'ddtu'], dataset_name="mase_ddtu_only")

## Plot of Scheduling per user

In [ ]:
def plot_scheduling_patterns(test_df, dataset_name="test_data"):
    """
    Plots scheduling patterns for each user over time based on action data,
    with markers colored according to the user's group, and showing group colors
    for both scheduled and unscheduled time points.
    
    Parameters:
    test_df (pd.DataFrame): DataFrame containing timestep, action, and group columns
    dataset_name (str): Name of the dataset for folder creation
    """
    # Create visualization folder
    vis_name = "scheduling_patterns"
    folder_path = create_visualization_folder_path(vis_name, dataset_name)
    
    # Function to extract scheduled users from action string
    def get_scheduled_users(action_str):
        if not isinstance(action_str, str):
            return []
        matches = re.findall(r'\d+', action_str)
        return [int(m) for m in matches]
    
    # Define colors for each group
    group_colors = {
        0: '#1f77b4',  # Blue
        1: '#ff7f0e',  # Orange
        2: '#2ca02c',  # Green
        3: '#d62728',  # Red
        4: '#9467bd',  # Purple
        5: '#8c564b',  # Brown
        6: '#e377c2'   # Pink
    }
    
    # Create subplots with a line plot for each user
    fig = make_subplots(
        rows=7, cols=1,
        subplot_titles=[f"User {i+1} Scheduling" for i in range(7)],
        shared_xaxes=True,
        vertical_spacing=0.03
    )
    
    # Prepare matrices for both scheduled status and group membership
    scheduling_matrix = np.zeros((len(test_df), 7))
    group_matrix = np.zeros((len(test_df), 7), dtype=int)
    
    # Extract scheduled status and group for each user at each timestep
    for i, row in enumerate(test_df.itertuples()):
        # Get scheduled users
        scheduled_users = get_scheduled_users(row.action)
        for user in scheduled_users:
            if 0 <= user < 7:  # Ensure it's a valid user index
                scheduling_matrix[i, user] = 1
        
        # Get group for each user
        if hasattr(row, 'group') and isinstance(row.group, list):
            for user_idx in range(min(7, len(row.group))):
                group_matrix[i, user_idx] = row.group[user_idx] if row.group[user_idx] is not None else 0
    
    # First add connecting lines for each user's scheduling status
    for user_idx in range(7):
        # Add the connecting line for all points
        fig.add_trace(
            go.Scatter(
                x=test_df['timestep'],
                y=scheduling_matrix[:, user_idx],
                mode='lines',
                line=dict(color='gray', width=1),  # Use light gray for the line
                name=f'User {user_idx+1} (line)',
                showlegend=False  # Don't show in legend
            ),
            row=user_idx+1, col=1
        )
    
    # Add traces for each user's points
    for user_idx in range(7):
        # Process each group separately for this user
        for group_idx in range(3):  # Process only first 3 groups
            # Find all points where this user is in this group
            group_points = np.where(group_matrix[:, user_idx] == group_idx)[0]
            
            if len(group_points) > 0:
                # Get timesteps for these points
                timesteps = test_df.iloc[group_points]['timestep'].values
                
                # Get scheduled status for these points
                is_scheduled = scheduling_matrix[group_points, user_idx]
                
                # Points where user is in this group AND scheduled
                scheduled_points = group_points[is_scheduled == 1]
                
                # Points where user is in this group but NOT scheduled
                unscheduled_points = group_points[is_scheduled == 0]
                
                # Add trace for scheduled points
                if len(scheduled_points) > 0:
                    scheduled_timesteps = test_df.iloc[scheduled_points]['timestep'].values
                    fig.add_trace(
                        go.Scatter(
                            x=scheduled_timesteps,
                            y=np.ones(len(scheduled_points)),
                            mode='markers',
                            marker=dict(
                                color=group_colors[group_idx],
                                size=8
                            ),
                            name=f'User {user_idx+1} Group {group_idx}',
                            showlegend=True
                        ),
                        row=user_idx+1, col=1
                    )
                else:
                    # Add dummy trace for legend if no scheduled points
                    fig.add_trace(
                        go.Scatter(
                            x=[None],
                            y=[None],
                            mode='markers',
                            marker=dict(
                                color=group_colors[group_idx],
                                size=8
                            ),
                            name=f'User {user_idx+1} Group {group_idx}',
                            showlegend=True
                        ),
                        row=user_idx+1, col=1
                    )
                
                # Add trace for unscheduled points
                if len(unscheduled_points) > 0:
                    unscheduled_timesteps = test_df.iloc[unscheduled_points]['timestep'].values
                    fig.add_trace(
                        go.Scatter(
                            x=unscheduled_timesteps,
                            y=np.zeros(len(unscheduled_points)),
                            mode='markers',
                            marker=dict(
                                color=group_colors[group_idx],
                                size=8,
                                opacity=0.3  # Make unscheduled markers semi-transparent
                            ),
                            name=f'User {user_idx+1} Group {group_idx} (Not Scheduled)',
                            showlegend=False  # Don't show separate legend for unscheduled
                        ),
                        row=user_idx+1, col=1
                    )
    
    # Update layout for better visualization
    fig.update_layout(
        height=1800,  # Taller to accommodate 7 rows
        # width=1200,
        title_text="User Scheduling Patterns Over Time (Colored by Group)",
        hovermode='x unified',
        # Move legend outside the plot area with increased font size
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=-0.15,  # Move below the plot
            xanchor="center",
            x=0.5,
            font=dict(size=14),  # Increased font size for legend
            groupclick="toggleitem"
        ),
        font=dict(size=12)  # Increase the font size globally
    )
    
    # Update axes labels and ranges
    for i in range(1, 8):
        fig.update_yaxes(
            title_text="Scheduled",
            title_font=dict(size=14),  # Increased font size for y-axis title
            tickfont=dict(size=12),    # Increased font size for y-axis tick labels
            range=[-0.1, 1.1],         # Slightly expand y range for better visibility
            tickvals=[0, 1],
            ticktext=["No", "Yes"],
            row=i, col=1
        )
        
        # Only add x-axis label for the bottom plot
        if i == 7:
            fig.update_xaxes(
                title_text="Timestep", 
                title_font=dict(size=14),  # Increased font size for x-axis title
                tickfont=dict(size=12),    # Increased font size for x-axis tick labels
                row=i, col=1
            )
    
    # Update subplot titles with larger font
    for i in fig['layout']['annotations']:
        i['font'] = dict(size=16)  # Increase subplot title font size
    
    # Save and display the plot
    png_filename = os.path.join(folder_path, f"scheduling_patterns_by_user_group_{dataset_name}.png")
    fig.write_image(png_filename, height=2000, width=1200)  # Slightly taller to accommodate legend
    print(f"Scheduling patterns plot saved as PNG to: {png_filename}")
    
    html_filename = os.path.join(folder_path, f"scheduling_patterns_by_user_group_{dataset_name}.html")
    fig.write_html(html_filename)
    print(f"Scheduling patterns plot saved as HTML to: {html_filename}")
    
    # Return the figure for display
    return fig

# Example usage:
fig = plot_scheduling_patterns(test_df, "test_dataset")
fig.show()

## Visualize the Markers and the approximator effect

In [ ]:
def plot_user_with_multiple_kpis(
    flat_df: pd.DataFrame,
    markers_df: pd.DataFrame,
    kpis: List[str] = ['mase', 'dtu'],
    users: List[int] = None,  # Specify which users to plot; default is all users 1-7.
    timestep_col: str = 'timestep',
    marker_kpi_col: str = 'kpi',
    dataset_name: str = 'mimo',
    output_dir: str = './visualizations',
    specific_name: str = "",
    title: Optional[str] = None,
    do_save: bool = False
):
    """
    Creates a figure for each user showing multiple KPIs with their markers.
    Highlights timesteps where the user was scheduled in the action.

    Parameters:
        flat_df (pd.DataFrame): The flattened dataframe with individual user KPI columns.
        markers_df (pd.DataFrame): The markers dataframe with quantile data.
        kpis (List[str]): List of base KPI names to plot (without user suffix). Any "mse" will be converted to "mase".
        users (List[int]): List of user numbers to plot. If None, will plot all users (1-7).
        timestep_col (str): Name of the timestep column.
        marker_kpi_col (str): Name of the KPI column in markers_df.
        dataset_name (str): Name of the dataset for saving files.
        output_dir (str): Directory to save output files.
        specific_name (str): Optional name to include in output filenames.
        title (str): Plot title template (will be formatted with user number).
        do_save (bool): Whether to save the figures or just display them.
    """
    # Ensure that any 'mse' in kpis is replaced with 'mase'
    kpis = ['mase' if kpi.lower() == 'mse' else kpi for kpi in kpis]

    # Create output directory if saving is enabled
    if do_save:
        vis_folder = os.path.join(output_dir, f"user_multi_kpi_{dataset_name}")
        if specific_name:
            vis_folder = os.path.join(vis_folder, specific_name)
        os.makedirs(vis_folder, exist_ok=True)

    # Define plot settings
    colors = [
        '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd',
        '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf'
    ]

    # Default to plotting users 1 through 7 if not specified
    if users is None:
        users = list(range(1, 8))

    # Function to check if a user is in the action at a given timestep
    def is_user_in_action(row, user_num):
        action_str = row['action']
        if isinstance(action_str, str):
            # Parse the action string to get the list of users
            # The action is typically in format like "(0, 1, 2, 3, 5)"
            # Need to extract numbers and check if user_num-1 is in it
            # (subtract 1 since action often uses 0-indexed users)
            matches = re.findall(r'\d+', action_str)
            action_users = [int(match) for match in matches]
            return (user_num - 1) in action_users
        return False

    # Create a figure for each user with all specified KPIs
    for user_num in users:
        num_kpis = len(kpis)
        subplot_titles = [f"User {user_num} {kpi.upper()}" for kpi in kpis]

        fig = make_subplots(
            rows=num_kpis,
            cols=1,
            subplot_titles=subplot_titles,
            shared_xaxes=True,
            vertical_spacing=0.08
        )

        # Plot each KPI for the current user
        for kpi_idx, kpi_base in enumerate(kpis):
            kpi_col = f"{kpi_base}_user_{user_num}"

            # Plot the actual KPI values if the column exists
            if kpi_col in flat_df.columns:
                # Create main KPI line
                fig.add_trace(
                    go.Scatter(
                        x=flat_df[timestep_col],
                        y=flat_df[kpi_col],
                        mode='lines',
                        name=f'{kpi_base.upper()} Actual',
                        line=dict(color=colors[kpi_idx % len(colors)], width=2),
                        legendgroup=f'kpi_{kpi_base}',
                        showlegend=True
                    ),
                    row=kpi_idx+1, col=1
                )

                # Add markers for timesteps where this user is in the action
                action_timesteps = []
                action_values = []
                for idx, row in flat_df.iterrows():
                    if is_user_in_action(row, user_num):
                        action_timesteps.append(row[timestep_col])
                        action_values.append(row[kpi_col])
                
                # Only add the action markers trace if there are any points
                if action_timesteps:
                    fig.add_trace(
                        go.Scatter(
                            x=action_timesteps,
                            y=action_values,
                            mode='markers',
                            marker=dict(
                                symbol='star',
                                size=10,
                                color=colors[kpi_idx % len(colors)],
                                line=dict(width=1, color='black')
                            ),
                            name=f'Scheduled in Action',
                            legendgroup='action',
                            showlegend=(kpi_idx == 0)  # Only show in legend once
                        ),
                        row=kpi_idx+1, col=1
                    )

                # Get marker data for this KPI
                marker_data = markers_df[markers_df[marker_kpi_col] == kpi_base].copy()

                # Plot each marker line (percentiles)
                for marker_idx, marker in enumerate(['min', 'p20', 'p40', 'p60', 'p80', 'max']):
                    if marker in marker_data.columns:
                        # Use more muted colors for the marker lines
                        marker_opacity = 0.4 
                        
                        fig.add_trace(
                            go.Scatter(
                                x=marker_data[timestep_col],
                                y=marker_data[marker],
                                mode='lines',
                                name=f'{marker.upper()}',
                                line=dict(
                                    color=colors[kpi_idx % len(colors)],
                                    width=1,
                                    dash='dash'
                                ),
                                opacity=marker_opacity,
                                legendgroup=f'markers_{kpi_base}',
                                showlegend=(kpi_idx == 0 and marker_idx == 0)  # Only show one marker in legend
                            ),
                            row=kpi_idx+1, col=1
                        )

        # Improved layout settings for the figure
        height_per_kpi = 350
        fig.update_layout(
            height=height_per_kpi * num_kpis,
            title_text=title.format(user_num=user_num) if title else f"User {user_num} - Multiple KPIs with Markers",
            title_font=dict(size=20, family="Arial", color="black"),
            title_x=0.5,
            margin=dict(t=100, b=50, l=100, r=50),  # Add more space for title and legend
            legend=dict(
                orientation="h",
                yanchor="bottom",
                y=1.05,  # Position above the plot
                xanchor="center",
                x=0.5,
                bgcolor="rgba(255, 255, 255, 0.8)",  # Semi-transparent background
                bordercolor="Gray",
                borderwidth=1,
                font=dict(size=12),
                itemsizing="constant"  # Make legend items the same size
            ),
            hovermode='x unified'
        )

        # Set y-axis labels for each KPI subplot
        for i, kpi_base in enumerate(kpis):
            fig.update_yaxes(
                title_text=f"{kpi_base.upper()} Value", 
                row=i+1, 
                col=1,
                title_font=dict(size=14),
                gridcolor='rgba(200, 200, 200, 0.2)'  # Lighter grid lines
            )

        # Set x-axis label only for the bottom subplot
        fig.update_xaxes(
            title_text="Timestep", 
            row=num_kpis, 
            col=1,
            title_font=dict(size=14),
            gridcolor='rgba(200, 200, 200, 0.2)'  # Lighter grid lines
        )

        # Save or display the plot
        if do_save:
            filename_base = f"user_{user_num}_multi_kpi"
            if specific_name:
                filename_base += f"_{specific_name}"

            png_path = os.path.join(vis_folder, f"{filename_base}.png")
            fig.write_image(png_path)
            print(f"Plot saved as PNG: {png_path}")

            html_path = os.path.join(vis_folder, f"{filename_base}.html")
            fig.write_html(html_path)
            print(f"Plot saved as HTML: {html_path}")
        else:
            fig.show()

def plot_kpis_with_markers_per_user(
    flat_df: pd.DataFrame,
    markers_df: pd.DataFrame,
    kpis: List[str] = ['mase', 'dtu'],
    num_users: int = 7,
    timestep_col: str = 'timestep',
    marker_kpi_col: str = 'kpi',
    dataset_name: str = 'mimo',
    output_dir: str = './visualizations',
    specific_name: str = "",
    title: Optional[str] = None,
    do_save: bool = False
):
    """
    Plots KPI values for each user along with their markers.
    Highlights timesteps where the user was scheduled in the action.

    Parameters:
        flat_df (pd.DataFrame): The flattened dataframe with individual user KPI columns.
        markers_df (pd.DataFrame): The markers dataframe with quantile data.
        kpis (List[str]): List of base KPI names to plot (without user suffix). If "mse" is provided, it will be converted to "mase".
        num_users (int): Number of users to plot.
        timestep_col (str): Name of the timestep column.
        marker_kpi_col (str): Name of the KPI column in markers_df.
        dataset_name (str): Name of the dataset for saving files.
        output_dir (str): Directory to save output files.
        specific_name (str): Optional name to include in output filenames.
        title (str): Plot title.
        do_save (bool): Whether to save the figures or just display them.
    """
    # Ensure that any 'mse' in kpis is replaced with 'mase'
    kpis = ['mase' if kpi.lower() == 'mse' else kpi for kpi in kpis]

    # Create output directory if saving is enabled
    if do_save:
        vis_folder = os.path.join(output_dir, f"kpi_markers_{dataset_name}")
        if specific_name:
            vis_folder = os.path.join(vis_folder, specific_name)
        os.makedirs(vis_folder, exist_ok=True)

    # Define plot settings
    colors = [
        '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd',
        '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf'
    ]

    # Function to check if a user is in the action at a given timestep
    def is_user_in_action(row, user_num):
        action_str = row['action']
        if isinstance(action_str, str):
            # Parse the action string to get the list of users
            matches = re.findall(r'\d+', action_str)
            action_users = [int(match) for match in matches]
            return (user_num - 1) in action_users
        return False

    # Process each KPI type (e.g. mase, dtu, etc.)
    for kpi_base in kpis:
        # Create a subplot for each user
        fig = make_subplots(
            rows=num_users,
            cols=1,
            subplot_titles=[f"User {i+1} {kpi_base.upper()}" for i in range(num_users)],
            shared_xaxes=True,
            vertical_spacing=0.04
        )

        # Plot each user's data
        for user_idx in range(num_users):
            user_num = user_idx + 1
            kpi_col = f"{kpi_base}_user_{user_num}"

            # Plot the actual KPI values for this user if available
            if kpi_col in flat_df.columns:
                fig.add_trace(
                    go.Scatter(
                        x=flat_df[timestep_col],
                        y=flat_df[kpi_col],
                        mode='lines',
                        name=f'User {user_num} Actual',
                        line=dict(color=colors[user_idx % len(colors)], width=2),
                        legendgroup=f'user{user_num}',
                        showlegend=True
                    ),
                    row=user_idx+1, col=1
                )

                # Add markers for timesteps where this user is in the action
                action_timesteps = []
                action_values = []
                for idx, row in flat_df.iterrows():
                    if is_user_in_action(row, user_num):
                        action_timesteps.append(row[timestep_col])
                        action_values.append(row[kpi_col])
                
                # Only add the action markers trace if there are any points
                if action_timesteps:
                    fig.add_trace(
                        go.Scatter(
                            x=action_timesteps,
                            y=action_values,
                            mode='markers',
                            marker=dict(
                                symbol='star',
                                size=10,
                                color=colors[user_idx % len(colors)],
                                line=dict(width=1, color='black')
                            ),
                            name=f'User {user_num} Scheduled',
                            legendgroup=f'user{user_num}_action',
                            showlegend=(user_idx == 0)  # Only show in legend for the first user
                        ),
                        row=user_idx+1, col=1
                    )

                # Get marker data for this KPI
                marker_data = markers_df[markers_df[marker_kpi_col] == kpi_base].copy()

                # Plot each marker line (percentiles)
                for marker_idx, marker in enumerate(['min', 'p20', 'p40', 'p60', 'p80', 'max']):
                    if marker in marker_data.columns:
                        fig.add_trace(
                            go.Scatter(
                                x=marker_data[timestep_col],
                                y=marker_data[marker],
                                mode='lines',
                                name=f'{marker.upper()}',
                                line=dict(
                                    color=colors[user_idx % len(colors)],
                                    width=1,
                                    dash='dash'
                                ),
                                opacity=0.4,
                                legendgroup='markers',
                                showlegend=(user_idx == 0 and marker_idx == 0)  # Only show one in legend
                            ),
                            row=user_idx+1, col=1
                        )

        # Update layout settings
        height_per_user = 250
        fig.update_layout(
            height=height_per_user * num_users,
            title_text=title if title else f"{kpi_base.upper()} Values Per User with Markers",
            title_font=dict(size=20, family="Arial", color="black"),
            title_x=0.5,
            margin=dict(t=100, b=50, l=100, r=50),  # Add more space for title and legend
            legend=dict(
                orientation="h",
                yanchor="bottom",
                y=1.05,  # Position above the plot
                xanchor="center",
                x=0.5,
                bgcolor="rgba(255, 255, 255, 0.8)",  # Semi-transparent background
                bordercolor="Gray",
                borderwidth=1,
                font=dict(size=12),
                itemsizing="constant"  # Make legend items the same size
            ),
            hovermode='x unified'
        )

        # Update y-axis labels for each subplot
        for i in range(num_users):
            fig.update_yaxes(
                title_text=f"User {i+1} {kpi_base.upper()}", 
                row=i+1, 
                col=1,
                gridcolor='rgba(200, 200, 200, 0.2)'  # Lighter grid lines
            )

        # Set x-axis label only for the bottom subplot
        fig.update_xaxes(
            title_text="Timestep", 
            row=num_users, 
            col=1,
            gridcolor='rgba(200, 200, 200, 0.2)'  # Lighter grid lines
        )

        # Save or show the plot
        if do_save:
            filename_base = f"{kpi_base}_users_with_markers"
            if specific_name:
                filename_base += f"_{specific_name}"

            png_path = os.path.join(vis_folder, f"{filename_base}.png")
            fig.write_image(png_path)
            print(f"Plot saved as PNG: {png_path}")

            html_path = os.path.join(vis_folder, f"{filename_base}.html")
            fig.write_html(html_path)
            print(f"Plot saved as HTML: {html_path}")
        else:
            fig.show()

def plot_kpis_with_summarized_markers(
    df: pd.DataFrame,
    markers_df: pd.DataFrame,
    kpis: List[str] = ['mase', 'dtu'],
    timestep_col: str = 'timestep',
    marker_kpi_col: str = 'kpi',
    dataset_name: str = 'mimo',
    output_dir: str = './visualizations',
    specific_name: str = "",
    title: Optional[str] = None,
    do_save: bool = False
):
    """
    Plots the mean KPI values with marker quantiles.

    Parameters:
        df (pd.DataFrame): The original dataframe with array KPI columns.
        markers_df (pd.DataFrame): The markers dataframe with quantile data.
        kpis (List[str]): List of KPI names to plot. Any occurrence of "mse" will be converted to "mase".
        timestep_col (str): Name of the timestep column.
        marker_kpi_col (str): Name of the KPI column in markers_df.
        dataset_name (str): Name of the dataset for saving files.
        output_dir (str): Directory to save output files.
        specific_name (str): Optional name to include in output filenames.
        title (str): Plot title.
        do_save (bool): Whether to save the figures or just display them.
    """
    # Ensure that any 'mse' in kpis is replaced with 'mase'
    kpis = ['mase' if kpi.lower() == 'mse' else kpi for kpi in kpis]

    # Create output directory if saving is enabled
    if do_save:
        vis_folder = os.path.join(output_dir, f"kpi_markers_summary_{dataset_name}")
        if specific_name:
            vis_folder = os.path.join(vis_folder, specific_name)
        os.makedirs(vis_folder, exist_ok=True)

    # Create subplots for each KPI
    fig = make_subplots(
        rows=len(kpis),
        cols=1,
        subplot_titles=[kpi.upper() for kpi in kpis],
        shared_xaxes=True,
        vertical_spacing=0.1
    )

    # Helper function to calculate mean of array values
    def calc_mean(arr):
        if isinstance(arr, list):
            return np.mean(arr)
        return arr

    # Process each KPI
    for i, kpi in enumerate(kpis):
        # For flat dataframes with _user_ columns, we need to aggregate
        user_cols = [col for col in df.columns if col.startswith(f"{kpi}_user_")]
        
        if user_cols:
            # If we have user-specific columns, calculate mean across users
            mean_values = df[user_cols].mean(axis=1)
        else:
            # Otherwise use the original logic
            mean_values = df[kpi].apply(calc_mean)

        # Plot mean values
        fig.add_trace(
            go.Scatter(
                x=df[timestep_col],
                y=mean_values,
                mode='lines+markers',
                name=f'{kpi.upper()} Mean',
                line=dict(color='blue', width=2),
                legendgroup=kpi
            ),
            row=i+1, col=1
        )

        # Get marker data for this KPI
        marker_data = markers_df[markers_df[marker_kpi_col] == kpi].copy()

        # Plot quantile markers
        for marker_idx, marker in enumerate(['min', 'p20', 'p40', 'p60', 'p80', 'max']):
            if marker in marker_data.columns:
                fig.add_trace(
                    go.Scatter(
                        x=marker_data[timestep_col],
                        y=marker_data[marker],
                        mode='lines',
                        name=f'{marker.upper()}',
                        line=dict(color='blue', width=1, dash='dash'),
                        opacity=0.4,
                        legendgroup=kpi,
                        showlegend=(i == 0 and marker_idx == 0)  # Only show one marker in legend
                    ),
                    row=i+1, col=1
                )

    # Update layout settings
    fig.update_layout(
        height=600 * len(kpis),
        title_text=title if title else "KPI Mean Values with Markers",
        title_font=dict(size=20, family="Arial", color="black"),
        title_x=0.5,
        margin=dict(t=100, b=50, l=80, r=50),  # Add more space for title and legend
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.05,  # Position above the plot
            xanchor="center",
            x=0.5,
            bgcolor="rgba(255, 255, 255, 0.8)",  # Semi-transparent background
            bordercolor="Gray",
            borderwidth=1,
            font=dict(size=12),
            itemsizing="constant"  # Make legend items the same size
        ),
        hovermode='x unified'
    )
    
    # Update axes for each subplot
    for i, kpi in enumerate(kpis):
        fig.update_yaxes(
            title_text=f"{kpi.upper()} Value", 
            row=i+1, 
            col=1,
            gridcolor='rgba(200, 200, 200, 0.2)'  # Lighter grid lines
        )
        
    # Set x-axis label only for the bottom subplot
    fig.update_xaxes(
        title_text="Timestep", 
        row=len(kpis), 
        col=1,
        gridcolor='rgba(200, 200, 200, 0.2)'  # Lighter grid lines
    )

    # Save or display the plot
    if do_save:
        filename_base = "kpi_summary_with_markers"
        if specific_name:
            filename_base += f"_{specific_name}"

        png_path = os.path.join(vis_folder, f"{filename_base}.png")
        fig.write_image(png_path)
        print(f"Summary plot saved as PNG: {png_path}")

        html_path = os.path.join(vis_folder, f"{filename_base}.html")
        fig.write_html(html_path)
        print(f"Summary plot saved as HTML: {html_path}")
    else:
        fig.show()



In [ ]:
plot_user_with_multiple_kpis(
    flat_df=train_flat_df,
    markers_df=train_markers_df,
    kpis=['mase', 'ddtu'],
    users=[1, 3, 5],  # Only create plots for users 1, 3, and 5
    timestep_col='timestep',
    marker_kpi_col='kpi',
    dataset_name='train_data',
    specific_name='user_based_layout',
    title='User {user_num} KPIs with Quantile Markers',
    do_save=False  # Set to False to just display figures
)


## Visualize the Symbolic Representation

### Temporal View

In [15]:
def extract_symbolic_values_for_user(symbolic_df, kpi, user_idx):
    """
    Extract symbolic values for a specific user from the symbolic dataframe.
    """
    symbolic_values = []
    
    for row in symbolic_df.itertuples():
        kpi_values = getattr(row, kpi)
        
        if isinstance(kpi_values, list) and len(kpi_values) > user_idx:
            symbolic_values.append(kpi_values[user_idx])
        else:
            symbolic_values.append(None)
            
    return symbolic_values

def get_value_positions_for_kpi(symbolic_values):
    """
    Create a mapping of symbolic values to y-axis positions for better visualization.
    """
    import re
    
    # Extract unique values
    unique_values = set(v for v in symbolic_values if v is not None)
    
    # Sort values based on category, trend, and predicate
    sorted_values = sorted(
        unique_values,
        key=lambda x: (
            # Category ordering (VeryHigh to VeryLow)
            {'VeryHigh': 5, 'High': 4, 'Medium': 3, 'Low': 2, 'VeryLow': 1}.get(
                re.search(r'(VeryHigh|High|Medium|Low|VeryLow)', x).group() if re.search(r'(VeryHigh|High|Medium|Low|VeryLow)', x) else '',
                0
            ),
            # Trend ordering (if present)
            {'spiking': 3, 'fluctuating': 2, 'dropping': 1}.get(
                re.search(r'(spiking|fluctuating|dropping)', x).group() if re.search(r'(spiking|fluctuating|dropping)', x) else '',
                0
            ),
            # Predicate ordering
            {'inc': 3, 'const': 2, 'dec': 1}.get(
                re.search(r'(inc|const|dec)', x).group() if re.search(r'(inc|const|dec)', x) else '',
                0
            )
        )
    )
    
    # Create mapping with increased spacing
    current_pos = 0
    value_to_position = {}
    last_category = None
    
    for value in sorted_values:
        # Extract category from the symbolic value
        category_match = re.search(r'(VeryHigh|High|Medium|Low|VeryLow)', value)
        category = category_match.group() if category_match else None
        
        # Add extra spacing between different categories
        if last_category is None or category != last_category:
            if last_category is not None:
                current_pos += 4  # Increased spacing between categories
            last_category = category
        
        value_to_position[value] = current_pos
        current_pos += 2  # Increased spacing between values within a category
    
    return value_to_position, sorted_values

def plot_individual_kpi_metrics(test_df, symbolic_df, kpi, user_idx, dataset_name="test_data"):
    """
    Creates an individual figure for a specific KPI and user with numerical values on top,
    symbolic values in the middle, and actions at the bottom.
    """
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    import os
    import re

    # Create visualization folder
    vis_name = "individual_kpi_metrics"
    folder_path = os.path.join("visualizations", vis_name, dataset_name)
    os.makedirs(folder_path, exist_ok=True)

    # Define colors for different KPIs
    colors = {
        'mse': '#1f77b4',       # Blue
        'dtu': '#ff7f0e',       # Orange
        'ddtu': '#2ca02c',      # Green
        'rddtu_a': '#d62728',   # Red
        'rddtu_b': '#9467bd',   # Purple
        'rddtu_c': '#8c564b',   # Brown
        'group': '#e377c2',     # Pink
        'mase': '#1f77b4',      # Blue
        'mase1': '#ff7f0e',     # Orange
        'mase2': '#2ca02c',     # Green
        'mase_all': '#d62728',  # Red
        'mase_fr1': '#ff7f0e',  # Orange
        'mase_fr2': '#2ca02c',  # Green
        'action': '#8c564b'     # Brown
    }

    # Create a 3x1 subplot (numerical on top, symbolic in middle, actions at bottom)
    fig = make_subplots(
        rows=3,
        cols=1,
        subplot_titles=[
            f"User {user_idx+1} {kpi.upper()} (Numerical)",
            f"User {user_idx+1} {kpi.upper()} (Symbolic)",
            f"User {user_idx+1} Agent Actions"
        ],
        vertical_spacing=0.02,  # Reduced spacing between subplots
        shared_xaxes=True,
        row_heights=[0.3, 0.5, 0.2]  # Adjusted row heights
    )

    # Special handling for mase_all which is a list of lists
    if kpi == 'mase_all':
        # Extract the three values for each timestep for this user
        mase_values = []
        mase_fr1_values = []
        mase_fr2_values = []

        for row in test_df.itertuples():
            kpi_values = getattr(row, kpi, [])
            if isinstance(kpi_values, list) and len(kpi_values) > user_idx:
                user_values = kpi_values[user_idx]
                if isinstance(user_values, list) and len(user_values) >= 3:
                    mase_values.append(user_values[0])      # First element: mase
                    mase_fr1_values.append(user_values[1])  # Second element: mase_fr1
                    mase_fr2_values.append(user_values[2])  # Third element: mase_fr2
                else:
                    mase_values.append(None)
                    mase_fr1_values.append(None)
                    mase_fr2_values.append(None)
            else:
                mase_values.append(None)
                mase_fr1_values.append(None)
                mase_fr2_values.append(None)

        # Add trace for mase (first value in each list)
        fig.add_trace(
            go.Scatter(
                x=test_df['timestep'],
                y=mase_values,
                mode='lines+markers',
                name='MASE',
                line=dict(color=colors.get('mase', '#1f77b4')),
                hovertemplate='Timestep: %{x}<br>MASE: %{y}<extra></extra>',
                showlegend=True
            ),
            row=1, col=1
        )

        # Add trace for mase_fr1 (second value in each list)
        fig.add_trace(
            go.Scatter(
                x=test_df['timestep'],
                y=mase_fr1_values,
                mode='lines+markers',
                name='MASE_FR1',
                line=dict(color=colors.get('mase_fr1', '#ff7f0e')),
                hovertemplate='Timestep: %{x}<br>MASE_FR1: %{y}<extra></extra>',
                showlegend=True
            ),
            row=1, col=1
        )

        # Add trace for mase_fr2 (third value in each list)
        fig.add_trace(
            go.Scatter(
                x=test_df['timestep'],
                y=mase_fr2_values,
                mode='lines+markers',
                name='MASE_FR2',
                line=dict(color=colors.get('mase_fr2', '#2ca02c')),
                hovertemplate='Timestep: %{x}<br>MASE_FR2: %{y}<extra></extra>',
                showlegend=True
            ),
            row=1, col=1
        )
    else:
        # Normal KPI handling (not mase_all)
        # Extract numerical values for this user and KPI
        num_values = []
        for row in test_df.itertuples():
            kpi_values = getattr(row, kpi, [])
            if isinstance(kpi_values, list) and len(kpi_values) > user_idx:
                num_values.append(kpi_values[user_idx])
            elif not isinstance(kpi_values, list):
                # Handle the case where kpi_values is a single value (not a list)
                num_values.append(kpi_values if row.Index == user_idx else None)
            else:
                num_values.append(None)

        # Add numerical trace (top row)
        fig.add_trace(
            go.Scatter(
                x=test_df['timestep'],
                y=num_values,
                mode='lines+markers',
                name=f'Numerical {kpi.upper()}',
                line=dict(color=colors.get(kpi, '#1f77b4')),
                hovertemplate='Timestep: %{x}<br>Value: %{y}<extra></extra>',
                showlegend=True
            ),
            row=1, col=1
        )

    # Extract symbolic values for this user and KPI
    sym_values = []
    for row in symbolic_df.itertuples():
        kpi_values = getattr(row, kpi)
        if isinstance(kpi_values, list) and len(kpi_values) > user_idx:
            sym_values.append(kpi_values[user_idx])
        elif not isinstance(kpi_values, list):
            # If not a list, it might be a direct value
            sym_values.append(kpi_values if row.Index == user_idx else None)
        else:
            sym_values.append(None)

    # Get position mappings for symbolic values
    position_map, sorted_values = get_value_positions_for_kpi(sym_values)

    # Convert symbolic values to positions
    positions = [position_map.get(val) if val is not None else None for val in sym_values]

    # Add symbolic trace (middle row)
    fig.add_trace(
        go.Scatter(
            x=symbolic_df['timestep'],
            y=positions,
            mode='lines+markers',
            name=f'Symbolic {kpi.upper()}',
            line=dict(color=colors.get(kpi, '#1f77b4')),
            hovertemplate='Timestep: %{x}<br>Value: %{text}<extra></extra>',
            text=sym_values,
            showlegend=True
        ),
        row=2, col=1
    )

    # Extract action values for this user
    action_values = []
    for row in symbolic_df.itertuples():
        if hasattr(row, 'action'):
            action = getattr(row, 'action')
            if isinstance(action, list) and len(action) > user_idx:
                action_values.append(action[user_idx])
            else:
                action_values.append(action)
        else:
            action_values.append(None)

    # Get unique actions for y-axis mapping
    unique_actions = list(set([a for a in action_values if a is not None]))
    action_mapping = {action: i for i, action in enumerate(unique_actions)}

    # Map actions to positions
    action_positions = [action_mapping.get(val) if val is not None else None for val in action_values]

    # Add action trace (bottom row as a line plot with markers)
    fig.add_trace(
        go.Scatter(
            x=symbolic_df['timestep'],
            y=action_positions,
            mode='lines+markers',
            name='Agent Actions',
            line=dict(color=colors.get('action', '#8c564b')),
            hovertemplate='Timestep: %{x}<br>Action: %{text}<extra></extra>',
            text=action_values,
            showlegend=True
        ),
        row=3, col=1
    )

    # Update y-axis for symbolic plot
    fig.update_yaxes(
        tickmode='array',
        ticktext=sorted_values,
        tickvals=[position_map[v] for v in sorted_values],
        row=2, col=1,
        title_text=f"{kpi.upper()} Symbolic Value",
        tickfont=dict(size=12)  # Increased font size
    )

    # Update numerical y-axis
    fig.update_yaxes(
        title_text=f"{kpi.upper()} Value",
        title_font=dict(size=14),  # Increased title font size
        tickfont=dict(size=14),    # Increased tick font size
        row=1, col=1
    )

    # Update action y-axis
    fig.update_yaxes(
        tickmode='array',
        ticktext=unique_actions,
        tickvals=[action_mapping[a] for a in unique_actions],
        title_text="Agent Actions",
        title_font=dict(size=14),  # Increased title font size
        tickfont=dict(size=14),    # Increased tick font size
        row=3, col=1
    )

    # Update x-axes
    fig.update_xaxes(
        showticklabels=True,  # Now show labels on bottom plot only
        title_text="Timestep",
        title_font=dict(size=14),  # Increased title font size
        tickfont=dict(size=14),    # Increased tick font size
        row=3, col=1,
        showgrid=True,
        gridcolor='lightgray'
    )

    # Hide x-axis labels on first and second subplots
    fig.update_xaxes(
        showticklabels=False,
        showgrid=True,
        gridcolor='lightgray',
        row=1, col=1
    )

    fig.update_xaxes(
        showticklabels=False,
        showgrid=True,
        gridcolor='lightgray',
        row=2, col=1
    )

    # Update layout
    fig.update_layout(
        height=1600,  # Reduced overall height
        title_text=f"User {user_idx+1} - {kpi.upper()} Numerical and Symbolic Representations",
        title_font=dict(size=20),  # Larger title font
        hovermode='x unified',
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=-0.15,
            xanchor="center",
            x=0.5,
            font=dict(size=14)  # Larger legend font
        ),
        plot_bgcolor='rgba(240, 240, 240, 0.9)',
        paper_bgcolor='white',
        margin=dict(l=100, r=50, t=100, b=150)  # Adjusted margins
    )

    # Save the figure
    png_filename = os.path.join(folder_path, f"{kpi}_user{user_idx+1}_{dataset_name}.png")
    fig.write_image(png_filename)
    print(f"{kpi.upper()} plot for User {user_idx+1} saved as PNG to: {png_filename}")

    html_filename = os.path.join(folder_path, f"{kpi}_user{user_idx+1}_{dataset_name}.html")
    fig.write_html(html_filename)
    print(f"{kpi.upper()} plot for User {user_idx+1} saved as HTML to: {html_filename}")

    return fig

def plot_all_kpis_for_user(test_df, symbolic_df, kpis=['mse', 'dtu', 'ddtu', 'rddtu_a', 'rddtu_b'], 
                           user_idx=0, dataset_name="test_data"):
    """
    Creates individual figures for each KPI for a specific user.
    
    Args:
        test_df (pd.DataFrame): DataFrame with numerical values
        symbolic_df (pd.DataFrame): DataFrame with symbolic representations
        kpis (list): List of KPIs to plot
        user_idx (int): The user index to plot (0-6)
        dataset_name (str): Name of the dataset for folder creation
        
    Returns:
        dict: Dictionary mapping KPI names to their respective figure objects
    """
    figures = {}
    
    for kpi in kpis:
        figures[kpi] = plot_individual_kpi_metrics(test_df, symbolic_df, kpi, user_idx, dataset_name)
        
    return figures

# Example usage:
# For a single KPI and user
fig = plot_individual_kpi_metrics(test_df, test_symbolic_df, kpi='mase_all', user_idx=0, dataset_name="test_data")
fig.show()

MASE_ALL plot for User 1 saved as PNG to: visualizations/individual_kpi_metrics/test_data/mase_all_user1_test_data.png
MASE_ALL plot for User 1 saved as HTML to: visualizations/individual_kpi_metrics/test_data/mase_all_user1_test_data.html


### Probabilistic View

In [16]:
def create_symbolic_strings(kpi_name):
    """
    Creates all possible symbolic strings for a KPI.
    
    Args:
        kpi_name (str): Name of the KPI
    
    Returns:
        list: All possible symbolic strings for the KPI
    """
    predicates = ['dec', 'const', 'inc']
    categories = ['VeryLow', 'Low', 'Medium', 'High', 'VeryHigh']
    symbolic_strings = [f"{predicate}({kpi_name}, {category})" for category in categories for predicate in predicates]
    return symbolic_strings

def extract_symbolic_values_for_user(symbolic_df, kpi, user_idx):
    """
    Extract symbolic values for a specific user from the symbolic dataframe.
    
    Args:
        symbolic_df (pd.DataFrame): DataFrame containing symbolic representations
        kpi (str): KPI to analyze
        user_idx (int): Index of the user
        
    Returns:
        list: Symbolic values for the specified user
    """
    symbolic_values = []
    
    for row in symbolic_df.itertuples():
        kpi_values = getattr(row, kpi)
        
        if isinstance(kpi_values, list) and len(kpi_values) > user_idx:
            symbolic_values.append(kpi_values[user_idx])
        else:
            symbolic_values.append(None)
    
    return symbolic_values

def create_user_symbolic_probability_distribution(symbolic_df, kpi, dataset_name, specific_name="", base_font_size=14, num_users=7):
    """
    Creates a figure showing the probability distribution of symbolic values for each user.
    
    Args:
        symbolic_df (pd.DataFrame): DataFrame containing symbolic representations
        kpi (str): KPI to analyze ('mse' or 'dtu')
        dataset_name (str): Name of the dataset for folder creation
        specific_name (str, optional): Additional specification for folder path
        base_font_size (int, optional): Base font size to scale all text elements
        num_users (int, optional): Number of users to process
    
    Returns:
        go.Figure: The plotly figure object
    """
    import re
    import os
    import plotly.graph_objects as go
    
    # Create visualization folder
    vis_name = "symbolic_probability_by_user"
    folder_path = create_visualization_folder_path(vis_name, dataset_name, specific_name)
    
    # Define font sizes based on the base font size
    title_font_size = int(base_font_size * 1.7)  # Title
    axis_title_font_size = int(base_font_size * 1.3)  # Axis titles
    tick_font_size = int(base_font_size * 0.9)  # Axis ticks
    legend_font_size = base_font_size  # Legend
    
    # Colors for each user - extended to handle more users if needed
    colors = [
        '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728',
        '#9467bd', '#8c564b', '#e377c2', '#7f7f7f',
        '#bcbd22', '#17becf', '#aec7e8', '#ffbb78'
    ]
    
    # Create the figure
    fig = go.Figure()
    
    # Generate all possible symbolic strings for the KPI using the helper function
    sorted_values = create_symbolic_strings(kpi)
    
    # Create value to position mapping
    value_to_position = {value: idx for idx, value in enumerate(sorted_values)}
    
    # Extract symbolic values for each user
    user_values_dict = {}
    for user_idx in range(num_users):
        user_values = extract_symbolic_values_for_user(symbolic_df, kpi, user_idx)
        user_values_dict[user_idx] = user_values
    
    # Calculate probability distributions for each user and add traces
    for user_idx in range(num_users):
        user_values = user_values_dict[user_idx]
        
        # Count occurrences and compute probabilities
        value_counts = {}
        valid_values = [v for v in user_values if v is not None]
        total_count = len(valid_values)
        
        if total_count == 0:  # Skip users with no valid data
            continue
            
        for value in valid_values:
            value_counts[value] = value_counts.get(value, 0) + 1
        
        # Convert to probabilities
        value_probabilities = {k: v / total_count for k, v in value_counts.items()}
        
        # Fill in probabilities for all possible values (even those not seen by this user)
        x_values = []
        y_values = []
        hover_texts = []
        
        for value in sorted_values:
            x_values.append(value_to_position[value])
            probability = value_probabilities.get(value, 0)
            y_values.append(probability)
            hover_texts.append(f"User {user_idx+1}<br>Value: {value}<br>Probability: {probability:.4f}")
        
        # Add trace for this user as a line+marker plot
        fig.add_trace(go.Scatter(
            x=x_values,
            y=y_values,
            name=f'User {user_idx+1}',
            mode='lines+markers',  # Use lines with markers
            line=dict(
                color=colors[user_idx % len(colors)],  # Use modulo to handle more users than colors
                width=2
            ),
            marker=dict(
                color=colors[user_idx % len(colors)],
                size=8,
                symbol='circle'
            ),
            hovertext=hover_texts,
            hoverinfo='text'
        ))
    
    # Update layout
    fig.update_layout(
        title=dict(
            text=f'Probability Distribution of {kpi.upper()} Symbolic Values by User',
            font=dict(size=title_font_size, family="Arial", color="black", weight="bold"),
            x=0.5
        ),
        xaxis=dict(
            title=dict(
                text='Symbolic Value',
                font=dict(size=axis_title_font_size, family="Arial", color="black", weight="bold")
            ),
            tickmode='array',
            ticktext=sorted_values,
            tickvals=[value_to_position[v] for v in sorted_values],
            tickangle=45,  # Rotate labels for better readability
            tickfont=dict(size=tick_font_size),
            showgrid=True,
            gridcolor='lightgray'
        ),
        yaxis=dict(
            title=dict(
                text='Probability',
                font=dict(size=axis_title_font_size, family="Arial", color="black", weight="bold")
            ),
            tickfont=dict(size=tick_font_size),
            showgrid=True,
            gridcolor='lightgray',
            range=[0, 1]  # Fix y-axis range from 0 to 1
        ),
        height=800,  # Keep only height setting
        hovermode='closest',
        legend=dict(
            font=dict(size=legend_font_size),
            orientation="v",  # Vertical legend
            yanchor="top",
            y=1,
            xanchor="left",
            x=1.02,  # Position legend outside the plot area to the right
            bordercolor="Black",  # Add a border
            borderwidth=1  # Border width
        ),
        plot_bgcolor='rgba(240, 240, 240, 0.9)',
        paper_bgcolor='white',
        margin=dict(l=80, r=150, t=100, b=180)  # Increased right margin for legend
    )
    
    # Save the figure
    png_filename = os.path.join(folder_path, f"probability_distribution_{kpi}_{dataset_name}.png")
    fig.write_image(png_filename)
    print(f"Probability distribution plot for {kpi} saved as PNG to: {png_filename}")
    
    html_filename = os.path.join(folder_path, f"probability_distribution_{kpi}_{dataset_name}.html")
    fig.write_html(html_filename)
    print(f"Probability distribution plot for {kpi} saved as HTML to: {html_filename}")
    
    return fig


In [17]:
fig = create_user_symbolic_probability_distribution(test_symbolic_df, 'rddtu_a', "test_data", base_font_size=18)
fig.show()

Probability distribution plot for rddtu_a saved as PNG to: user_fr_visualizations/symbolic_probability_by_user/test_data/probability_distribution_rddtu_a_test_data.png
Probability distribution plot for rddtu_a saved as HTML to: user_fr_visualizations/symbolic_probability_by_user/test_data/probability_distribution_rddtu_a_test_data.html


## Analyze MI for all kpis including new representation of the symbolic representation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mutual_info_score
from scipy.stats import chi2_contingency
import datetime
import os
from typing import Optional, List, Union

def get_user_data(df, user_id, kpi_cols=None):
    """
    Extracts KPIs and action for a specific user from the dataframe.
    
    Args:
        df (pd.DataFrame): The input dataframe
        user_id (int): The index of the user in the arrays (0-based)
        kpi_cols (List[str], optional): List of KPI columns to extract. If None, extracts all available KPIs.
        
    Returns:
        pd.DataFrame: A new dataframe with specified KPIs and action for the specified user
    """
    # Determine available KPI columns in the dataframe
    available_kpis = [col for col in ['mase', 'mase_all', 'dtu', 'ddtu', 'rddtu_a', 'rddtu_b'] if col in df.columns]
    
    # If no specific KPI columns are requested, use all available ones
    if kpi_cols is None:
        kpi_cols = available_kpis
    else:
        # Ensure requested KPIs are available
        for kpi in kpi_cols:
            if kpi not in available_kpis:
                raise ValueError(f"KPI '{kpi}' not found in dataframe. Available KPIs: {available_kpis}")
    
    # Create a new dataframe to store the results
    result_df = []
    
    # Process each row in the dataframe
    for index, row in df.iterrows():
        # Initialize new row with timestep
        new_row = {'timestep': row['timestep']}
        
        # Extract elements for each requested KPI
        for kpi in kpi_cols:
            new_row[kpi] = row[kpi][user_id]
        
        # Add action
        new_row['action'] = row['action'][user_id]
        
        # Add to result dataframe
        result_df.append(new_row)
    
    result_df = pd.DataFrame(result_df)
    
    return result_df

def create_symbolic_action_strings():
    """
    Creates all possible symbolic strings for actions, including group information.
    
    Returns:
        list: All possible symbolic action strings
    """
    action_types = ['notSched', 'Sched']
    groups = ['G0', 'G1', 'G2']  # Assuming groups 0, 1, and 2
    symbolic_strings = [f"{action_type}(user, {group})" for action_type in action_types for group in groups]
    return symbolic_strings

def create_symbolic_strings(kpi_name, forecast_variable=False):
    """
    Creates all possible symbolic strings for a KPI.
    
    Args:
        kpi_name (str): Name of the KPI
        include_pattern (bool): If True, adds a pattern dimension (dropping, fluctuating, spiking)
    
    Returns:
        list: All possible symbolic strings for the KPI
    """
    predicates = ['dec', 'const', 'inc']
    categories = ['VeryLow', 'Low', 'Medium', 'High', 'VeryHigh']
    
    if not forecast_variable:
        # Original functionality - predicate(kpi, category)
        symbolic_strings = [f"{predicate}({kpi_name}, {category})" for category in categories for predicate in predicates]
    else:
        # Extended functionality - predicate(kpi, category, pattern)
        patterns = ['dropping', 'fluctuating', 'spiking']
        symbolic_strings = [
            f"{predicate}({kpi_name}, {category}, {pattern})" 
            for category in categories 
            for predicate in predicates 
            for pattern in patterns
        ]
    
    return symbolic_strings

def sort_symbolic_kpi_strings_key(symbolic_string):
    """
    Key function for sorting symbolic KPI strings by category level.
    """
    level_order = ["VeryHigh", "High", "Medium", "Low", "VeryLow"]
    for level in level_order:
        if level in symbolic_string:
            return level_order.index(level)
    return len(level_order)

def sort_symbolic_kpi_strings(symbolic_strings):
    """
    Sorts symbolic KPI strings in reverse order (VeryHigh to VeryLow).
    """
    return sorted(symbolic_strings, key=sort_symbolic_kpi_strings_key, reverse=True)

def calculate_cramers_v(freq_matrix):
    """
    Calculates Cramer's V for a given frequency matrix.
    Handles sparse matrices by filtering out all-zero rows and columns.
    """
    # Filter out rows and columns with all zeros to avoid chi2_contingency errors
    filtered_matrix = freq_matrix.loc[(freq_matrix.sum(axis=1) > 0), (freq_matrix.sum(axis=0) > 0)]
    
    # If matrix is too small after filtering
    if filtered_matrix.shape[0] < 2 or filtered_matrix.shape[1] < 2:
        return 0.0
    
    try:
        chi2 = chi2_contingency(filtered_matrix)[0]
        n = filtered_matrix.values.sum()
        min_dim = min(filtered_matrix.shape) - 1
        if min_dim == 0:
            return 0.0
        return np.sqrt(chi2 / (n * min_dim))
    except ValueError as e:
        print(f"Warning: Could not calculate Cramer's V: {str(e)}")
        return 0.0  # Return 0 if calculation fails

def calculate_correlation_ratio(df, row_kpi_col, col_action_col):
    """
    Calculates the Correlation Ratio (Eta) between a categorical KPI and action.
    Handles the new action format with group information.
    """
    # Extract action type (Sched or notSched) from the new format
    df['action_type'] = df[col_action_col].apply(lambda x: x.split('(')[0])
    
    # Create a numerical mapping for actions
    action_to_num = {action: idx for idx, action in enumerate(df['action_type'].unique())}
    df['numerical_action'] = df['action_type'].map(action_to_num)
    
    overall_variance = df['numerical_action'].var()
    if overall_variance == 0:
        return 0.0

    between_variance = 0.0
    for category in df[row_kpi_col].unique():
        category_data = df[df[row_kpi_col] == category]['numerical_action']
        if len(category_data) > 0:  # Check to avoid empty categories
            category_mean = category_data.mean()
            between_variance += len(category_data) * (category_mean - df['numerical_action'].mean())**2

    eta_squared = between_variance / (len(df) * overall_variance)
    
    # Clean up temporary columns
    df.drop(columns=['numerical_action', 'action_type'], inplace=True, errors='ignore')
    
    return np.sqrt(eta_squared)

def create_kpi_action_frequency_matrix(result_df, kpi_col):
    """
    Creates a frequency matrix showing distribution of actions for each KPI state.
    """
    # Create a cross-tabulation of kpi vs action
    freq_matrix = pd.crosstab(result_df[kpi_col], result_df['action'])
    
    # Calculate percentages
    total = freq_matrix.values.sum()
    percentage_matrix = (freq_matrix / total) * 100
    
    # Sort the kpi values in a logical order
    if '_all' in kpi_col:
        level_order = create_symbolic_strings(kpi_col, forecast_variable=True)
    else:
        level_order = create_symbolic_strings(kpi_col)
    
    # Get all possible action values
    all_action_values = create_symbolic_action_strings()
    
    # Reindex to sort the matrix rows in the logical order, filling missing values with 0
    percentage_matrix = percentage_matrix.reindex(index=level_order, fill_value=0)
    
    # Reindex to include all possible action values, filling missing values with 0
    percentage_matrix = percentage_matrix.reindex(columns=all_action_values, fill_value=0)
    
    return percentage_matrix

def plot_kpi_action_heatmap_enhanced(user_df, kpi_col, title=None, vmax=20, figsize=None, cmap='YlOrRd'):
    """
    Creates an enhanced heatmap visualization with MI, Cramer's V, and Eta metrics.
    
    Args:
        user_df (pd.DataFrame): User dataframe with KPI and action data
        kpi_col (str): KPI column to analyze
        title (str, optional): Custom title for the plot
        vmax (float): Maximum value for color scale
        figsize (tuple, optional): Figure size as (width, height)
        cmap (str): Colormap to use
        
    Returns:
        tuple: (fig, ax, freq_matrix) - the figure, axis, and frequency matrix
    """
    # Calculate the frequency matrix
    freq_matrix = create_kpi_action_frequency_matrix(user_df, kpi_col)
    
    # Calculate statistics
    mi_value = mutual_info_score(user_df[kpi_col], user_df['action'])
    print(f"Mutual Information: {mi_value:.2f}")
    
    cramers_v_value = calculate_cramers_v(freq_matrix)
    print(f"Cramer's V: {cramers_v_value:.2f}")
    
    correlation_ratio_value = calculate_correlation_ratio(user_df, kpi_col, 'action')
    print(f"Correlation Ratio (Eta): {correlation_ratio_value:.2f}")
    
    # Set up figure size based on matrix dimensions
    if figsize is None:
        fig_height = max(8, len(freq_matrix.index) * 0.5)
        fig_width = max(10, len(freq_matrix.columns) * 1.0)
        figsize = (fig_width, fig_height)
    
    # Create figure and axes
    fig, ax = plt.subplots(figsize=figsize)
    
    # Set plot style
    sns.set(style="white")
    
    # Create heatmap
    sns.heatmap(freq_matrix, annot=True, fmt='.1f', cmap=cmap, 
                vmin=0, vmax=vmax, cbar_kws={'label': 'Percentage (%)'}, 
                linewidths=0.5, ax=ax)
    
    # Set title and subtitle with metrics
    if title is None:
        title = f"{kpi_col.capitalize()} Heatmap - Metrics"
    
    plt.suptitle(title, fontsize=16, y=0.98)
    subtitle = f"{kpi_col.capitalize()} vs Action\nMI: {mi_value:.2f}, Cramer's V: {cramers_v_value:.2f}, Eta: {correlation_ratio_value:.2f}"
    ax.set_title(subtitle, fontsize=14, pad=10)
    
    # Set axis labels
    ax.set_xlabel('Action', fontsize=12)
    ax.set_ylabel(f'{kpi_col.capitalize()} State', fontsize=12)
    
    # Rotate x labels
    plt.xticks(rotation=45, ha='right')
    
    # Adjust layout
    plt.tight_layout()
    
    return fig, ax, freq_matrix

def visualize_user_kpi_action_relationship(df, user_id, kpi_col):
    """
    End-to-end function to get user data, create and visualize a KPI vs action heatmap.
    
    Args:
        df (pd.DataFrame): Original dataframe with data for all users
        user_id (int): The index of the user to analyze
        kpi_col (str): KPI column to analyze
        
    Returns:
        tuple: (freq_matrix, fig, ax, user_df) - the frequency matrix, figure, axis, and user dataframe
    """
    # Get user data
    user_df = get_user_data(df, user_id, kpi_cols=[kpi_col])
    
    # Create and visualize the heatmap
    fig, ax, freq_matrix = plot_kpi_action_heatmap_enhanced(user_df, kpi_col=kpi_col)
    
    # Match the return order to what's expected in your call
    return freq_matrix, fig, ax, user_df

def create_visualization_folder_path(vis_name, dataset_name, specific_name=""):
    """
    Creates a folder path for saving visualizations.
    
    Args:
        vis_name (str): Name of the visualization
        dataset_name (str): Name of the dataset
        specific_name (str, optional): Specific identifier
        
    Returns:
        str: Full path to the folder
    """
    # Create base folder
    base_folder = "visualizations"
    
    # Create folder path
    folder_path = os.path.join(base_folder, vis_name, dataset_name)
    if specific_name:
        folder_path = os.path.join(folder_path, specific_name)
    
    # Create folders if they don't exist
    os.makedirs(folder_path, exist_ok=True)
    
    return folder_path

def visualize_multi_user_kpi_analysis(df, kpi_col, vmax=20, cmap='YlOrRd', 
                                     dataset_name='mimo', specific_name="", 
                                     show=True, do_save=True):
    """
    Creates a figure with 7 subplots (4 in first row, 3 in second row),
    each showing a KPI vs action heatmap for a different user.
    
    Args:
        df (pd.DataFrame): Original dataframe with data for all users
        kpi_col (str): KPI column to analyze
        vmax (float): Maximum value for color scale
        cmap (str): Colormap to use
        dataset_name (str): Name of the dataset for saving files
        specific_name (str): Optional specific identifier for the visualization
        show (bool): Whether to display the figure
        do_save (bool): Whether to save the figure
        
    Returns:
        tuple: (fig, axes) - the figure and axes objects
    """
    # Define visualization name
    vis_name = "kpi_action_heatmaps"
    
    # Create a figure with 2 rows, 4 columns
    fig = plt.figure(figsize=(24, 16))
    gs = fig.add_gridspec(2, 4)
    
    # Create axes for each subplot
    axes = []
    # First row: users 0-3 (4 subplots)
    for i in range(4):
        axes.append(fig.add_subplot(gs[0, i]))
    
    # Second row: users 4-6 (3 subplots)
    for i in range(3):
        axes.append(fig.add_subplot(gs[1, i]))
    
    # Main title for the entire figure
    main_title = f"{kpi_col.upper()} VS ACTION RELATIONSHIP ACROSS USERS"
    fig.suptitle(main_title, fontsize=20, y=0.98)
    
    # Process each user (7 users)
    for i, user_id in enumerate(range(7)):
        ax = axes[i]
        
        # Get user data
        user_df = get_user_data(df, user_id, kpi_cols=[kpi_col])
        
        # Calculate frequency matrix
        freq_matrix = create_kpi_action_frequency_matrix(user_df, kpi_col)
        
        # Calculate metrics
        mi_value = mutual_info_score(user_df[kpi_col], user_df['action'])
        cramers_v_value = calculate_cramers_v(freq_matrix)
        correlation_ratio_value = calculate_correlation_ratio(user_df, kpi_col, 'action')
        
        # Create heatmap for this user
        sns.heatmap(freq_matrix, annot=True, fmt='.1f', cmap=cmap, 
                    vmin=0, vmax=vmax, cbar_kws={'label': 'Percentage (%)'}, 
                    linewidths=0.5, ax=ax)
        
        # Set subplot title - now with title on first line and metrics on second line
        ax.set_title(f"User {user_id}\nMI: {mi_value:.2f}, V: {cramers_v_value:.2f}, Eta: {correlation_ratio_value:.2f}", 
                     fontsize=12, pad=10)
        
        # Set axis labels
        ax.set_xlabel('Action', fontsize=10)
        ax.set_ylabel(f'{kpi_col.upper()} State', fontsize=10)
        
        # Rotate x labels
        plt.setp(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
        plt.setp(ax.get_yticklabels(), fontsize=8)
    
    # Adjust spacing between subplots
    plt.subplots_adjust(wspace=0.3, hspace=0.5)  # Increased hspace for the two-line titles
    
    # Add a hidden subplot to fill the empty space in bottom right
    empty_ax = fig.add_subplot(gs[1, 3])
    empty_ax.axis('off')
    
    # Adjust layout
    plt.tight_layout(rect=[0, 0, 1, 0.96])  # Leave room for suptitle
    
    # Save figure if requested
    if do_save:
        # Use the create_visualization_folder_path function to get the save location
        plot_folder_path = create_visualization_folder_path(vis_name, dataset_name, specific_name)
        
        # Create timestamp for unique filename
        timestamp = datetime.datetime.now().strftime("%B-%d_%H%M")
        
        # Create the filename
        filename_parts = [vis_name, kpi_col, dataset_name]
        if specific_name:
            filename_parts.append(specific_name)
        
        base_filename = "_".join(part for part in filename_parts if part) + "_" + timestamp + ".png"
        
        # Full path to save the file
        save_path = os.path.join(plot_folder_path, base_filename)
        
        # Save the figure
        plt.savefig(save_path, bbox_inches='tight', dpi=300)
        print(f"Figure saved to: {save_path}")
    
    # Show figure if requested
    if show:
        plt.show()
    else:
        plt.close(fig)
    
    return fig, axes

def combine_all_users_data(df, kpi_cols):
    """
    Extracts and combines KPI and action data for all users.
    
    Args:
        df (pd.DataFrame): Original dataframe with data for all users
        kpi_cols (List[str]): List of KPI columns to extract
        
    Returns:
        dict: Dictionary of dataframes, one for each KPI with data from all users combined
    """
    # Determine number of users from the first row
    first_row = df.iloc[0]
    num_users = len(first_row['action']) if isinstance(first_row['action'], list) else 1
    
    # Create a dictionary to store combined data for each KPI
    combined_data = {kpi: [] for kpi in kpi_cols}
    
    # Process each row in the dataframe
    for _, row in df.iterrows():
        # For each user
        for user_id in range(num_users):
            # For each requested KPI
            for kpi in kpi_cols:
                if kpi in row and isinstance(row[kpi], list) and user_id < len(row[kpi]):
                    # Extract KPI value and action for this user
                    kpi_value = row[kpi][user_id]
                    action_value = row['action'][user_id] if isinstance(row['action'], list) else row['action']
                    
                    # Add to combined data
                    combined_data[kpi].append({
                        'user_id': user_id,
                        'timestep': row['timestep'],
                        kpi: kpi_value,
                        'action': action_value
                    })
    
    # Convert lists to dataframes
    return {kpi: pd.DataFrame(data) for kpi, data in combined_data.items()}

def visualize_aggregated_kpi_analysis(df, kpi_cols, vmax=20, cmap='YlOrRd', 
                                     dataset_name='mimo', specific_name="", 
                                     show=True, do_save=True, figsize=(12, 10)):
    """
    Creates a figure with one subplot per KPI, each showing an aggregated heatmap
    of all users' data for that KPI vs actions.
    
    Args:
        df (pd.DataFrame): Original dataframe with data for all users
        kpi_cols (List[str]): List of KPI columns to analyze
        vmax (float): Maximum value for color scale
        cmap (str): Colormap to use
        dataset_name (str): Name of the dataset for saving files
        specific_name (str): Optional specific identifier for the visualization
        show (bool): Whether to display the figure
        do_save (bool): Whether to save the figure
        figsize (tuple): Base figure size, will be adjusted based on number of KPIs
        
    Returns:
        tuple: (fig, axes) - the figure and axes objects
    """
    # Define visualization name
    vis_name = "aggregated_kpi_heatmaps"
    
    # Combine data from all users for each KPI
    combined_data = combine_all_users_data(df, kpi_cols)
    
    # Determine subplot layout based on number of KPIs
    num_kpis = len(kpi_cols)
    if num_kpis <= 2:
        nrows, ncols = 1, num_kpis
    else:
        nrows = (num_kpis + 1) // 2  # Ceiling division
        ncols = 2
    
    # Adjust figure size based on number of subplots
    adjusted_figsize = (figsize[0] * ncols, figsize[1] * nrows)
    
    # Create figure and subplots
    fig, axes = plt.subplots(nrows, ncols, figsize=adjusted_figsize)
    
    # Convert to array if only one subplot
    if num_kpis == 1:
        axes = np.array([axes])
    
    # Flatten axes array for easy iteration
    axes_flat = axes.flatten() if isinstance(axes, np.ndarray) else [axes]
    
    # Main title for the entire figure
    main_title = "AGGREGATED KPI VS ACTION RELATIONSHIPS"
    fig.suptitle(main_title, fontsize=20, y=0.98)
    
    # Process each KPI
    for i, kpi_col in enumerate(kpi_cols):
        if i < len(axes_flat):
            ax = axes_flat[i]
            
            # Skip if no data for this KPI
            if kpi_col not in combined_data or combined_data[kpi_col].empty:
                ax.text(0.5, 0.5, f"No data for {kpi_col}", ha='center', va='center')
                ax.axis('off')
                continue
            
            # Get combined data for this KPI
            kpi_df = combined_data[kpi_col]
            
            # Calculate frequency matrix
            freq_matrix = create_kpi_action_frequency_matrix(kpi_df, kpi_col)
            
            # Calculate metrics
            mi_value = mutual_info_score(kpi_df[kpi_col], kpi_df['action'])
            cramers_v_value = calculate_cramers_v(freq_matrix)
            correlation_ratio_value = calculate_correlation_ratio(kpi_df, kpi_col, 'action')
            
            # Create heatmap for this KPI
            sns.heatmap(freq_matrix, annot=True, fmt='.1f', cmap=cmap, 
                        vmin=0, vmax=vmax, cbar_kws={'label': 'Percentage (%)'}, 
                        linewidths=0.5, ax=ax)
            
            # Set subplot title
            ax.set_title(f"{kpi_col.upper()}\nMI: {mi_value:.2f}, V: {cramers_v_value:.2f}, Eta: {correlation_ratio_value:.2f}", 
                        fontsize=14, pad=10)
            
            # Set axis labels
            ax.set_xlabel('Action', fontsize=12)
            ax.set_ylabel(f'KPI State', fontsize=12)
            
            # Rotate x labels
            plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
    
    # Hide empty subplots
    for i in range(len(kpi_cols), len(axes_flat)):
        axes_flat[i].axis('off')
    
    # Adjust spacing between subplots
    plt.subplots_adjust(wspace=0.3, hspace=0.4)
    
    # Adjust layout
    plt.tight_layout(rect=[0, 0, 1, 0.95])  # Leave room for suptitle
    
    # Save figure if requested
    if do_save:
        # Use the create_visualization_folder_path function to get the save location
        plot_folder_path = create_visualization_folder_path(vis_name, dataset_name, specific_name)
        
        # Create timestamp for unique filename
        timestamp = datetime.datetime.now().strftime("%B-%d_%H%M")
        
        # Create the filename
        kpi_part = "_".join(kpi_cols)
        filename_parts = [vis_name, kpi_part, dataset_name]
        if specific_name:
            filename_parts.append(specific_name)
        
        base_filename = "_".join(part for part in filename_parts if part) + "_" + timestamp + ".png"
        
        # Full path to save the file
        save_path = os.path.join(plot_folder_path, base_filename)
        
        # Save the figure
        plt.savefig(save_path, bbox_inches='tight', dpi=300)
        print(f"Figure saved to: {save_path}")
    
    # Show figure if requested
    if show:
        plt.show()
    else:
        plt.close(fig)
    
    return fig, axes

def visualize_multivariate_kpi_action_relationship(df, kpi_x='mse', kpi_y='ddtu', 
                                             cmap='YlOrRd', normalize=True,
                                             vmax_multiplier=2.0, min_vmax=0.5,
                                             dataset_name='mimo', specific_name="", 
                                             show=True, do_save=True, figsize=(10, 8)):
    """
    Creates a figure with one subplot for each possible action, showing how combinations
    of two KPIs relate to the agent's decision-making.
    
    Args:
        df (pd.DataFrame): Original dataframe with data for all users
        kpi_x (str): KPI to display on the x-axis
        kpi_y (str): KPI to display on the y-axis
        cmap (str): Colormap to use
        normalize (bool): Whether to normalize values as percentages (True) or show raw counts (False)
        vmax_multiplier (float): Multiplier for the maximum value in each heatmap to set vmax
        min_vmax (float): Minimum vmax value to use if a heatmap has very small values
        dataset_name (str): Name of the dataset for saving files
        specific_name (str): Optional specific identifier for the visualization
        show (bool): Whether to display the figure
        do_save (bool): Whether to save the figure
        figsize (tuple): Base size of each subplot - will be multiplied by grid size
        
    Returns:
        tuple: (fig, axes) - the figure and axes objects
    """
    # Define visualization name
    vis_name = "multivariate_kpi_heatmaps"
    
    # Get all possible actions
    all_actions = create_symbolic_action_strings()
    
    # Get all possible symbolic values for the KPIs
    x_symbols = sort_symbolic_kpi_strings(create_symbolic_strings(kpi_x))
    y_symbols = sort_symbolic_kpi_strings(create_symbolic_strings(kpi_y))
    
    # Extract and combine data from all users
    combined_data = []
    
    # Determine number of users from the first row
    first_row = df.iloc[0]
    num_users = len(first_row['action']) if isinstance(first_row['action'], list) else 1
    
    # Process each row in the dataframe
    for _, row in df.iterrows():
        # For each user
        for user_id in range(num_users):
            if kpi_x in row and kpi_y in row and 'action' in row:
                # Extract KPI values and action for this user
                if isinstance(row[kpi_x], list) and user_id < len(row[kpi_x]):
                    x_value = row[kpi_x][user_id]
                    y_value = row[kpi_y][user_id]
                    action_value = row['action'][user_id] if isinstance(row['action'], list) else row['action']
                    
                    # Add to combined data
                    combined_data.append({
                        'user_id': user_id,
                        'timestep': row['timestep'],
                        kpi_x: x_value,
                        kpi_y: y_value,
                        'action': action_value
                    })
    
    # Convert to DataFrame
    combined_df = pd.DataFrame(combined_data)
    
    
    
    # Calculate overall MI for comparison
    total_mi = 0
    if len(combined_df) > 0:
        try:
            total_mi = mutual_info_score(combined_df[kpi_x], combined_df[kpi_y])
        except Exception as e:
            print(f"Could not calculate overall MI: {e}")
    
    # Create a figure with 6 subplots (2x3 grid)
    # Increase the figure size significantly
    fig = plt.figure(figsize=(figsize[0]*3, figsize[1]*2))
    gs = fig.add_gridspec(2, 3, hspace=0.4, wspace=0.3)
    axes = []
    
    # First row: first 3 actions
    for i in range(3):
        axes.append(fig.add_subplot(gs[0, i]))
    
    # Second row: remaining 3 actions
    for i in range(3):
        axes.append(fig.add_subplot(gs[1, i]))
    
    # Create a matrix for each action and store vmax values for the title
    vmax_values = []
    
    # Create a matrix for each action
    for i, action in enumerate(all_actions):
        ax = axes[i]
        
        # Filter data for this action
        action_data = combined_df[combined_df['action'] == action]
        
        # Calculate MI for this action if possible
        mi_value = 0.0
        if len(action_data) > 5 and len(action_data[kpi_x].unique()) > 1 and len(action_data[kpi_y].unique()) > 1:
            try:
                mi_value = mutual_info_score(action_data[kpi_x], action_data[kpi_y])
            except Exception as e:
                print(f"Could not calculate MI for {action}: {e}")
        
        # Create a cross-tabulation of kpi_x vs kpi_y for this action
        if not action_data.empty:
            # Calculate the frequency of this combination for this action
            action_matrix = pd.crosstab(index=action_data[kpi_y], 
                                        columns=action_data[kpi_x])
            
            # Reindex to ensure all symbolic states are represented
            action_matrix = action_matrix.reindex(index=y_symbols, columns=x_symbols, fill_value=0)
            
            # Normalize if requested
            if normalize:
                total = combined_df.shape[0]  # Total number of data points
                action_matrix = (action_matrix / total) * 100  # Convert to percentage
        else:
            # Create empty matrix with all zeros if no data
            action_matrix = pd.DataFrame(0, index=y_symbols, columns=x_symbols)
        
        # Calculate dynamic vmax for this subplot
        max_value = action_matrix.values.max()
        vmax = max(min_vmax, max_value * vmax_multiplier)
        vmax_values.append(vmax)
        
        # Create heatmap with improved annotation size and format and dynamic vmax
        sns.heatmap(action_matrix, 
                    annot=True, 
                    fmt='.1f' if normalize else '.0f', 
                    cmap=cmap, 
                    vmin=0, 
                    vmax=vmax,  # Dynamic vmax based on data
                    cbar_kws={'label': 'Percentage (%)' if normalize else 'Count'}, 
                    linewidths=0.5, 
                    annot_kws={"size": 9},  # Increase annotation font size
                    ax=ax)
        
        # Set title with action name, MI score, and vmax value
        ax.set_title(f"Action: {action}\nMI: {mi_value:.2f} (vmax: {vmax:.1f})", fontsize=14)
        
        # Set axis labels (only on outer subplots) with increased font size
        if i in [0, 3]:  # Left edge subplots
            ax.set_ylabel(f'{kpi_y.upper()} State', fontsize=12)
        else:
            ax.set_ylabel('')
            
        if i >= 3:  # Bottom row subplots
            ax.set_xlabel(f'{kpi_x.upper()} State', fontsize=12)
        else:
            ax.set_xlabel('')
        
        # Rotate and adjust x and y tick labels for better readability
        # For x ticks: rotate more and adjust horizontal alignment
        plt.setp(ax.get_xticklabels(), rotation=90, ha='center', fontsize=8)
        plt.setp(ax.get_yticklabels(), rotation=0, fontsize=8)
        
        # Add axis ticks to only show certain labels (to reduce crowding)
        ax.xaxis.set_tick_params(labelbottom=True)
        ax.yaxis.set_tick_params(labelleft=True)
    
    # Add a main title with larger font and overall MI
    main_title = f"ACTION SELECTION BASED ON {kpi_x.upper()} AND {kpi_y.upper()} STATES\nOverall MI: {total_mi:.2f}"
    fig.suptitle(main_title, fontsize=18, y=0.98)
    
    # Save figure if requested
    if do_save:
        # Use the create_visualization_folder_path function to get the save location
        plot_folder_path = create_visualization_folder_path(vis_name, dataset_name, specific_name)
        
        # Create timestamp for unique filename
        timestamp = datetime.datetime.now().strftime("%B-%d_%H%M")
        
        # Create the filename
        filename_parts = [vis_name, f"{kpi_x}_vs_{kpi_y}", dataset_name]
        if specific_name:
            filename_parts.append(specific_name)
        
        base_filename = "_".join(part for part in filename_parts if part) + "_" + timestamp + ".png"
        
        # Full path to save the file
        save_path = os.path.join(plot_folder_path, base_filename)
        
        # Save the figure with high DPI for better quality
        plt.savefig(save_path, bbox_inches='tight', dpi=300)
        print(f"Figure saved to: {save_path}")
    
    # Show figure if requested
    if show:
        plt.show()
    else:
        plt.close(fig)
    
    return fig, axes


In [ ]:
# # Example usage
# fig, axes = visualize_aggregated_kpi_analysis(
#     df=test_symbolic_df,
#     kpi_cols=['mase', 'ddtu', 'rddtu_a', 'rddtu_b'],  # You can include as many KPIs as you need
#     dataset_name='mimo',
#     specific_name='aggregated_view',
#     show=True,
#     do_save=True
# )

fig, axes = visualize_multi_user_kpi_analysis(
    df=test_symbolic_df,
    kpi_col='mase_all',
    dataset_name='mimo',
    specific_name='multi_user_view',
    show=True,
    do_save=False
)

# fig, axes = visualize_multivariate_kpi_action_relationship(
#     df=test_symbolic_df,
#     kpi_x='mase',
#     kpi_y='ddtu',
#     normalize=True,
#     vmax_multiplier=2.0,  # Set vmax to 2x the maximum value in each subplot
#     min_vmax=0.5,         # Minimum vmax to use if values are very small
#     dataset_name='mimo',
#     specific_name='dynamic_vmax',
#     show=True,
#     do_save=True
# )

## Graph Training

In [ ]:
def process_symbolic_data(symbolic_df, kpi_list, action_column='action'):
    """
    Process symbolic data and build knowledge graphs based on the updated format
    where each row contains lists of KPI values for multiple users.
    
    For each timestep:
    - Process each user's data in the lists
    - Track the previous state for each user
    - Apply transitions between corresponding user states

    Args:
        symbolic_df: DataFrame with symbolic representation where KPI values and actions are lists
        kpi_list: List of KPIs to process
        action_column: Column name to use for the action in the knowledge graph
                      (default: 'action')

    Returns:
        Dictionary of knowledge graphs, one per KPI
    """
    # Create a knowledge graph for each KPI
    rt_kg = {kpi: KnowledgeGraph([kpi]) for kpi in kpi_list}
    
    # Dictionary to store previous timestep states for each KPI and user
    previous_timestep_states = {}
    for kpi in kpi_list:
        previous_timestep_states[kpi] = {}
    
    # Get all unique timesteps to process them in order
    unique_timesteps = sorted(symbolic_df['timestep'].unique())
    
    for timestep in unique_timesteps:
        print(f"\rProcessing timestep {timestep}/{max(unique_timesteps)}", end='', flush=True)
        
        # Filter data for current timestep
        timestep_data = symbolic_df[symbolic_df['timestep'] == timestep]
        
        
        # We expect only one row per timestep in the new format
        if len(timestep_data) > 0:
            row = timestep_data.iloc[0]
            reward = row['reward']
            
            # Get the actions for all users
            actions = row[action_column]
            
            # print(row.to_dict())
            # For each KPI, process data for each user
            for kpi in kpi_list:
                if kpi in row and isinstance(row[kpi], list):
                    kpi_values = row[kpi]
                    
                    # Process each user's data
                    for user_idx, kpi_value in enumerate(kpi_values):
                        # Get the action for this user
                        user_action = actions[user_idx] if isinstance(actions, list) and user_idx < len(actions) else "unknown"
                        
                        
                        # Create unique user identifier
                        user_key = f"U{user_idx}"
                        
                        # Create symbolic form for current state
                        current_state = {
                            kpi: kpi_value,
                            'action': user_action,
                            'user': user_idx,
                            'reward': reward
                        }
                        
                        # Get previous state for this KPI and user
                        prev_state = previous_timestep_states[kpi].get(user_key)
                        
                        # print(f"User {user_idx}")
                        # print(f"  Previous state: {prev_state}")
                        # print(f"  Current state: {current_state}")
                        
                        rt_kg[kpi].update_graph(current_state, prev_state, print_log=False)
                        
                        # Store current state for use in next timestep
                        previous_timestep_states[kpi][user_key] = current_state
                        # print("\n")
                        # if user_idx > 2:
                        #     break
        # print("#" * 50)
        # if timestep > 10:
    
    return rt_kg

# Define the KPIs you want to process
kpi_list = ['mase', 'mase_all', 'dtu', 'ddtu', 'rddtu_a', 'rddtu_b']

# Example usage
rt_kg = process_symbolic_data(train_symbolic_df, kpi_list, action_column='action')

In [ ]:
plot_path = create_visualization_folder_path("knowledge_graphs", "test_data")
# Generate and save visualizations for each knowledge graph
for key, kg in rt_kg.items():
    print(f"\n{'='*50}")
    print(f"Generating graph for {key}...")
    
    # Get the graph and network visualization
    G, net = kg.get_graph()
    
    net.write_html(os.path.join(plot_path, f"{key}.html"))
    

## Graph Visualization - General Stats per kpi

In [ ]:
def process_symbolic_data(symbolic_df, kpi_list, action_column='action'):
    """
    Process symbolic data and build knowledge graphs based on the updated format
    where each row contains lists of KPI values for multiple users.
    
    For each timestep:
    - Process each user's data in the lists
    - Track the previous state for each user
    - Apply transitions between corresponding user states

    Args:
        symbolic_df: DataFrame with symbolic representation where KPI values and actions are lists
        kpi_list: List of KPIs to process
        action_column: Column name to use for the action in the knowledge graph
                      (default: 'action')

    Returns:
        Dictionary of knowledge graphs, one per KPI
    """
    # Create a knowledge graph for each KPI
    rt_kg = {kpi: KnowledgeGraph([kpi]) for kpi in kpi_list}
    
    # Dictionary to store previous timestep states for each KPI and user
    previous_timestep_states = {}
    for kpi in kpi_list:
        previous_timestep_states[kpi] = {}
    
    # Get all unique timesteps to process them in order
    unique_timesteps = sorted(symbolic_df['timestep'].unique())
    
    for timestep in unique_timesteps:
        print(f"\rProcessing timestep {timestep}/{max(unique_timesteps)}", end='', flush=True)
        
        # Filter data for current timestep
        timestep_data = symbolic_df[symbolic_df['timestep'] == timestep]
        
        
        # We expect only one row per timestep in the new format
        if len(timestep_data) > 0:
            row = timestep_data.iloc[0]
            reward = row['reward']
            
            # Get the actions for all users
            actions = row[action_column]
            
            # print(row.to_dict())
            # For each KPI, process data for each user
            for kpi in kpi_list:
                if kpi in row and isinstance(row[kpi], list):
                    kpi_values = row[kpi]
                    
                    # Process each user's data
                    for user_idx, kpi_value in enumerate(kpi_values):
                        # Get the action for this user
                        user_action = actions[user_idx] if isinstance(actions, list) and user_idx < len(actions) else "unknown"
                        
                        
                        # Create unique user identifier
                        user_key = f"U{user_idx}"
                        
                        # Create symbolic form for current state
                        current_state = {
                            kpi: kpi_value,
                            'action': user_action,
                            'user': user_idx,
                            'reward': reward
                        }
                        
                        # Get previous state for this KPI and user
                        prev_state = previous_timestep_states[kpi].get(user_key)
                        
                        # print(f"User {user_idx}")
                        # print(f"  Previous state: {prev_state}")
                        # print(f"  Current state: {current_state}")
                        
                        rt_kg[kpi].update_graph(current_state, prev_state, print_log=False)
                        
                        # Store current state for use in next timestep
                        previous_timestep_states[kpi][user_key] = current_state
                        # print("\n")
                        # if user_idx > 2:
                        #     break
        # print("#" * 50)
        # if timestep > 10:
    
    return rt_kg

# Define the KPIs you want to process
kpi_list = ['mase', 'mase_all', 'dtu', 'ddtu', 'rddtu_a', 'rddtu_b']

# Example usage
rt_kg = process_symbolic_data(train_symbolic_df, kpi_list, action_column='action')

In [ ]:
def plot_node_edge_bar_chart(df, folder_path, dataset_name, specific_name=""):
    """
    Generates and saves a bar chart of nodes and edges per KPI.

    Args:
        df (pd.DataFrame): DataFrame containing KPI network data.
        folder_path (str): Base folder path to save visualizations.
        dataset_name (str): Dataset name for folder structure.
        specific_name (str, optional): Specific name for folder path.
    """
    vis_name = "network_stats_bar_charts"
    plot_folder_path = os.path.join(folder_path, vis_name) # Updated path - removed dataset_name from here
    os.makedirs(plot_folder_path, exist_ok=True) # Make sure folder exists

    fig, ax = plt.subplots(figsize=(14, 8), dpi=150) # Increased figsize and dpi
    bar_width = 0.35
    x = np.arange(len(df))

    bars1 = ax.bar(x - bar_width/2, df['nodes'], bar_width, label='Nodes', color = 'skyblue')
    bars2 = ax.bar(x + bar_width/2, df['edges'], bar_width, label='Edges', color = 'coral')

    # Add numbers on top of the bars
    for bar in bars1:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2., height, f'{int(height)}', ha='center', va='bottom', fontsize=12) # Increased fontsize
    for bar in bars2:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2., height, f'{int(height)}', ha='center', va='bottom', fontsize=12) # Increased fontsize

    ax.set_xlabel('KPI', fontsize = 14) # Increased fontsize
    ax.set_ylabel('Count', fontsize = 14) # Increased fontsize
    ax.set_title('Number of Nodes and Edges per KPI', fontsize = 16) # Increased fontsize
    ax.set_xticks(x)
    ax.set_xticklabels(df['kpi'], fontsize=12) # Increased fontsize
    ax.legend(fontsize = 12) # Increased fontsize
    ax.grid(axis = 'y', linestyle = '--', alpha = 0.7)
    plt.tight_layout()

    timestamp = datetime.datetime.now().strftime("%B-%d_%H%M")
    filename = f"node_edge_bar_chart_{dataset_name}_{specific_name}_{timestamp}.png" if specific_name else f"node_edge_bar_chart_{dataset_name}_{timestamp}.png"
    filepath = os.path.join(plot_folder_path, filename)
    plt.savefig(filepath, dpi=150) # Ensure DPI is passed to savefig as well
    plt.close(fig) # Close the figure to free memory
    print(f"Node and Edge Bar Chart saved to: {filepath}")

def plot_degree_stats_bar_chart(df, folder_path, dataset_name, specific_name=""):
    """
    Generates and saves a bar chart of degree statistics per KPI.

    Args:
        df (pd.DataFrame): DataFrame containing KPI network data.
        folder_path (str): Base folder path to save visualizations.
        dataset_name (str): Dataset name for folder structure.
        specific_name (str, optional): Specific name for folder path.
    """
    vis_name = "network_stats_bar_charts"
    plot_folder_path = os.path.join(folder_path, vis_name) # Updated path - removed dataset_name from here
    os.makedirs(plot_folder_path, exist_ok=True) # Make sure folder exists


    fig, ax = plt.subplots(figsize=(14, 8), dpi=150) # Increased figsize and dpi
    bar_width = 0.2
    x = np.arange(len(df))

    bars1 = ax.bar(x - bar_width, df['avg_degree'], bar_width, label='Average Degree', color = 'lightgreen')
    bars2 = ax.bar(x, df['min_degree'], bar_width, label='Min Degree', color = 'lightcoral')
    bars3 = ax.bar(x + bar_width, df['max_degree'], bar_width, label='Max Degree', color = 'lightblue')

    # Add numbers on top of the bars
    for bar in bars1:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2., height, f'{height:.2f}', ha='center', va='bottom', fontsize=12) # Increased fontsize
    for bar in bars2:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2., height, f'{height:.2f}', ha='center', va='bottom', fontsize=12) # Increased fontsize
    for bar in bars3:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2., height, f'{height:.2f}', ha='center', va='bottom', fontsize=12) # Increased fontsize

    ax.set_xlabel('KPI', fontsize = 14) # Increased fontsize
    ax.set_ylabel('Degree', fontsize = 14) # Increased fontsize
    ax.set_title('Degree Statistics per KPI', fontsize = 16) # Increased fontsize
    ax.set_xticks(x)
    ax.set_xticklabels(df['kpi'], fontsize=12) # Increased fontsize
    ax.legend(fontsize = 12) # Increased fontsize
    ax.grid(axis = 'y', linestyle = '--', alpha = 0.7)
    plt.tight_layout()

    timestamp = datetime.datetime.now().strftime("%B-%d_%H%M")
    filename = f"degree_stats_bar_chart_{dataset_name}_{specific_name}_{timestamp}.png" if specific_name else f"degree_stats_bar_chart_{dataset_name}_{timestamp}.png"
    filepath = os.path.join(plot_folder_path, filename)
    plt.savefig(filepath, dpi=150) # Ensure DPI is passed to savefig as well
    plt.close(fig) # Close the figure to free memory
    print(f"Degree Statistics Bar Chart saved to: {filepath}")

def plot_density_bar_chart(df, folder_path, dataset_name, specific_name=""):
    """
    Generates and saves a bar chart of graph density per KPI.

    Args:
        df (pd.DataFrame): DataFrame containing KPI network data.
        folder_path (str): Base folder path to save visualizations.
        dataset_name (str): Dataset name for folder structure.
        specific_name (str, optional): Specific name for folder path.
    """
    vis_name = "network_stats_bar_charts"
    plot_folder_path = os.path.join(folder_path, vis_name) # Updated path - removed dataset_name from here
    os.makedirs(plot_folder_path, exist_ok=True) # Make sure folder exists

    fig, ax = plt.subplots(figsize=(10, 6), dpi=150) # Increased figsize and dpi
    bars = ax.bar(df['kpi'], df['density'], color='lightpink')

    # Add numbers on top of the bars
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2., height, f'{height:.2f}', ha='center', va='bottom', fontsize=12) # Increased fontsize

    ax.set_xlabel('KPI', fontsize = 14) # Increased fontsize
    ax.set_ylabel('Density', fontsize = 14) # Increased fontsize
    ax.set_title('Graph Density per KPI', fontsize = 16) # Increased fontsize
    ax.grid(axis = 'y', linestyle = '--', alpha = 0.7)
    plt.tight_layout()

    timestamp = datetime.datetime.now().strftime("%B-%d_%H%M")
    filename = f"density_bar_chart_{dataset_name}_{specific_name}_{timestamp}.png" if specific_name else f"density_bar_chart_{dataset_name}_{timestamp}.png"
    filepath = os.path.join(plot_folder_path, filename)
    plt.savefig(filepath, dpi=150) # Ensure DPI is passed to savefig as well
    plt.close(fig) # Close the figure to free memory
    print(f"Density Bar Chart saved to: {filepath}")

def plot_connectivity_table(df, folder_path, dataset_name, specific_name=""):
    """
    Generates and saves a table of graph connectivity per KPI.

    Args:
        df (pd.DataFrame): DataFrame containing KPI network data.
        folder_path (str): Base folder path to save visualizations.
        dataset_name (str): Dataset name for folder structure.
        specific_name (str, optional): Specific name for folder path.
    """
    vis_name = "network_stats_tables"
    plot_folder_path = os.path.join(folder_path, vis_name) # Updated path - removed dataset_name from here
    os.makedirs(plot_folder_path, exist_ok=True) # Make sure folder exists

    fig, ax = plt.subplots(figsize=(14, 5), dpi=150) # Increased figsize and dpi
    ax.axis('off')

    table_data = []
    for index, row in df.iterrows():
        row_data = [row['kpi']]
        if row['is_directed']:
           row_data.append(str(row['strongly_connected'])) # Convert bool to str for table
           row_data.append(str(row['weakly_connected'])) # Convert bool to str for table
        else:
            row_data.append(str(row['connected'])) # Convert bool to str for table
        table_data.append(row_data)

    col_labels = ['KPI']
    if df['is_directed'].any():
        col_labels.extend(['Strongly Connected', 'Weakly Connected'])
    else:
        col_labels.append('Connected')

    table = ax.table(cellText=table_data,
                     colLabels=col_labels,
                     loc='center', cellLoc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(12) # Increased fontsize for table
    table.scale(1.2, 2) # Scale table for better readability
    plt.tight_layout()

    timestamp = datetime.datetime.now().strftime("%B-%d_%H%M")
    filename = f"connectivity_table_{dataset_name}_{specific_name}_{timestamp}.png" if specific_name else f"connectivity_table_{dataset_name}_{timestamp}.png"
    filepath = os.path.join(plot_folder_path, filename)
    plt.savefig(filepath, bbox_inches='tight', dpi=150) # Use bbox_inches='tight' for tables and ensure DPI is passed
    plt.close(fig) # Close the figure to free memory
    print(f"Connectivity Table saved to: {filepath}")

def plot_density_vs_degree_scatter(df, folder_path, dataset_name, specific_name=""):
    """
    Generates and saves a scatter plot of density vs. average degree per KPI.

    Args:
        df (pd.DataFrame): DataFrame containing KPI network data.
        folder_path (str): Base folder path to save visualizations.
        dataset_name (str): Dataset name for folder structure.
        specific_name (str, optional): Specific name for folder path.
    """
    vis_name = "network_stats_scatter_plots"
    plot_folder_path = os.path.join(folder_path, vis_name) # Updated path - removed dataset_name from here
    os.makedirs(plot_folder_path, exist_ok=True) # Make sure folder exists

    plt.figure(figsize=(10, 8), dpi=150) # Increased figsize and dpi
    plt.scatter(df['density'], df['avg_degree'], c='purple', s = 150) # Increased marker size

    for i, txt in enumerate(df['kpi']):
        plt.annotate(txt, (df['density'][i], df['avg_degree'][i]), textcoords="offset points", xytext=(5,5), ha='left', fontsize=12) # Increased fontsize

    plt.xlabel('Graph Density', fontsize = 14) # Increased fontsize
    plt.ylabel('Average Degree', fontsize = 14) # Increased fontsize
    plt.title('Graph Density vs. Average Degree', fontsize = 16) # Increased fontsize
    plt.grid(True, linestyle = '--', alpha = 0.7)
    plt.tight_layout()

    timestamp = datetime.datetime.now().strftime("%B-%d_%H%M")
    filename = f"density_vs_degree_scatter_{dataset_name}_{specific_name}_{timestamp}.png" if specific_name else f"density_vs_degree_scatter_{dataset_name}_{timestamp}.png"
    filepath = os.path.join(plot_folder_path, filename)
    plt.savefig(filepath, dpi=150) # Ensure DPI is passed to savefig as well
    plt.close(plt.gcf()) # Close the figure to free memory
    print(f"Density vs. Average Degree Scatter Plot saved to: {filepath}")

# Assuming rt_kg is already defined and populated
kpis = rt_kg.keys()
dataset = "test_data" # Dataset name
specific_name_suffix = "" # Optional specific name for folder

all_kpi_data = []
network_vis_folder = create_visualization_folder_path("network_graphs", dataset, specific_name_suffix) # Folder for network graphs

for kpi in kpis:
    g, net = rt_kg[kpi].get_graph()

    num_nodes = g.number_of_nodes()
    num_edges = g.number_of_edges()
    density = nx.density(g)
    is_directed = g.is_directed()
    
    # Initialize these variables with default values
    is_strongly_connected = False
    is_weakly_connected = False
    is_connected = False
    
    # Only check connectivity if the graph has nodes
    if num_nodes > 0:
        if is_directed:
            is_strongly_connected = nx.is_strongly_connected(g)
            is_weakly_connected = nx.is_weakly_connected(g)
        else:
            is_connected = nx.is_connected(g)

    degrees = [deg for _, deg in g.degree()]
    avg_degree = sum(degrees) / len(degrees) if degrees else 0
    min_degree = min(degrees) if degrees else 0
    max_degree = max(degrees) if degrees else 0

    kpi_data = {
        'kpi': kpi,
        'nodes': num_nodes,
        'edges': num_edges,
        'density': density,
        'avg_degree': avg_degree,
        'min_degree': min_degree,
        'max_degree': max_degree,
        'is_directed': is_directed,
    }

    if is_directed:
        kpi_data['strongly_connected'] = is_strongly_connected
        kpi_data['weakly_connected'] = is_weakly_connected
    else:
        kpi_data['connected'] = is_connected

    all_kpi_data.append(kpi_data)
    timestamp = datetime.datetime.now().strftime("%B-%d_%H%M")
    network_filename = f"{kpi}_network_graph_{dataset}_{specific_name_suffix}_{timestamp}.html" if specific_name_suffix else f"{kpi}_network_graph_{dataset}_{timestamp}.html"
    network_filepath = os.path.join(network_vis_folder, network_filename)
    net.save_graph(network_filepath) # Save network graph in the dedicated folder
    print(f"Network graph for {kpi} saved to: {network_filepath}")


df = pd.DataFrame(all_kpi_data)
folder_path = create_visualization_folder_path("", dataset, specific_name_suffix) # Updated base folder path - vis_name is "" now

# Generate and save plots
plot_node_edge_bar_chart(df, folder_path, dataset, specific_name_suffix)
plot_degree_stats_bar_chart(df, folder_path, dataset, specific_name_suffix)
plot_density_bar_chart(df, folder_path, dataset, specific_name_suffix)
plot_connectivity_table(df, folder_path, dataset, specific_name_suffix)
plot_density_vs_degree_scatter(df, folder_path, dataset, specific_name_suffix)

## Graph node probability distribution

In this visualizatoin we want to show that for each node in the graph what was the probability of each of the all possible actions that could be taken.

In [ ]:
def process_symbolic_data(symbolic_df, kpi_list, action_column='action'):
    """
    Process symbolic data and build knowledge graphs based on the updated format
    where each row contains lists of KPI values for multiple users.
    
    For each timestep:
    - Process each user's data in the lists
    - Track the previous state for each user
    - Apply transitions between corresponding user states

    Args:
        symbolic_df: DataFrame with symbolic representation where KPI values and actions are lists
        kpi_list: List of KPIs to process
        action_column: Column name to use for the action in the knowledge graph
                      (default: 'action')

    Returns:
        Dictionary of knowledge graphs, one per KPI
    """
    # Create a knowledge graph for each KPI
    rt_kg = {kpi: KnowledgeGraph([kpi]) for kpi in kpi_list}
    
    # Dictionary to store previous timestep states for each KPI and user
    previous_timestep_states = {}
    for kpi in kpi_list:
        previous_timestep_states[kpi] = {}
    
    # Get all unique timesteps to process them in order
    unique_timesteps = sorted(symbolic_df['timestep'].unique())
    
    for timestep in unique_timesteps:
        print(f"\rProcessing timestep {timestep}/{max(unique_timesteps)}", end='', flush=True)
        
        # Filter data for current timestep
        timestep_data = symbolic_df[symbolic_df['timestep'] == timestep]
        
        
        # We expect only one row per timestep in the new format
        if len(timestep_data) > 0:
            row = timestep_data.iloc[0]
            reward = row['reward']
            
            # Get the actions for all users
            actions = row[action_column]
            
            # print(row.to_dict())
            # For each KPI, process data for each user
            for kpi in kpi_list:
                if kpi in row and isinstance(row[kpi], list):
                    kpi_values = row[kpi]
                    
                    # Process each user's data
                    for user_idx, kpi_value in enumerate(kpi_values):
                        # Get the action for this user
                        user_action = actions[user_idx] if isinstance(actions, list) and user_idx < len(actions) else "unknown"
                        
                        
                        # Create unique user identifier
                        user_key = f"U{user_idx}"
                        
                        # Create symbolic form for current state
                        current_state = {
                            kpi: kpi_value,
                            'action': user_action,
                            'user': user_idx,
                            'reward': reward
                        }
                        
                        # Get previous state for this KPI and user
                        prev_state = previous_timestep_states[kpi].get(user_key)
                        
                        # print(f"User {user_idx}")
                        # print(f"  Previous state: {prev_state}")
                        # print(f"  Current state: {current_state}")
                        
                        rt_kg[kpi].update_graph(current_state, prev_state, print_log=False)
                        
                        # Store current state for use in next timestep
                        previous_timestep_states[kpi][user_key] = current_state
                        # print("\n")
                        # if user_idx > 2:
                        #     break
        # print("#" * 50)
        # if timestep > 10:
    
    return rt_kg

# Define the KPIs you want to process
kpi_list = ['mase', 'mase_all', 'dtu', 'ddtu', 'rddtu_a', 'rddtu_b']

# Example usage
rt_kg = process_symbolic_data(train_symbolic_df, kpi_list, action_column='action')

In [ ]:
from collections import defaultdict


# WITH this function
def create_symbolic_action_strings():
    """
    Creates all possible symbolic strings for actions, including group information.

    Returns:
        list: All possible symbolic action strings
    """
    action_types = ['notSched', 'Sched']
    groups = ['G0', 'G1', 'G2']  # Assuming groups 0, 1, and 2
    symbolic_strings = [f"{action_type}(user, {group})" for action_type in action_types for group in groups]
    return symbolic_strings

def create_symbolic_strings(kpi_name, forecast_variable=False):
    """
    Creates all possible symbolic strings for a KPI.
    
    Args:
        kpi_name (str): Name of the KPI
        include_pattern (bool): If True, adds a pattern dimension (dropping, fluctuating, spiking)
    
    Returns:
        list: All possible symbolic strings for the KPI
    """
    predicates = ['dec', 'const', 'inc']
    categories = ['VeryLow', 'Low', 'Medium', 'High', 'VeryHigh']
    
    if not forecast_variable:
        # Original functionality - predicate(kpi, category)
        symbolic_strings = [f"{predicate}({kpi_name}, {category})" for category in categories for predicate in predicates]
    else:
        # Extended functionality - predicate(kpi, category, pattern)
        patterns = ['dropping', 'fluctuating', 'spiking']
        symbolic_strings = [
            f"{predicate}({kpi_name}, {category}, {pattern})" 
            for category in categories 
            for predicate in predicates 
            for pattern in patterns
        ]
    
    return symbolic_strings

def calculate_action_probabilities(kg, reference_actions):
    """
    Calculate the probability of actions for each node in the knowledge graph.

    Args:
    kg (KnowledgeGraph): The knowledge graph object
    reference_actions (List[str]): List of all possible actions

    Returns:
    Dict[str, Dict[str, float]]: A dictionary of node names to action probabilities
    """
    node_probabilities = {}
    
    for node in kg.G.nodes:
        # print(f"Calculating probabilities for node: {node}")
        combined_actions = defaultdict(int)
        total_count = 0

        for _, successor, edge_data in kg.G.out_edges(node, data=True):
            # print(f"Edge data:")
            # pprint(edge_data)
            actions = edge_data.get('actions', {})
            for action, data in actions.items():
                count = data.get('count', 0)
                combined_actions[action] += count
                total_count += count
        
        # print("looped through edges")
        # print(f"Combined actions for node {node}:")
        # pprint(combined_actions)
        
        # print(reference_actions)
        
        probabilities = {}
        for action in reference_actions:
            probabilities[action] = combined_actions[action] / total_count if total_count > 0 else 0
        
        # print(f"Probabilities for node {node}:")
        # pprint(probabilities)
        
        node_probabilities[node] = probabilities

    return node_probabilities

def _plot_action_probabilities_plotly(node_probabilities, reference_actions, kpi_name, plot_folder_path, dataset_name, specific_name="", base_font_size=14):
    """
    Helper function to create and save the Plotly action probability plot.

    Args:
        node_probabilities: Dictionary of node probabilities
        reference_actions: List of action strings
        kpi_name: Name of the KPI
        plot_folder_path: Path to save the plot
        dataset_name: Name of the dataset
        specific_name: Additional name specification
        base_font_size: Base font size to control all text elements (default: 14)
    """
    # Calculate relative font sizes
    title_size = base_font_size * 1.7  # Main title
    axis_title_size = base_font_size * 1.3  # Axis titles
    tick_font_size = base_font_size * 0.85  # Tick labels
    legend_title_size = base_font_size * 1.1  # Legend title
    legend_text_size = base_font_size  # Legend text
    hover_font_size = base_font_size  # Hover text

    fig = go.Figure()

    # Get sorted KPI states for legend ordering
    if "_all" in kpi_name:
        sorted_kpi_states = create_symbolic_strings(kpi_name, forecast_variable=True)
    else:
        sorted_kpi_states = create_symbolic_strings(kpi_name)
    
    sorted_nodes = sorted(node_probabilities.keys(),
                     key=lambda x: sorted_kpi_states.index(x) if x in sorted_kpi_states else float('inf'))
    
    for node in sorted_nodes:
        probabilities = node_probabilities[node]
        y_values = [probabilities[action] for action in reference_actions]

        fig.add_trace(go.Scatter(
            x=reference_actions,
            y=y_values,
            mode='lines+markers',
            name=node,
            line=dict(width=2),
            marker=dict(size=8),
            hovertemplate="<b>Action:</b> %{x}<br>" +
                          "<b>Probability:</b> %{y:.3f}<br>" +
                          "<b>State:</b> " + node +
                          "<extra></extra>"
        ))

    fig.update_layout(
        title=dict(
            text=f'Action Probabilities for {kpi_name} KPI <br> (Dataset: {dataset_name})',
            font=dict(size=title_size, family="Arial", color="black", weight="bold"),
            x=0.5
        ),
        xaxis_title=dict(
            text='Scheduling Action',
            font=dict(size=axis_title_size, family="Arial", color="black", weight="bold")
        ),
        yaxis_title=dict(
            text="Probability",
            font=dict(size=axis_title_size, family="Arial", color="black", weight="bold")
        ),
        hovermode='x unified',
        hoverlabel=dict(
            bgcolor="white",
            font_size=hover_font_size,
            font_family="Arial",
            namelength=-1
        ),
        height=700,
        margin=dict(l=100, r=200, t=100, b=150),
        font=dict(size=base_font_size, family="Arial"),
        legend=dict(
            font=dict(size=legend_text_size, family="Arial"),
            yanchor="top",
            y=1,
            xanchor="left",
            x=1.05,
            orientation="v",
            traceorder='normal',
            title=dict(
                text="KPI State",
                font=dict(size=legend_title_size, family="Arial", weight="bold")
            )
        ),
        yaxis=dict(
            tickfont=dict(size=tick_font_size, family="Arial"),
            tickangle=0,
            showgrid=True,
            gridcolor='lightgray',
            range=[0, 1]
        ),
        xaxis=dict(
            tickmode='array',
            ticktext=reference_actions,
            tickvals=list(range(len(reference_actions))),
            tickfont=dict(size=tick_font_size, family="Arial"),
            tickangle=60,
            showgrid=True,
            gridcolor='lightgray'
        ),
        plot_bgcolor='rgba(240, 240, 240, 0.9)',
        paper_bgcolor='white'
    )

    fig.show()
    
    # timestamp = datetime.datetime.now().strftime("%H%M")
    # filename_base = f"action_prob_{kpi_name}_{dataset_name}_{specific_name}_{timestamp}" if specific_name else f"action_prob_{kpi_name}_{dataset_name}_{timestamp}"
    # png_filepath = os.path.join(plot_folder_path, filename_base + ".png")
    # html_filepath = os.path.join(plot_folder_path, filename_base + ".html")

    # fig.write_image(png_filepath, scale=2)
    # fig.write_html(html_filepath)

    # print(f"Action Probability Distribution plot for {kpi_name} saved to: {png_filepath} and {html_filepath}")

def plot_kpi_action_probabilities(rt_kg_test, kpi_name, folder_path, dataset_name, specific_name="", base_font_size=14):
    """
    Generates and saves action probability distribution plot for a given KPI.

    Args:
        rt_kg_test (dict): Dictionary of KnowledgeGraph objects.
        kpi_name (str): Name of the KPI to plot for.
        folder_path (str): Base folder path to save visualizations.
        dataset_name (str): Dataset name for folder structure.
        specific_name (str, optional): Specific name for folder path.
        base_font_size (int, optional): Base font size for all text elements. Default is 14.
    """
    vis_name = "action_probability_distributions"
    plot_folder_path = create_visualization_folder_path(vis_name, dataset_name, specific_name)

    # Use the new function to get the expanded action list with group information
    reference_actions = create_symbolic_action_strings()

    node_probabilities = calculate_action_probabilities(rt_kg_test[kpi_name], reference_actions)
    _plot_action_probabilities_plotly(node_probabilities, reference_actions, kpi_name, plot_folder_path, dataset_name, specific_name, base_font_size)


# Define the base font size once and use it for all plots
base_font_size = 18  # You can adjust this to make everything smaller or larger


# Assuming rt_kg_test and dataset are defined from previous notebook cells
dataset_name = 'test_data'  # Use dataset from notebook, e.g., 'car'
specific_name_suffix = "action_prob_dist" # Example specific name
folder_path = "visualizations" # Base visualization folder path


kpis_to_plot = rt_kg.keys()  # List of KPIs to plot

for kpi in kpis_to_plot:
    print(f"kpi: {kpi}")
    plot_kpi_action_probabilities(
        rt_kg,
        kpi,
        folder_path,
        dataset_name,
        specific_name_suffix,
        base_font_size=base_font_size
    )


## KPI influence calculation

In this section we wnat to see wether we can calculate the influence of each kpi on agent's action or not.

Train the graph on training dataset

The graph doesn't contain any information about the KPI states in the test dataset.

In [ ]:
def process_symbolic_data(symbolic_df, kpi_list, action_column='action'):
    """
    Process symbolic data and build knowledge graphs based on the updated format
    where each row contains lists of KPI values for multiple users.
    
    For each timestep:
    - Process each user's data in the lists
    - Track the previous state for each user
    - Apply transitions between corresponding user states

    Args:
        symbolic_df: DataFrame with symbolic representation where KPI values and actions are lists
        kpi_list: List of KPIs to process
        action_column: Column name to use for the action in the knowledge graph
                      (default: 'action')

    Returns:
        Dictionary of knowledge graphs, one per KPI
    """
    # Create a knowledge graph for each KPI
    rt_kg = {kpi: KnowledgeGraph([kpi]) for kpi in kpi_list}
    
    # Dictionary to store previous timestep states for each KPI and user
    previous_timestep_states = {}
    for kpi in kpi_list:
        previous_timestep_states[kpi] = {}
    
    # Get all unique timesteps to process them in order
    unique_timesteps = sorted(symbolic_df['timestep'].unique())
    
    for timestep in unique_timesteps:
        print(f"\rProcessing timestep {timestep}/{max(unique_timesteps)}", end='', flush=True)
        
        # Filter data for current timestep
        timestep_data = symbolic_df[symbolic_df['timestep'] == timestep]
        
        
        # We expect only one row per timestep in the new format
        if len(timestep_data) > 0:
            row = timestep_data.iloc[0]
            reward = row['reward']
            
            # Get the actions for all users
            actions = row[action_column]
            
            # print(row.to_dict())
            # For each KPI, process data for each user
            for kpi in kpi_list:
                if kpi in row and isinstance(row[kpi], list):
                    kpi_values = row[kpi]
                    
                    # Process each user's data
                    for user_idx, kpi_value in enumerate(kpi_values):
                        # Get the action for this user
                        user_action = actions[user_idx] if isinstance(actions, list) and user_idx < len(actions) else "unknown"
                        
                        
                        # Create unique user identifier
                        user_key = f"U{user_idx}"
                        
                        # Create symbolic form for current state
                        current_state = {
                            kpi: kpi_value,
                            'action': user_action,
                            'user': user_idx,
                            'reward': reward
                        }
                        
                        # Get previous state for this KPI and user
                        prev_state = previous_timestep_states[kpi].get(user_key)
                        
                        # print(f"User {user_idx}")
                        # print(f"  Previous state: {prev_state}")
                        # print(f"  Current state: {current_state}")
                        
                        rt_kg[kpi].update_graph(current_state, prev_state, print_log=False)
                        
                        # Store current state for use in next timestep
                        previous_timestep_states[kpi][user_key] = current_state
                        # print("\n")
                        # if user_idx > 2:
                        #     break
        # print("#" * 50)
        # if timestep > 10:
    
    return rt_kg

# Define the KPIs you want to process
kpi_list = ['mase', 'mase_all', 'ddtu', 'rddtu_a', 'rddtu_b']

# Example usage
rt_kg = process_symbolic_data(train_symbolic_df, kpi_list, action_column='action')

### IS with delta function in (binary, decaying and hierarchical) + representation

In [ ]:
from types import SimpleNamespace
import numpy as np
import re
from collections import defaultdict


def kl_divergence(p_dist, q_dist):
    """
    Calculates the Kullback-Leibler divergence D_KL(P || Q).
    """
    kl_div = 0.0
    actions_p = set(p_dist.keys())
    actions_q = set(q_dist.keys())
    all_actions = actions_p.union(actions_q)

    for action in all_actions:
        p_prob = p_dist.get(action, 0.0)
        q_prob = q_dist.get(action, 0.0)

        if p_prob > 0.0:
            if q_prob <= 0.0:
                continue
            else:
                kl_div += p_prob * np.log(p_prob / q_prob)

    return kl_div

def get_recommended_action(action_distribution):
    """
    Returns the action with the highest probability from the action distribution.
    """
    if not action_distribution:
        return None
    return max(action_distribution, key=action_distribution.get)

def binary_delta_function(recommended_action, actual_action):
    """
    Implements the binary delta function for scheduling actions.
    """
    if recommended_action is None or actual_action is None:
        return 0.0
    return 1.0 if recommended_action == actual_action else 0.0

def extract_scheduling_info(action_str):
    """
    Extracts scheduling information from action strings like 'Sched(user, G0)' or 'notSched(user, G0)'.

    Returns:
        tuple: (is_scheduled, group_id) where:
               - is_scheduled is True if action is 'Sched', False if 'notSched'
               - group_id is the group identifier (e.g., 'G0', 'G1', etc.)
    """
    if action_str is None:
        return None, None

    is_scheduled = action_str.startswith('Sched')

    # Extract group information
    match = re.search(r'(user, (G\d+))', action_str)
    if match:
        group_id = match.group(2)
        return is_scheduled, group_id
    else:
        return is_scheduled, None

def decaying_delta_function(recommended_action, actual_action, sigma=0.5):
    """
    Implements a decaying delta function for scheduling actions.
    The similarity is based on whether both actions schedule/not schedule,
    regardless of the group.
    """
    if recommended_action is None or actual_action is None:
        return 0.0

    rec_scheduled, rec_group = extract_scheduling_info(recommended_action)
    act_scheduled, act_group = extract_scheduling_info(actual_action)

    if rec_scheduled is None or act_scheduled is None:
        return 0.0

    # If scheduling decision matches
    if rec_scheduled == act_scheduled:
        # If groups also match, full similarity
        if rec_group == act_group:
            return 1.0
        # If only scheduling decision matches, partial similarity
        else:
            return 0.5
    # Different scheduling decisions
    return 0.0

def hierarchical_delta_function(recommended_action, actual_action):
    """
    Implements a hierarchical delta function for scheduling actions.
    First level: Schedule/Not Schedule
    Second level: Group match
    """
    if recommended_action is None or actual_action is None:
        return 0.0

    rec_scheduled, rec_group = extract_scheduling_info(recommended_action)
    act_scheduled, act_group = extract_scheduling_info(actual_action)

    if rec_scheduled is None or act_scheduled is None:
        return 0.0

    # First level - scheduling decision
    if rec_scheduled != act_scheduled:
        return 0.0

    # Second level - group match
    if rec_group == act_group:
        return 1.0
    else:
        return 0.5

def divergence_weighted_influence(rt_kg, kpis, row, agent_action, delta_function_type='binary', sigma_decay=0.5):
    """
    Calculates the divergence-weighted influence for each KPI, allowing different delta functions.
    """
    # print(f"\n\nCalculating Divergence-Weighted Influence for row: {row}")
    all_P_k = {}
    for kpi_name in kpis:
        # CHANGE THIS LINE - access KG directly by KPI name, not by KPI_group
        kg = rt_kg[kpi_name]  # Use the KPI name directly
        kpi_value = getattr(row, kpi_name)

        edges = kg.G.out_edges(kpi_value, data=True)

        P_k = defaultdict(float)
        total_occurrence = sum(edge[2]['occurrence'] for edge in edges)
        if total_occurrence == 0:
            print(f"Warning: No outgoing edges from state {kpi_value} for KPI {kpi_name}. P_k(a) will be empty.")
        else:
            for _, _, edge_data in edges:
                for action, action_data in edge_data['actions'].items():
                    P_k[action] += (action_data['count'] / total_occurrence)
        all_P_k[kpi_name] = P_k
    
    
    # pprint(all_P_k)
    
    # print(f"P_K Distribution:")
    # pprint(all_P_k)
    
    # Calculate Baseline Distribution P_emptyset(a)
    P_emptyset = defaultdict(float)
    num_kpis = len(kpis)
    all_actions = set()

    for kpi_name in kpis:
        P_k = all_P_k[kpi_name]
        all_actions.update(P_k.keys())

    for action in all_actions:
        P_emptyset[action] = 0.0

    for kpi_name in kpis:
        P_k = all_P_k[kpi_name]
        for action, prob in P_k.items():
            P_emptyset[action] += prob

    for action in P_emptyset:
        P_emptyset[action] /= num_kpis
    
    # print(f"Baseline Distribution P_emptyset(a):")
    # pprint(P_emptyset)
    
    influence_scores = {}
    kl_divergences = {}
    delta_values = {}
    
    for kpi_name in kpis:
        # print(f"\n\n Calculating Influence Scores for KPI: {kpi_name}")
        P_k = all_P_k[kpi_name]
        kl_div_value = kl_divergence(P_k, P_emptyset)
        kl_divergences[kpi_name] = kl_div_value
        
        recommended_action_kpi = get_recommended_action(P_k)
        
        # print(f"KPI: {kpi_name} ({getattr(row, kpi_name)})")
        # pprint(f"P_K Distribution:")
        # pprint(P_k)
        # pprint(f"Recommended Action: {recommended_action_kpi}")
        
        delta_value = 0 # Initialize delta_value
        
        # print(f"\n--> Agent Action: {agent_action}")
        # print(f"\n--> Recommended Action: {recommended_action_kpi}")
                
        if delta_function_type == 'binary':
            # print(f"\n--> Delta Function Type: Binary")
            delta_value = binary_delta_function(recommended_action_kpi, agent_action)
        elif delta_function_type == 'decaying':
            # print(f"\n--> Delta Function Type: Decaying with Sigma: {sigma_decay}")
            delta_value = decaying_delta_function(recommended_action_kpi, agent_action, sigma_decay)
            # print(f"Delta Value: {delta_value}")
            # print(f"influence score is {kl_div_value * delta_value}")
        elif delta_function_type == 'hierarchical':
            # print(f"\n--> Delta Function Type: Hierarchical")
            delta_value = hierarchical_delta_function(recommended_action_kpi, agent_action)
            # print(f"Delta Value: {delta_value}")
            # print(f"influence score is {kl_div_value * delta_value}")
        else:
            raise ValueError(f"Unknown delta_function_type: {delta_function_type}")

        delta_values[kpi_name] = delta_value
        influence_score = kl_div_value * delta_value
        influence_scores[kpi_name] = influence_score

    return influence_scores, kl_divergences, delta_values


influence_scores_list_binary = [] # List for binary delta influence scores
influence_scores_list_decaying = [] # List for decaying delta influence scores
influence_scores_list_hierarchical = [] # List for hierarchical delta influence scores

kpis_list_test = rt_kg.keys()

for index, row in test_symbolic_df.iterrows():
    num_users = len(row['group'])  # Dynamically determine user count

    all_users_influence_scores_binary = []
    all_users_influence_scores_decaying = []
    all_users_influence_scores_hierarchical = []

    for user_idx in range(num_users):
        # Create a temporary user row with single values
        user_row = SimpleNamespace()
        user_row.timestep = row['timestep']
        user_row.group = row['group'][user_idx]

        # Extract values for this user from each KPI
        for kpi_name in kpis_list_test:
            if kpi_name in row and user_idx < len(row[kpi_name]):
                setattr(user_row, kpi_name, row[kpi_name][user_idx])
            else:
                setattr(user_row, kpi_name, None)

        agent_action = row['action'][user_idx]

        # Calculate influence with Binary Delta Function
        influence_score_binary, kl_divergences_binary, delta_values_binary = divergence_weighted_influence(
            rt_kg, kpis_list_test, user_row, agent_action, delta_function_type='binary')
        all_users_influence_scores_binary.append(influence_score_binary)
        
        # Calculate influence with Decaying Delta Function
        influence_score_decaying, kl_divergences_decaying, delta_values_decaying = divergence_weighted_influence(
            rt_kg, kpis_list_test, user_row, agent_action, delta_function_type='decaying', sigma_decay=0.5)
        all_users_influence_scores_decaying.append(influence_score_decaying)
        
        # Calculate influence with Hierarchical Delta Function
        influence_score_hierarchical, kl_divergences_hierarchical, delta_values_hierarchical = divergence_weighted_influence(
            rt_kg, kpis_list_test, user_row, agent_action, delta_function_type='hierarchical')
        all_users_influence_scores_hierarchical.append(influence_score_hierarchical)
        
    influence_scores_list_binary.append({
        "timestep": row['timestep'],
        "influence_scores": all_users_influence_scores_binary,
        "delta_type": "binary"
    })
    
    influence_scores_list_decaying.append({
        "timestep": row['timestep'],
        "influence_scores": all_users_influence_scores_decaying,
        "delta_type": "decaying"
    })
    
    influence_scores_list_hierarchical.append({
        "timestep": row['timestep'],
        "influence_scores": all_users_influence_scores_hierarchical,
        "delta_type": "hierarchical"
    })

IS_df_binary = pd.DataFrame(influence_scores_list_binary)
IS_df_decaying = pd.DataFrame(influence_scores_list_decaying)
IS_df_hierarchical = pd.DataFrame(influence_scores_list_hierarchical)

# print("\n\nInfluence Scores DataFrame - Binary Delta (IS_df_binary):")
# print(IS_df_binary)
# # print("\nInfluence Scores DataFrame - Decaying Delta (IS_df_decaying):")
# # print(IS_df_decaying)
# # print("\nInfluence Scores DataFrame - Hierarchical Delta (IS_df_hierarchical):")
# # print(IS_df_hierarchical)

### Visualize - numerical values

In [ ]:
def plot_kpis_with_influence_scores(test_df, is_dfs, is_styles, kpi_name, timestep_col='timestep', output_filename=None):
    """
    Creates a visualization with 7 subplots (one for each user 0-6) showing raw KPI values and multiple influence scores.

    Parameters:
    -----------
    test_df : pandas.DataFrame
        DataFrame containing raw KPI values with columns like timestep, mse, dtu, etc.
    is_dfs : list
        List of DataFrames containing influence scores with columns timestep, influence_scores, delta_type
    is_styles : list
        List of line styles for each is_df (e.g., ['dash', 'dot', 'dashdot'])
    kpi_name : str
        Name of the KPI to plot ('mse', 'dtu', or 'mase_all')
    timestep_col : str, optional
        Name of the timestep column in test_df (default: 'timestep')
    output_filename : str, optional
        If provided, save the figure to this filename (HTML only)

    Returns:
    --------
    plotly.graph_objects.Figure
    """
    # Create a subplot for each user
    fig = make_subplots(
        rows=7, cols=1,
        subplot_titles=[f"User {i}" for i in range(7)],
        shared_xaxes=True,
        vertical_spacing=0.02,
        specs=[[{"secondary_y": True}] for _ in range(7)]
    )

    # Define colors for better visualization
    raw_color = '#1f77b4'  # Blue for raw values
    mase_fr1_color = '#9467bd'  # Purple for mase_fr1
    mase_fr2_color = '#17becf'  # Cyan for mase_fr2
    is_colors = ['#ff7f0e', '#2ca02c', '#d62728']  # Orange, Green, Red for influence scores
    scheduled_marker_color = 'green'  # Green markers for scheduled actions

    # Ensure lowercase column names for test_df
    test_df = test_df.copy()
    test_df.columns = [col.lower() if isinstance(col, str) else col for col in test_df.columns]

    # Extract action tuples using ast
    def parse_action(action_str):
        try:
            if isinstance(action_str, tuple):
                return action_str
            elif isinstance(action_str, str):
                return ast.literal_eval(action_str)
            else:
                return []
        except:
            return []

    # Process and add traces for each user
    for user_idx in range(7):
        # Extract raw KPI values for this user
        raw_values = []
        mase_fr1_values = [] 
        mase_fr2_values = []
        scheduled_timesteps = []
        timesteps = []

        for idx, row in test_df.iterrows():
            timestep = row[timestep_col]
            timesteps.append(timestep)

            # Parse action and check if user is scheduled
            action = parse_action(row['action'])
            if user_idx in action:
                scheduled_timesteps.append(timestep)

            # Special handling for mase_all
            if kpi_name == 'mase_all':
                if isinstance(row[kpi_name], list) and len(row[kpi_name]) > user_idx:
                    user_mase_values = row[kpi_name][user_idx]
                    if isinstance(user_mase_values, list) and len(user_mase_values) >= 3:
                        # Extract the three values from the user's mase_all entry
                        raw_values.append(user_mase_values[0])  # First value as main MASE_all
                        mase_fr1_values.append(user_mase_values[1])  # Second value as mase_fr1
                        mase_fr2_values.append(user_mase_values[2])  # Third value as mase_fr2
                    else:
                        raw_values.append(None)
                        mase_fr1_values.append(None)
                        mase_fr2_values.append(None)
                else:
                    raw_values.append(None)
                    mase_fr1_values.append(None)
                    mase_fr2_values.append(None)
            else:
                # Extract KPI value for regular KPIs
                if isinstance(row[kpi_name], (list, np.ndarray)) and len(row[kpi_name]) > user_idx:
                    raw_values.append(row[kpi_name][user_idx])
                else:
                    raw_values.append(None)  # Handle missing data

        # Add trace for raw values (as a line)
        fig.add_trace(
            go.Scatter(
                x=timesteps,
                y=raw_values,
                mode='lines',
                name=f'Raw {kpi_name.upper()}',
                line=dict(color=raw_color),
                legendgroup='raw',
                showlegend=(user_idx == 0)  # Only show in legend for first user
            ),
            row=user_idx+1, col=1, secondary_y=False
        )

        # Add additional traces for mase_all
        if kpi_name == 'mase_all':
            # Add mase_fr1 trace
            fig.add_trace(
                go.Scatter(
                    x=timesteps,
                    y=mase_fr1_values,
                    mode='lines',
                    name='mase_fr1',
                    line=dict(color=mase_fr1_color),
                    legendgroup='mase_fr1',
                    showlegend=(user_idx == 0)  # Only show in legend for first user
                ),
                row=user_idx+1, col=1, secondary_y=False
            )
            
            # Add mase_fr2 trace
            fig.add_trace(
                go.Scatter(
                    x=timesteps,
                    y=mase_fr2_values,
                    mode='lines',
                    name='mase_fr2',
                    line=dict(color=mase_fr2_color),
                    legendgroup='mase_fr2',
                    showlegend=(user_idx == 0)  # Only show in legend for first user
                ),
                row=user_idx+1, col=1, secondary_y=False
            )

        # Add markers for scheduled timesteps
        if scheduled_timesteps:
            # Get the values at the scheduled timesteps
            scheduled_values = []
            for ts in scheduled_timesteps:
                idx = timesteps.index(ts)
                scheduled_values.append(raw_values[idx])

            fig.add_trace(
                go.Scatter(
                    x=scheduled_timesteps,
                    y=scheduled_values,
                    mode='markers',
                    marker=dict(
                        color=scheduled_marker_color,
                        size=10,
                        symbol='circle'
                    ),
                    name='Scheduled',
                    legendgroup='scheduled',
                    showlegend=(user_idx == 0)  # Only show in legend for first user
                ),
                row=user_idx+1, col=1, secondary_y=False
            )

        # Process and add influence score traces for each IS dataframe
        for df_idx, (is_df, is_style) in enumerate(zip(is_dfs, is_styles)):
            is_values = []
            is_timesteps = []
            delta_type = is_df['delta_type'].iloc[0] if 'delta_type' in is_df else f"type_{df_idx}"

            for _, row in is_df.iterrows():
                is_timesteps.append(row['timestep'])
                influence_scores = row['influence_scores']

                # Check if influence_scores is a list of dictionaries
                if isinstance(influence_scores, list) and len(influence_scores) > user_idx:
                    if isinstance(influence_scores[user_idx], dict) and kpi_name in influence_scores[user_idx]:
                        is_values.append(influence_scores[user_idx][kpi_name])
                    else:
                        is_values.append(None)
                else:
                    is_values.append(None)

            # Add trace for influence scores with appropriate style
            fig.add_trace(
                go.Scatter(
                    x=is_timesteps,
                    y=is_values,
                    mode='lines',
                    name=f'IS {delta_type}',
                    line=dict(
                        color=is_colors[df_idx % len(is_colors)], 
                        dash=is_style
                    ),
                    legendgroup=f'is_{delta_type}',
                    showlegend=(user_idx == 0)  # Only show in legend for first user
                ),
                row=user_idx+1, col=1, secondary_y=True
            )

        # Update y-axis titles
        fig.update_yaxes(
            title_text=f"{kpi_name.upper()} Value",
            secondary_y=False,
            row=user_idx+1, col=1
        )

        fig.update_yaxes(
            title_text="Influence Score",
            secondary_y=True,
            row=user_idx+1, col=1
        )

    # Update layout
    fig.update_layout(
        height=1400,  # Taller figure to accommodate 7 subplots
        title_text=f"{kpi_name.upper()} with Influence Scores",
        hovermode='x unified',
        legend_title="Metrics",
    )

    # Update x-axis title only for the bottom subplot
    fig.update_xaxes(title_text="Timestep", row=7, col=1)

    # Save if filename provided (HTML only)
    if output_filename:
        fig.write_html(f"{output_filename}.html")

    return fig

def plot_multiple_kpis_with_influence_scores(test_df, is_dfs, kpi_list, is_styles=None, output_dir='plots', timestamp_suffix=False, timestep_col='timestep'):
    """
    Creates visualizations for multiple KPIs from a list with multiple influence score dataframes.

    Parameters:
    -----------
    test_df : pandas.DataFrame
        DataFrame containing raw KPI values with columns like timestep, mse, dtu, etc.
    is_dfs : list
        List of DataFrames containing influence scores with columns timestep, influence_scores, delta_type
    kpi_list : list
        List of KPI names to plot (e.g., ['mse', 'dtu', 'ddtu'])
    is_styles : list, optional
        List of line styles for each is_df (default: ['dash', 'dot', 'dashdot'])
    output_dir : str, optional
        Directory to save plots (default: 'plots')
    timestamp_suffix : bool, optional
        Whether to add a timestamp suffix to output filenames (default: False)
    timestep_col : str, optional
        Name of the timestep column in test_df (default: 'timestep')

    Returns:
    --------
    dict: Dictionary of figures indexed by KPI name
    """
    # Set default styles if none provided
    if is_styles is None:
        is_styles = ['dash', 'dot', 'dashdot'][:len(is_dfs)]
        
    # Create output directory if it doesn't exist
    vis_name = "IS_Plots"
    folder_path = create_visualization_folder_path(vis_name, "test_data")

    # Add timestamp suffix to filenames if requested
    time_suffix = datetime.datetime.now().strftime("_%Y%m%d_%H%M") if timestamp_suffix else ""

    # Dictionary to store all figures
    figures = {}

    # Get delta types for filename
    delta_types = []
    for is_df in is_dfs:
        if 'delta_type' in is_df:
            delta_type = is_df['delta_type'].iloc[0]
            if delta_type not in delta_types:
                delta_types.append(delta_type)
    
    delta_str = "_".join(delta_types) if delta_types else "combined"

    # Plot each KPI
    for kpi_name in kpi_list:
        print(f"Plotting {kpi_name.upper()}...")

        # Generate shorter output filename
        output_filename = os.path.join(folder_path, f"{kpi_name}_{delta_str}{time_suffix}")

        # Create the figure
        fig = plot_kpis_with_influence_scores(
            test_df=test_df,
            is_dfs=is_dfs,
            is_styles=is_styles,
            kpi_name=kpi_name,
            timestep_col=timestep_col,
            output_filename=output_filename
        )

        # Store the figure in the dictionary
        figures[kpi_name] = fig

        print(f"Saved plot to {output_filename}.html")

    return figures

# Example usage:
# Define list of KPIs to plot
kpi_list = ['mase', 'mase_all', 'ddtu', 'rddtu_b']  # Add all your KPIs here

# Create directory for plots
output_directory = "is_plots"

# Define line styles for each IS dataframe
is_styles = ['dash', 'dot']

# Generate all plots with multiple IS dataframes
figures = plot_multiple_kpis_with_influence_scores(
    test_df=test_df,
    is_dfs=[IS_df_binary, IS_df_decaying],  # Multiple IS dataframes
    kpi_list=kpi_list,
    is_styles=is_styles,
    output_dir=output_directory
)

# Display a specific figure if needed
# figures['mase'].show()

### Visualize - specialized Scenario (focus on _all kpi)

In [ ]:
def plot_user_mase_with_influence_scores_and_symbolic(test_df, test_symbolic_df, is_dfs, is_styles, user_idx, timestep_col='timestep', output_filename=None):
    """
    Creates a visualization for a specific user with three subplots:
    1. MASE values (mase and three values from mase_all)
    2. Influence scores for those KPIs
    3. Symbolic descriptions of mase and mase_all
    
    Parameters:
    -----------
    test_df : pandas.DataFrame
        DataFrame containing raw KPI values with columns like timestep, mase, mase_all, etc.
    test_symbolic_df : pandas.DataFrame
        DataFrame containing symbolic descriptions of KPIs
    is_dfs : list
        List of DataFrames containing influence scores with columns timestep, influence_scores, delta_type
    is_styles : list
        List of line styles for each is_df (e.g., ['dash', 'dot', 'dashdot'])
    user_idx : int
        Index of the user to plot (0-6)
    timestep_col : str, optional
        Name of the timestep column in test_df (default: 'timestep')
    output_filename : str, optional
        If provided, save the figure to this filename (HTML only)
        
    Returns:
    --------
    plotly.graph_objects.Figure
    """
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    import numpy as np
    import ast
    
    # Create a subplot with 3 rows, 1 column with dual y-axes for the third row
    fig = make_subplots(
        rows=3, cols=1,
        subplot_titles=[
            f"User {user_idx}: MASE Values", 
            f"User {user_idx}: Influence Scores",
            f"User {user_idx}: Symbolic Values"
        ],
        vertical_spacing=0.08,  # Reduced spacing between subplots
        row_heights=[0.3, 0.2, 0.5],  # Rebalanced: More space for symbolic, less for others
        shared_xaxes=True,
        specs=[
            [{"secondary_y": False}],  # First row, no secondary y-axis
            [{"secondary_y": False}],  # Second row, no secondary y-axis
            [{"secondary_y": True}]    # Third row, with secondary y-axis
        ]
    )
    
    # Define colors for better visualization
    mase_color = '#1f77b4'  # Blue for mase
    mase_all_color = '#ff7f0e'  # Orange for mase_all (first value)
    mase_fr1_color = '#9467bd'  # Purple for mase_fr1 (second value)
    mase_fr2_color = '#17becf'  # Cyan for mase_fr2 (third value)
    
    is_colors = ['#d62728', '#2ca02c', '#8c564b']  # Red, Green, Brown for influence scores
    
    # Ensure lowercase column names for test_df and test_symbolic_df
    test_df = test_df.copy()
    test_df.columns = [col.lower() if isinstance(col, str) else col for col in test_df.columns]
    
    test_symbolic_df = test_symbolic_df.copy()
    test_symbolic_df.columns = [col.lower() if isinstance(col, str) else col for col in test_symbolic_df.columns]
    
    # Extract action tuples using ast
    def parse_action(action_str):
        try:
            if isinstance(action_str, tuple):
                return action_str
            elif isinstance(action_str, str):
                return ast.literal_eval(action_str)
            else:
                return []
        except:
            return []
    
    # Extract data for this user (numeric values)
    mase_values = []
    mase_all_values = []
    mase_fr1_values = []
    mase_fr2_values = []
    scheduled_timesteps = []
    timesteps = []
    
    for idx, row in test_df.iterrows():
        timestep = row[timestep_col]
        timesteps.append(timestep)
        
        # Parse action and check if user is scheduled
        action = parse_action(row['action'])
        if user_idx in action:
            scheduled_timesteps.append(timestep)
        
        # Extract mase value
        if 'mase' in row and isinstance(row['mase'], (list, np.ndarray)) and len(row['mase']) > user_idx:
            mase_values.append(row['mase'][user_idx])
        else:
            mase_values.append(None)
        
        # Extract mase_all values (all three components)
        if 'mase_all' in row and isinstance(row['mase_all'], list) and len(row['mase_all']) > user_idx:
            user_mase_values = row['mase_all'][user_idx]
            if isinstance(user_mase_values, list) and len(user_mase_values) >= 3:
                mase_all_values.append(user_mase_values[0])  # First value as main MASE_all
                mase_fr1_values.append(user_mase_values[1])  # Second value as mase_fr1
                mase_fr2_values.append(user_mase_values[2])  # Third value as mase_fr2
            else:
                mase_all_values.append(None)
                mase_fr1_values.append(None)
                mase_fr2_values.append(None)
        else:
            mase_all_values.append(None)
            mase_fr1_values.append(None)
            mase_fr2_values.append(None)
    
    # Extract symbolic data for this user
    symbolic_timesteps = []
    symbolic_mase = []
    symbolic_mase_all = []
    
    for idx, row in test_symbolic_df.iterrows():
        timestep = row[timestep_col]
        symbolic_timesteps.append(timestep)
        
        # Extract symbolic mase value for this user
        if 'mase' in row and isinstance(row['mase'], list) and len(row['mase']) > user_idx:
            symbolic_mase.append(row['mase'][user_idx])
        else:
            symbolic_mase.append(None)
        
        # Extract symbolic mase_all value for this user
        if 'mase_all' in row and isinstance(row['mase_all'], list) and len(row['mase_all']) > user_idx:
            symbolic_mase_all.append(row['mase_all'][user_idx])
        else:
            symbolic_mase_all.append(None)
    
    # Add traces for MASE values in the first subplot
    
    # Add mase trace
    fig.add_trace(
        go.Scatter(
            x=timesteps,
            y=mase_values,
            mode='lines',
            name='MASE',
            line=dict(color=mase_color, width=2.5),
            legendgroup='mase'
        ),
        row=1, col=1
    )
    
    # Add mase_all (first component) trace
    fig.add_trace(
        go.Scatter(
            x=timesteps,
            y=mase_all_values,
            mode='lines',
            name='MASE_all',
            line=dict(color=mase_all_color, width=2.5),
            legendgroup='mase_all'
        ),
        row=1, col=1
    )
    
    # Add mase_fr1 (second component) trace
    fig.add_trace(
        go.Scatter(
            x=timesteps,
            y=mase_fr1_values,
            mode='lines',
            name='mase_fr1',
            line=dict(color=mase_fr1_color, width=2.5),
            legendgroup='mase_fr1'
        ),
        row=1, col=1
    )
    
    # Add mase_fr2 (third component) trace
    fig.add_trace(
        go.Scatter(
            x=timesteps,
            y=mase_fr2_values,
            mode='lines',
            name='mase_fr2',
            line=dict(color=mase_fr2_color, width=2.5),
            legendgroup='mase_fr2'
        ),
        row=1, col=1
    )
    
    # Add markers for scheduled timesteps (if any)
    if scheduled_timesteps:
        # Get the mase values at the scheduled timesteps
        scheduled_values = []
        for ts in scheduled_timesteps:
            idx = timesteps.index(ts)
            scheduled_values.append(mase_values[idx])
        
        fig.add_trace(
            go.Scatter(
                x=scheduled_timesteps,
                y=scheduled_values,
                mode='markers',
                marker=dict(
                    color='green',
                    size=12,  # Increased marker size
                    symbol='circle'
                ),
                name='Scheduled',
                legendgroup='scheduled'
            ),
            row=1, col=1
        )
    
    # Process and add influence score traces for each IS dataframe in the second subplot
    kpi_names = ['mase', 'mase_all']
    
    for kpi_idx, kpi_name in enumerate(kpi_names):
        for df_idx, (is_df, is_style) in enumerate(zip(is_dfs, is_styles)):
            is_values = []
            is_timesteps = []
            delta_type = is_df['delta_type'].iloc[0] if 'delta_type' in is_df else f"type_{df_idx}"
            
            for _, row in is_df.iterrows():
                is_timesteps.append(row['timestep'])
                influence_scores = row['influence_scores']
                
                # Check if influence_scores is a list of dictionaries
                if isinstance(influence_scores, list) and len(influence_scores) > user_idx:
                    if isinstance(influence_scores[user_idx], dict) and kpi_name in influence_scores[user_idx]:
                        is_values.append(influence_scores[user_idx][kpi_name])
                    else:
                        is_values.append(None)
                else:
                    is_values.append(None)
            
            # Determine trace color based on KPI
            line_color = mase_color if kpi_name == 'mase' else mase_all_color
            
            # Add trace for influence scores with appropriate style
            fig.add_trace(
                go.Scatter(
                    x=is_timesteps,
                    y=is_values,
                    mode='lines',
                    name=f'IS {kpi_name} {delta_type}',
                    line=dict(
                        color=line_color,
                        dash=is_style,
                        width=2.5  # Thicker lines for better visibility
                    ),
                    legendgroup=f'is_{kpi_name}_{delta_type}'
                ),
                row=2, col=1
            )
    
    # Add symbolic traces to third subplot using proper positioning
    import re
    
    # Function to extract value categories and create position mappings
    def get_value_positions_for_kpi(symbolic_values):
        """
        Create a mapping of symbolic values to y-axis positions.
        """
        # Extract unique values
        unique_values = set(v for v in symbolic_values if v is not None)
        
        # Sort values based on category, trend, and predicate
        sorted_values = sorted(
            unique_values,
            key=lambda x: (
                # Category ordering (VeryHigh to VeryLow)
                {'VeryHigh': 5, 'High': 4, 'Medium': 3, 'Low': 2, 'VeryLow': 1}.get(
                    re.search(r'(VeryHigh|High|Medium|Low|VeryLow)', x).group() if re.search(r'(VeryHigh|High|Medium|Low|VeryLow)', x) else '',
                    0
                ),
                # Trend ordering (if present)
                {'spiking': 3, 'fluctuating': 2, 'dropping': 1}.get(
                    re.search(r'(spiking|fluctuating|dropping)', x).group() if re.search(r'(spiking|fluctuating|dropping)', x) else '',
                    0
                ),
                # Predicate ordering
                {'inc': 3, 'const': 2, 'dec': 1}.get(
                    re.search(r'(inc|const|dec)', x).group() if re.search(r'(inc|const|dec)', x) else '',
                    0
                )
            )
        )
        
        # Create mapping with increased spacing
        current_pos = 0
        value_to_position = {}
        last_category = None
        
        for value in sorted_values:
            # Extract category from the symbolic value
            category_match = re.search(r'(VeryHigh|High|Medium|Low|VeryLow)', value)
            category = category_match.group() if category_match else None
            
            # Add extra spacing between different categories
            if last_category is None or category != last_category:
                if last_category is not None:
                    current_pos += 4  # Increased spacing between categories
                last_category = category
            
            value_to_position[value] = current_pos
            current_pos += 2  # Increased spacing between values within a category
        
        return value_to_position, sorted_values
    
    # Create a subplot with 2 separate y-axes for the symbolic values
    # First, get position mappings for MASE symbolic values
    mase_position_map, mase_sorted_values = get_value_positions_for_kpi(symbolic_mase)
    
    # Adjust MASE positions to be on the left side
    for key in mase_position_map:
        mase_position_map[key] = mase_position_map[key] 
    
    # Get position mappings for MASE_ALL symbolic values 
    mase_all_position_map, mase_all_sorted_values = get_value_positions_for_kpi(symbolic_mase_all)
    
    # Convert symbolic values to positions
    mase_positions = [mase_position_map.get(val) if val is not None else None for val in symbolic_mase]
    mase_all_positions = [mase_all_position_map.get(val) if val is not None else None for val in symbolic_mase_all]
    
    # Add MASE symbolic trace with proper positioning (left y-axis)
    fig.add_trace(
        go.Scatter(
            x=symbolic_timesteps,
            y=mase_positions,
            mode='lines+markers',
            name='Symbolic MASE',
            line=dict(color=mase_color, width=2.5),
            marker=dict(size=10, symbol='circle'),  # Larger markers for better visibility
            legendgroup='symbolic_mase',
            hovertemplate='Timestep: %{x}<br>Value: %{text}<extra></extra>',
            text=symbolic_mase
        ),
        row=3, col=1, secondary_y=False
    )
    
    # Add MASE_ALL symbolic trace with proper positioning (right y-axis)
    fig.add_trace(
        go.Scatter(
            x=symbolic_timesteps,
            y=mase_all_positions,
            mode='lines+markers',
            name='Symbolic MASE_ALL',
            line=dict(color=mase_all_color, width=2.5),
            marker=dict(size=10, symbol='circle'),  # Larger markers for better visibility
            legendgroup='symbolic_mase_all',
            hovertemplate='Timestep: %{x}<br>Value: %{text}<extra></extra>',
            text=symbolic_mase_all
        ),
        row=3, col=1, secondary_y=True
    )
    
    # Update y-axis for symbolic subplot with ticklabels showing the symbolic categories
    all_sorted_values = list(set(mase_sorted_values) | set(mase_all_sorted_values))
    all_positions = {}
    for val in mase_sorted_values:
        all_positions[val] = mase_position_map[val]
    for val in mase_all_sorted_values:
        all_positions[val] = mase_all_position_map[val]
    
    sorted_all_values = sorted(all_sorted_values, 
                              key=lambda x: all_positions[x])
    
    # Update layout
    fig.update_layout(
        height=1500,  # Kept the same overall height
        hovermode='x unified',
        legend_title="Metrics",
        title_text=f"User {user_idx} MASE Analysis",
        font=dict(
            size=14  # Increased font size
        ),
        title_font=dict(
            size=20  # Larger title font
        ),
        legend=dict(
            font=dict(
                size=14  # Increased legend font size
            )
        ),
        # Adjust margins to give more vertical space
        margin=dict(t=100, b=80, l=80, r=40)
    )
    
    # Update y-axis titles with increased font size
    fig.update_yaxes(
        title_text="MASE Value", 
        row=1, 
        col=1,
        title_font=dict(size=16),  # Increased axis title font
        tickfont=dict(size=14)     # Increased tick label font
    )
    fig.update_yaxes(
        title_text="Influence Score", 
        row=2, 
        col=1,
        title_font=dict(size=16),  # Increased axis title font
        tickfont=dict(size=14)     # Increased tick label font
    )
    # Update y-axis for symbolic subplots - left axis for MASE
    fig.update_yaxes(
        title_text="MASE Symbolic Values", 
        row=3, 
        col=1,
        secondary_y=False,
        title_font=dict(size=18, color=mase_color),   # Increased axis title font with color
        tickfont=dict(size=14, color=mase_color),     # Increased tick label font with color
        showticklabels=True,                          # Show tick labels to display symbolic values
        tickmode='array',                             # Use array mode for custom tick positions
        ticktext=mase_sorted_values,                  # Use the sorted symbolic values as labels
        tickvals=[mase_position_map[v] for v in mase_sorted_values],  # Use their positions as values
        showgrid=True,
        gridcolor='rgba(211,211,211,0.5)',
        tickangle=0,                                  # Keep text horizontal
        side='left'                                   # Put the axis on the left side
    )
    
    # Update y-axis for symbolic subplots - right axis for MASE_ALL
    fig.update_yaxes(
        title_text="MASE_ALL Symbolic Values", 
        row=3, 
        col=1,
        secondary_y=True,
        title_font=dict(size=18, color=mase_all_color),   # Increased axis title font with color
        tickfont=dict(size=14, color=mase_all_color),     # Increased tick label font with color
        showticklabels=True,                              # Show tick labels to display symbolic values
        tickmode='array',                                 # Use array mode for custom tick positions
        ticktext=mase_all_sorted_values,                  # Use the sorted symbolic values as labels
        tickvals=[mase_all_position_map[v] for v in mase_all_sorted_values],  # Use positions as values
        showgrid=False,                                   # Don't show grid for secondary axis
        tickangle=0,                                      # Keep text horizontal
        side='right'                                      # Put the axis on the right side
    )
    
    # Update x-axis title only for the bottom subplot with increased font size
    fig.update_xaxes(
        title_text="Timestep", 
        row=3, 
        col=1,
        title_font=dict(size=16),  # Increased axis title font
        tickfont=dict(size=14)     # Increased tick label font
    )
    
    # Add light gridlines for better readability to numeric subplots
    fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='rgba(211,211,211,0.5)', row=1, col=1)
    fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='rgba(211,211,211,0.5)', row=2, col=1)
    fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='rgba(211,211,211,0.5)', row=3, col=1)
    fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='rgba(211,211,211,0.5)', row=1, col=1)
    fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='rgba(211,211,211,0.5)', row=2, col=1)
    
    # Adjust y-axis range for the symbolic subplot based on the actual values - left axis (MASE)
    min_mase_position = min(pos for pos in mase_positions if pos is not None)
    max_mase_position = max(pos for pos in mase_positions if pos is not None)
    padding = (max_mase_position - min_mase_position) * 0.1  # Add 10% padding
    fig.update_yaxes(range=[min_mase_position - padding, max_mase_position + padding], 
                     row=3, col=1, secondary_y=False)
    
    # Adjust y-axis range for the symbolic subplot based on the actual values - right axis (MASE_ALL)
    min_mase_all_position = min(pos for pos in mase_all_positions if pos is not None)
    max_mase_all_position = max(pos for pos in mase_all_positions if pos is not None)
    padding_all = (max_mase_all_position - min_mase_all_position) * 0.1  # Add 10% padding
    fig.update_yaxes(range=[min_mase_all_position - padding_all, max_mase_all_position + padding_all], 
                     row=3, col=1, secondary_y=True)
    
    # Save if filename provided (HTML only)
    if output_filename:
        fig.write_html(f"{output_filename}.html")
    
    return fig

def plot_all_users_mase_with_influence_scores_and_symbolic(test_df, test_symbolic_df, is_dfs, is_styles=None, timestamp_suffix=False, timestep_col='timestep'):
    """
    Creates visualizations for each user (0-6) showing MASE values, influence scores, and symbolic values.
    
    Parameters:
    -----------
    test_df : pandas.DataFrame
        DataFrame containing raw KPI values with columns like timestep, mase, mase_all, etc.
    test_symbolic_df : pandas.DataFrame
        DataFrame containing symbolic descriptions of KPIs
    is_dfs : list
        List of DataFrames containing influence scores with columns timestep, influence_scores, delta_type
    is_styles : list, optional
        List of line styles for each is_df (default: ['dash', 'dot', 'dashdot'])
    timestamp_suffix : bool, optional
        Whether to add a timestamp suffix to output filenames (default: False)
    timestep_col : str, optional
        Name of the timestep column in test_df (default: 'timestep')
        
    Returns:
    --------
    dict: Dictionary of figures indexed by user index
    """
    import os
    import datetime
    
    # Set default styles if none provided - using wider dashedwidth for better visibility
    if is_styles is None:
        is_styles = ['dash', 'dot', 'dashdot'][:len(is_dfs)]
    
    # Ensure line width is sufficient for good visibility
    line_width = 2  # Default line width for better visibility
    
    # Create output directory if it doesn't exist
    output_dir = create_visualization_folder_path("IS_Plots_Symbolic", "test_data")
    
    # Add timestamp suffix to filenames if requested
    time_suffix = datetime.datetime.now().strftime("_%Y%m%d_%H%M") if timestamp_suffix else ""
    
    # Dictionary to store all figures
    figures = {}
    
    # Get delta types for filename
    delta_types = []
    for is_df in is_dfs:
        if 'delta_type' in is_df:
            delta_type = is_df['delta_type'].iloc[0]
            if delta_type not in delta_types:
                delta_types.append(delta_type)
    
    delta_str = "_".join(delta_types) if delta_types else "combined"
    
    # Plot for each user
    for user_idx in range(7):
        print(f"Plotting User {user_idx}...")
        
        # Generate output filename
        output_filename = os.path.join(output_dir, f"user_{user_idx}_mase_symbolic_{delta_str}{time_suffix}")
        
        # Create the figure
        fig = plot_user_mase_with_influence_scores_and_symbolic(
            test_df=test_df,
            test_symbolic_df=test_symbolic_df,
            is_dfs=is_dfs,
            is_styles=is_styles,
            user_idx=user_idx,
            timestep_col=timestep_col,
            output_filename=output_filename
        )
        
        # Store the figure in the dictionary
        figures[user_idx] = fig
        
        print(f"Saved plot to {output_filename}.html")
    
    return figures

# Example usage:
# Define line styles for each IS dataframe
is_styles = ['dash', 'dot']

# Generate plots for all users including symbolic data
figures = plot_all_users_mase_with_influence_scores_and_symbolic(
    test_df=test_df,
    test_symbolic_df=test_symbolic_df,
    is_dfs=[IS_df_binary, IS_df_decaying],  # Multiple IS dataframes
    is_styles=is_styles
)

# Display a specific user's figure if needed
# figures[0].show()  # Show User 0's figure


### Visualize with symbolic values

In [ ]:
def get_symbolic_level_order():
    """Defines the desired order for symbolic levels."""
    return ["VeryHigh", "High", "Medium", "Low", "VeryLow"]

def extract_symbolic_level(symbolic_value):
    """Extracts the symbolic level (like VeryHigh, Low) from the symbolic string."""
    for level in get_symbolic_level_order():
        if level in symbolic_value:
            return level
    return symbolic_value # Return original if level not found, for robustness

def sort_symbolic_values(symbolic_values):
    """Sorts symbolic values based on the defined level order."""
    level_order = get_symbolic_level_order()
    level_rank = {level: i for i, level in enumerate(level_order)}

    def sort_key(symbolic_value):
        level = extract_symbolic_level(symbolic_value)
        return level_rank.get(level, len(level_order)) # Rank unknown levels last

    return sorted(list(set(symbolic_values)), key=sort_key, reverse=True) # reverse=True for VeryHigh on top

def plot_kpis_symbolic_and_is_improved(symbolic_df, is_dfs, is_styles, timestep_col, file_name_col, kpis, dataset_name, specific_name="", title=None):
    """
    Plots KPIs and IS, reducing vertical space between subplots.
    """
    vis_name = "kpi_symbolic_and_multiple_is_improved"
    folder_path = create_visualization_folder_path(vis_name, dataset_name, specific_name)

    n_kpis = len(kpis)
    # Reduce vertical_spacing
    fig = make_subplots(rows=n_kpis, cols=1,
                        subplot_titles=[f"<b>{kpi} (Symbolic Y-Axis)</b>" for kpi in kpis],
                        shared_xaxes=True, vertical_spacing=0.02,  # Reduced spacing
                        specs=[[{'secondary_y': True}] for _ in range(n_kpis)])

    file_names = natsorted(symbolic_df[file_name_col].unique())
    colors = qualitative.Plotly
    color_mapping = {file_name: colors[i % len(colors)] for i, file_name in enumerate(file_names)}

    for i, kpi in enumerate(kpis):
        unique_symbolic_values = symbolic_df[kpi].unique()
        sorted_symbolic_values = sort_symbolic_values(unique_symbolic_values)
        value_to_position = {value: pos for pos, value in enumerate(sorted_symbolic_values)}

        num_unique = len(sorted_symbolic_values)
        max_ticks = 10
        tickvals = []
        ticktext = []

        if num_unique <= max_ticks:
            tickvals = list(range(num_unique))
            ticktext = sorted_symbolic_values
        else:
            step = max(1, (num_unique - 1) // (max_ticks - 1))
            indices = list(range(0, num_unique, step))
            if indices[-1] != num_unique - 1:
                indices[-1] = num_unique - 1
            tickvals = indices
            ticktext = [sorted_symbolic_values[idx] for idx in indices]

        for j, file_name in enumerate(file_names):
            symbolic_df_filtered = symbolic_df[symbolic_df[file_name_col] == file_name]
            color = color_mapping[file_name]

            fig.add_trace(
                go.Scatter(
                    x=symbolic_df_filtered[timestep_col],
                    y=symbolic_df_filtered[kpi].map(value_to_position),
                    mode='lines',
                    name=f"{file_name} (Symbolic)",
                    line=dict(color=color),
                    legendgroup=file_name,
                    showlegend=(i == 0)
                ),
                row=i + 1, col=1, secondary_y=False
            )

            for is_df, style in zip(is_dfs, is_styles):
                is_df_type = is_df['delta_type'].iloc[0]
                is_df_filtered = is_df[is_df[file_name_col] == file_name]
                fig.add_trace(
                    go.Scatter(
                        x=is_df_filtered[timestep_col],
                        y=is_df_filtered[kpi],
                        mode='lines',
                        name=f"{file_name} (IS - {is_df_type}, Style: {style})",
                        line=dict(color=color, dash=style),
                        legendgroup=file_name,
                        showlegend=(i == 0)
                    ),
                    row=i + 1, col=1, secondary_y=True
                )

        fig.update_yaxes(
            title_text="<b>Symbolic Value</b>",
            secondary_y=False,
            row=i + 1, col=1,
            tickmode='array',
            tickvals=tickvals,
            ticktext=ticktext,
            tickangle=0,
            title_font=dict(size=16),
            tickfont=dict(size=14),
            automargin=True,
        )
        fig.update_yaxes(title_text="<b>Influence Score</b>", secondary_y=True, row=i + 1, col=1, title_font=dict(size=16), tickfont=dict(size=14))

    # Adjust overall height calculation
    fig.update_layout(
        height=max(450 * n_kpis, 1000),  # Adjusted height calculation
        title_text=f"<b>{title if title else 'Symbolic KPIs and Influence Scores'}</b>",
        title_font=dict(size=24, family="Arial", color="black"),
        title_x=0.5,
        legend_title="<b>File Name and Data Type</b>",
        hovermode='x unified',
        font=dict(size=16, family="Arial"),
        margin=dict(l=250, r=50, b=100, t=100)
    )
    base_filename = f"kpi_symbolic_and_is_improved_{dataset_name}"
    if specific_name:
        base_filename += f"_{specific_name}"

    png_filename = os.path.join(folder_path, f"{base_filename}.png")
    fig.write_image(png_filename)
    print(f"Plot saved as PNG: {png_filename}")

    html_filename = os.path.join(folder_path, f"{base_filename}.html")
    fig.write_html(html_filename)
    print(f"Plot saved as HTML: {html_filename}")

    fig.show()

# Example Usage (same as before, but with the new function):
kpi_columns = [col for col in IS_df_binary.columns if col not in ['Timestep', 'File_name', 'delta_type']]

plot_kpis_symbolic_and_is_improved(
    symbolic_df=symbolic_df.copy(),  # Replace with your symbolic DataFrame
    is_dfs=[IS_df_hierarchical.copy(), IS_df_decaying.copy()],  # Replace with your IS DataFrames
    is_styles=['dash', 'dot'],
    timestep_col='Timestep',
    file_name_col='File_name',
    kpis=kpi_columns,  # Replace with your KPI columns
    dataset_name=dataset,
    specific_name="hierarchical_vs_decaying_improved_compact",  # Descriptive name
    title="Symbolic KPIs vs. Hierarchical and Decaying Influence Scores (Improved Y-Axis)"
)

### Visualize using heatmap for symbolic without softmax

In [ ]:
def sort_symbolic_kpi_strings_key(symbolic_string):
    """
    Key function for sorting symbolic KPI strings.
    """
    level_order = ["VeryHigh", "High", "Medium", "Low", "VeryLow"]
    parts = symbolic_string.split(", ")
    if len(parts) == 3:
        category = parts[1].strip()
        for level in level_order:
            if level in category:
                return level_order.index(level)
    else:
        for level in level_order:
            if level in symbolic_string:
                return level_order.index(level)
    return len(level_order)

def get_all_possible_symbolic_values(kpi_name):
    """
    Generates all possible symbolic values for a given KPI.
    """
    predicates = ['inc', 'const', 'dec']
    categories = ['VeryLow', 'Low', 'Medium', 'High', 'VeryHigh']
    if kpi_name.endswith('_all'):
        extra_states = ['dropping', 'spiking', 'fluctuating']
        return [f"{p}({kpi_name}, {e})" for p in predicates for e in extra_states]
    else:
        return [f"{p}({kpi_name}, {c})" for p in predicates for c in categories]

def plot_all_heatmaps_in_one(symbolic_df, IS_df_binary, kpi_columns, dataset_name, specific_name="", bins=np.arange(0, 1.4, 0.1)):
    """
    Generates all frequency heatmaps as subplots in a single figure and saves it.
    """
    n_kpis = len(kpi_columns)
    n_cols = 3  # Number of columns for subplots (adjust as needed)
    n_rows = (n_kpis + n_cols - 1) // n_cols  # Calculate rows needed

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 5 * n_rows))  # Adjust figsize
    axes = axes.flatten()  # Flatten the axes array for easy iteration

    for i, kpi in enumerate(kpi_columns):
        if kpi not in symbolic_df.columns or kpi not in IS_df_binary.columns:
            print(f"Warning: KPI '{kpi}' not found in both DataFrames. Skipping.")
            continue

        # Get unique and sorted symbolic values
        unique_symbolic_values = symbolic_df[kpi].unique()
        sorted_unique_symbolic_values = sorted(unique_symbolic_values, key=sort_symbolic_kpi_strings_key, reverse=True)

        # Bin continuous values
        IS_df_binary_binned = IS_df_binary.copy()
        IS_df_binary_binned[kpi] = pd.cut(IS_df_binary[kpi], bins=bins, labels=[f"{bins[i]:.1f}" for i in range(len(bins)-1)], right=False)
        IS_df_binary_binned = IS_df_binary_binned.dropna(subset=[kpi])
        valid_indices = IS_df_binary_binned.index
        symbolic_df_filtered = symbolic_df.loc[valid_indices]
        IS_df_binary_binned = IS_df_binary_binned.loc[valid_indices]

        # Compute frequency counts
        freq_table = pd.crosstab(symbolic_df_filtered[kpi], IS_df_binary_binned[kpi])

        # Reindex with sorted unique symbolic values and bins
        freq_table = freq_table.reindex(index=sorted_unique_symbolic_values, columns=[f"{bins[i]:.1f}" for i in range(len(bins)-1)], fill_value=0)

        # Create annotation mask
        annot_mask = freq_table.values > 0

        # Plot the heatmap on the appropriate subplot
        sns.heatmap(freq_table, annot=np.where(annot_mask, freq_table.values, ''), fmt="", cmap="viridis", cbar_kws={'label': 'Count'}, annot_kws={"size": 8}, ax=axes[i])
        axes[i].set_title(f"{kpi}")  # Set subplot title
        axes[i].set_xlabel("Binned IS")
        axes[i].set_ylabel("Symbolic State")

    # Hide any unused subplots
    for j in range(i + 1, len(axes)):
        axes[j].axis('off')

    plt.tight_layout()
    
    # Save the figure
    vis_name = "IS_Score_heatmaps"
    folder_path = create_visualization_folder_path(vis_name, dataset_name, specific_name)
    base_filename = f"all_heatmaps_{dataset_name}"
    if specific_name:
        base_filename += f"_{specific_name}"
    png_filename = os.path.join(folder_path, f"{base_filename}.png")
    plt.savefig(png_filename)
    print(f"Heatmap saved as: {png_filename}")

    plt.show()


# Example Usage (assuming you have symbolic_df, IS_df_binary, kpi_columns, and dataset defined)

kpi_columns = [col for col in IS_df_binary.columns if col not in ['Timestep', 'File_name', 'delta_type']]  # Make sure this is defined!

plot_all_heatmaps_in_one(symbolic_df, IS_df_binary, kpi_columns, dataset_name=dataset)

### Visualize using heatmap for symbolic with softmax

In [ ]:
def softmax(x):
    """Compute softmax values for each sets of scores in x."""
    e_x = np.exp(x - np.max(x))  # Subtract max for numerical stability
    return e_x / e_x.sum()

def create_softmax_dataframe(df, kpi_columns):
    """
    Creates a new DataFrame with softmax values applied to KPI columns at each timestep.

    Args:
        df: The original DataFrame.
        kpi_columns: A list of column names representing the KPIs.

    Returns:
        A new DataFrame with the softmaxed values, along with original 'Timestep', 'File_name', and 'delta_type'.
        Returns None if there is any error.
    """
    # Input validation and error handling
    if not isinstance(df, pd.DataFrame):
        raise TypeError("df must be a pandas DataFrame")
    if not isinstance(kpi_columns, list):
        raise TypeError("kpi_columns must be a list")
    for col in kpi_columns:
        if col not in df.columns:
            raise ValueError(f"KPI column '{col}' not found in DataFrame")
    if not all(pd.api.types.is_numeric_dtype(df[col]) for col in kpi_columns):
        raise ValueError("All KPI columns must be numeric")
    if 'Timestep' not in df.columns or 'File_name' not in df.columns or 'delta_type' not in df.columns:
        raise ValueError("'Timestep', 'File_name', and 'delta_type' columns must be present in DataFrame")
    

    # Initialize lists to store the results.  We build lists of dictionaries,
    # then create the dataframe at the end, which is much more efficient than
    # appending to a dataframe row by row.
    softmax_data = []
    
    # Iterate through unique combinations of 'File_name' and 'delta_type'.
    for (file_name, delta_type), group in df.groupby(['File_name', 'delta_type']):
      for timestep, time_group in group.groupby('Timestep'):
        # Get the KPI values for the current timestep
        kpi_values = time_group[kpi_columns].values.flatten()

        # Apply softmax
        softmax_values = softmax(kpi_values)

        # Create a dictionary for the current row, preserving Timestep, File_name, and delta_type
        row_dict = {
            'Timestep': timestep,
            'File_name': file_name,
            'delta_type': delta_type
        }
        # Add the softmax values to the dictionary, using the original KPI column names
        row_dict.update({kpi_col: softmax_val for kpi_col, softmax_val in zip(kpi_columns, softmax_values)})
        softmax_data.append(row_dict)

    # Create the new DataFrame
    softmax_df = pd.DataFrame(softmax_data)
    return softmax_df


# Example usage (assuming IS_df_binary and kpi_columns are defined as before)
kpi_columns = [col for col in IS_df_binary.columns if col not in ['Timestep', 'File_name', 'delta_type']]
softmax_df = create_softmax_dataframe(IS_df_binary, kpi_columns)

def sort_symbolic_kpi_strings_key(symbolic_string):
    """
    Key function for sorting symbolic KPI strings.
    """
    level_order = ["VeryHigh", "High", "Medium", "Low", "VeryLow"]
    parts = symbolic_string.split(", ")
    if len(parts) == 3:
        category = parts[1].strip()
        for level in level_order:
            if level in category:
                return level_order.index(level)
    else:
        for level in level_order:
            if level in symbolic_string:
                return level_order.index(level)
    return len(level_order)

def get_all_possible_symbolic_values(kpi_name):
    """
    Generates all possible symbolic values for a given KPI.
    """
    predicates = ['inc', 'const', 'dec']
    categories = ['VeryLow', 'Low', 'Medium', 'High', 'VeryHigh']
    if kpi_name.endswith('_all'):
        extra_states = ['dropping', 'spiking', 'fluctuating']
        return [f"{p}({kpi_name}, {e})" for p in predicates for e in extra_states]
    else:
        return [f"{p}({kpi_name}, {c})" for p in predicates for c in categories]

def plot_all_heatmaps_in_one(symbolic_df, IS_df_binary, kpi_columns, dataset_name, specific_name="", bins=np.arange(0, 0.4, 0.05)):
    """
    Generates all frequency heatmaps as subplots in a single figure and saves it.
    Uses bins from 0 to 0.4 with steps of 0.04 for the x-axis.
    """
    n_kpis = len(kpi_columns)
    n_cols = 3  # Number of columns for subplots (adjust as needed)
    n_rows = (n_kpis + n_cols - 1) // n_cols  # Calculate rows needed

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 5 * n_rows))  # Adjust figsize
    if n_kpis==1:
        axes = np.array([axes])
    axes = axes.flatten()  # Flatten the axes array for easy iteration


    for i, kpi in enumerate(kpi_columns):
        if kpi not in symbolic_df.columns or kpi not in IS_df_binary.columns:
            print(f"Warning: KPI '{kpi}' not found in both DataFrames. Skipping.")
            continue

        # Get unique and sorted symbolic values
        unique_symbolic_values = symbolic_df[kpi].unique()
        sorted_unique_symbolic_values = sorted(unique_symbolic_values, key=sort_symbolic_kpi_strings_key, reverse=True)

        # Bin continuous values
        IS_df_binary_binned = IS_df_binary.copy()
        # Use the provided bins (0 to 0.4 with steps of 0.04)
        IS_df_binary_binned[kpi] = pd.cut(IS_df_binary[kpi], bins=bins, labels=[f"{bins[i]:.2f}" for i in range(len(bins)-1)], right=False)
        IS_df_binary_binned = IS_df_binary_binned.dropna(subset=[kpi])
        valid_indices = IS_df_binary_binned.index
        symbolic_df_filtered = symbolic_df.loc[valid_indices]
        IS_df_binary_binned = IS_df_binary_binned.loc[valid_indices]


        # Compute frequency counts
        freq_table = pd.crosstab(symbolic_df_filtered[kpi], IS_df_binary_binned[kpi])

        # Reindex with sorted unique symbolic values and bins
        freq_table = freq_table.reindex(index=sorted_unique_symbolic_values, columns=[f"{bins[i]:.2f}" for i in range(len(bins)-1)], fill_value=0)

        # Create annotation mask
        annot_mask = freq_table.values > 0

        # Plot the heatmap on the appropriate subplot
        sns.heatmap(freq_table, annot=np.where(annot_mask, freq_table.values, ''), fmt="", cmap="viridis", cbar_kws={'label': 'Count'}, annot_kws={"size": 8}, ax=axes[i])
        axes[i].set_title(f"{kpi}")  # Set subplot title
        axes[i].set_xlabel("Binned IS (Softmax)")  # Updated x-axis label
        axes[i].set_ylabel("Symbolic State")

    # Hide any unused subplots
    for j in range(i + 1, len(axes)):
        axes[j].axis('off')

    plt.tight_layout()
    
    # Save the figure
    vis_name = "softmax_heatmaps"  # Changed folder name
    folder_path = create_visualization_folder_path(vis_name, dataset_name, specific_name)
    base_filename = f"all_heatmaps_{dataset_name}"
    if specific_name:
        base_filename += f"_{specific_name}"
    png_filename = os.path.join(folder_path, f"{base_filename}.png")
    plt.savefig(png_filename)
    print(f"Heatmap saved as: {png_filename}")

    plt.show()


# Example Usage (assuming you have symbolic_df, IS_df_binary (now softmax_df), kpi_columns, and dataset defined)

kpi_columns = [col for col in softmax_df.columns if col not in ['Timestep', 'File_name', 'delta_type']]

plot_all_heatmaps_in_one(symbolic_df, softmax_df, kpi_columns, dataset_name=dataset, specific_name="softmaxed")

In [ ]:
import plotly.express as px

# Melt the DataFrame so that each KPI's influence scores become a variable
df_melted = IS_df_binary.melt(
    id_vars=['Timestep', 'File_name', 'delta_type'],
    value_vars=kpi_columns,  # using previously defined kpi_columns
    var_name='KPI',
    value_name='Influence_Score'
)

fig = px.box(
    df_melted,
    x='KPI',
    y='Influence_Score',
    points="all",
    title="Distribution of Influence Scores per KPI (Binary)"
)
fig.show()